# Sionna RT — OSM Scene Builder
## Physics-based radio scene construction from OpenStreetMap + EA LiDAR

Builds a complete 3-D radio scene for **Sionna 2.0** and **Sionna 0.19** ray tracing from:
- **OpenStreetMap** — buildings, roads, water, vegetation, trees, railways, barriers
- **EA LiDAR / SRTM** — DTM (bare earth terrain) + DSM (surface with clutter)
- **nDSM = DSM − DTM** — measured height of every building, tree and structure

---

## Build Sequence

### Before rebuilding — delete stale files

```bash
cd ~/sionna_rt/nottingham_ofcom2018_915mhz_dem/scene_v4_full/meshes/

# Stale PLYs (disabled features + old vegetation geometry)
rm -f veg_itu_vegetation.ply \
      trees_itu_vegetation.ply \
      bld_itu_medium_dry_ground_embankments.ply \
      infra_itu_medium_dry_ground_road_embankments.ply \
      infra_itu_medium_dry_ground_road_cuttings.ply \
      surface_itu_asphalt_carparks.ply

# Stale XML (regenerated by CELL B3 / CELL B1)
rm -f ../scene_with_roads.xml ../scene_with_roads_019.xml
```

> Skip terrain/nDSM deletion — `dem.tif` and `ndsm.tif` are unchanged.

---

### PHASE 1 — Scene Builder (this notebook)

| Step | Cell | One word | Skip when |
|------|------|----------|-----------|
| 1 | **CELL 0** | **Config** | Never |
| 2 | **CELL 1** | **Imports** | Once per session |
| 3 | **CELL 2b** | **Download** | `dem.tif` exists |
| 4 | **CELL 2e** | **Merge** | Merged files valid |
| 5 | **CELL 2d** | **nDSM** | `ndsm.tif` exists |
| 6 | **CELL 3** | **Terrain** | `FLAT_TERRAIN=True` |
| 7 | **CELL 4** | **Build** | Never after scene change |
| 8 | **CELL B3** | **XML-S2** | Never |
| 9 | **CELL B1** | **XML-019** | Never |

> **Shortcut** (terrain unchanged): skip Steps 3–6, run Steps 7–9 only.

---

### PHASE 2 — Differentiable RT Calibration (`sionna019_differentiable_rt_fixed.ipynb`)

| Step | Cell | One word | Output |
|------|------|----------|--------|
| 1 | CELL 0–3 | **Setup** | — |
| 2 | CELL 4 | **Load** | scene loaded |
| 3 | CELL 4A | **Materials** | ITU-R params set |
| 4 | CELL 6–7 | **Receivers** | 1200 RX placed |
| 5 | CELL 8b | **Targets** | calib_rssi_meas |
| 6 | **CELL 8** | **Pre-solve** | pg_at_rx_pre (~5 min) |
| 7 | **CELL 10b** | **Scalar** | scalar_offset_915mhz.json |
| 8 | **CELL 11b** | **Materials** | calibrated_materials.json |

> Cell 8: use `CALIB_DEPTH=8`, `NUM_SAMPLES_PS=2_000_000`, `CALIB_BATCH=2`

---

### PHASE 3 — Validation (`sionna2_915mhz_dem_simulation.ipynb`)

| Step | Cell | One word |
|------|------|----------|
| 1 | CELL 1 | **Config** |
| 2 | CELL 3 | **Load** |
| 3 | **CELL 4A** | **Materials** |
| 4 | CELL 5 | **Receivers** |
| 5 | CELL 7 | **Place** |
| 6 | **CELL 8** | **Simulate** |
| 7 | **CELL 8e** | **Evaluate** |

---

## Knob Reference

| Knob | Current value | Effect |
|------|---------------|--------|
| `SCENE_WEST/EAST/SOUTH/NORTH` | fixed bbox | Manual scene extent (WGS84) — no centre+radius derivation |
| `CENTRE_LAT/LON` | `51.52924, -0.099565` | Reference scene centre (UTM auto-detect only) |
| `FLAT_TERRAIN` | `False` | `True` = skip LiDAR cells |
| `USE_NDSM_HEIGHTS` | `True` | Building + veg heights from LiDAR nDSM |
| `TERRAIN_SOURCE` | `'auto'` | England→EA LiDAR, USA→USGS, global→SRTM |
| `NDSM_PROVIDER` | `'auto'` | Same auto-detection as terrain |
| `CITY_MAX_HEIGHT_M` | `20.0` | Cap at ~6 floors — avoids LiDAR spire noise |
| `VEG_3D_GEOMETRY` | `True` | Hollow canopy shell — blocks horizontal rays ✓ |
| `VEG_CANOPY_M` | `8.0` | Default canopy height (m) when nDSM absent |
| `VEG_HEIGHT_CAP_M` | `10.0` | Hard cap on canopy extrusion |
| `INCLUDE_EMBANKMENTS` | `False` | Disabled — blocks close receivers at 915 MHz |
| `INCLUDE_ROAD_CUTTINGS` | `False` | Disabled — V-trench shadows near-TX |
| `INCLUDE_SURFACE_PARKS` | `False` | Disabled — destructive interference at range |

---

## Key Design Decisions

### Vegetation — hollow canopy shell (recommended)
`VEG_3D_GEOMETRY=True` creates **perimeter walls + flat roof cap** (open bottom).
Horizontal rays at RX height 1.5 m diffract and scatter at the forest edge — physically
correct. Height capped at `VEG_HEIGHT_CAP_M=10 m`. Material: itu_vegetation (S=0.40).

### Building heights — nDSM first
`USE_NDSM_HEIGHTS=True` samples LiDAR nDSM at centroid. Falls back to:
OSM `height=` tag → `building:levels × 3.5 m` → `DEFAULT_HEIGHT_M=8 m`.

### Vegetation height — nDSM first (same as buildings)
`_veg_height()` samples nDSM at polygon centroid, then `VEG_CLASS_HEIGHT_M` by tag,
then `VEG_CANOPY_M` default. Individual tree (natural=tree) heights same priority.

### Portability
Set `CITY_NAME`, `CENTRE_LAT`, `CENTRE_LON`, `RADIUS_KM`, `FREQUENCY_HZ`.
UTM zone, bbox and terrain provider all auto-derive. Set `RT_BASE_DIR` env var
or edit `_ROOT` for a custom output path.


## CELL 0 — Configuration

**Edit this cell for every new campaign.** All other cells read from these variables.

### Knob reference

| Knob | Default | Effect |
|------|---------|--------|
| `SCENARIO_NAME` | `'nottingham_...'` | Folder name under `~/sionna_rt/` |
| `SCENE_WEST/EAST/SOUTH/NORTH` | Nottingham bbox | Scene extent in WGS84 |
| `UTM_EPSG` | `32630` | UTM zone (30N = UK; change per country) |
| `FLAT_TERRAIN` | `False` | `True` = flat z=0 plane (skip LiDAR cells) |
| `TERRAIN_GRID_N` | `500` | Terrain PLY resolution (500×500 recommended) |
| `USE_NDSM_HEIGHTS` | `True` | Use LiDAR nDSM for building heights |
| `NDSM_BUILDING_MIN_M` | `2.0` | Ignore nDSM < this (noise / ground return) |
| `NDSM_PROVIDER` | `'ea'` | LiDAR source: `'ea'` / `'usgs'` / `'opentopo'` |
| `LIDAR_DIR` | `/home/.../lidar` | Folder with pre-downloaded LiDAR tiles |
| `VEG_3D_GEOMETRY` | `True` | `True` = hollow canopy shell (walls+cap, recommended); `False` = flat ground patches + P.833 post-hoc |
| `VEG_MIN_AREA_M2` | `50.0` | Skip vegetation polygons smaller than this |
| `MIN_BUILDING_AREA_M2` | `30.0` | Skip building footprints smaller than this |
| `DEFAULT_HEIGHT_M` | `8.0` | Fallback building height when no tag or nDSM |
| `CITY_MAX_HEIGHT_M` | `40.0` | Cap building height (avoids LiDAR artefacts) |
| `ROOF_PITCH_DEG` | `30.0` | Pitch angle for untagged pitched roofs |
| `FREQUENCY_HZ` | `915.95 MHz` | Used for documentation / cross-reference |

> **To port to a new city:** change `SCENARIO_NAME`, the four bbox values,
> `UTM_EPSG`, and `LIDAR_DIR`. Nothing else needs editing.


In [ ]:
# ============================================================
# BUILD SEQUENCE — SIONNA 2.0 DEM SCENE (with roads)
# ============================================================
# Step 1 : Run CELL 0   — project config
# Step 2 : Run CELL 1   — imports
# Step 3 : Run CELL 2   — terrain PLY  (skip if terrain.ply exists)
# Step 4 : Run CELL 2b  — nDSM heights (skip if already done)
# Step 5 : Run CELL 4   — OSM buildings + ALL roads → PLYs
# Step 6 : Run CELL B3  — write scene_with_roads.xml  (Sionna 2.0)
# ── Scene is now ready for sionna2_915mhz_dem_simulation.ipynb ──
#
# TO ALSO PREPARE SIONNA 0.19 (differentiable RT):
# Step 7 : Run CELL B1  — convert scene_with_roads.xml
#                          → scene_with_roads_019.xml
# ── Scene is now ready for sionna019_differentiable_rt_fixed.ipynb ──
# ============================================================

# ============================================================
# CELL 0 — CONFIG  (edit this block only)
# ============================================================
import os
import math

# ╔══════════════════════════════════════════════════════════╗
# ║                  SCENARIO CONFIGURATION                  ║
# ║           Edit this block for each new campaign          ║
# ╚══════════════════════════════════════════════════════════╝

SCENARIO_NAME   = 'london_ofcom_915mhz_dem'
OS_API_KEY          = ''            # Optional: OS Data Hub API key for Building Height Attribute
EPC_EMAIL           = ''            # EPC registered email (get-energy-performance-data.communities.gov.uk)
EPC_API_KEY         = ''            # EPC API key for wall/roof/glazing material data
CITY_NAME       = 'london'              # used for auto-naming output directory

# ── Scene bbox (WGS84) — auto-derived from the measurement CSV, no area cap ──
# AUTO_BBOX=True (default): bbox is computed from the full extent of every
# RX in london915.csv + the TX, so the scene always covers ALL loaded
# receivers regardless of how many are in the file (1200, 4000, ...).
# AUTO_BBOX=False: falls back to the fixed manual values below.
AUTO_BBOX        = False
AUTO_BBOX_MARGIN_KM = 0.5     # padding beyond the furthest RX/TX
# london915.csv has 64,924 rows spanning all of London, but the DEM sim
# notebook only ever loads the first NUM_RX rows sequentially (one
# contiguous drive route) - bbox must match that same subset, not the
# whole file, or the scene balloons to city-wide while the real RX
# route is a tiny cluster in one corner.
AUTO_BBOX_NUM_RX = 4000

# Fallback / reference bbox — used only if AUTO_BBOX=False or the CSV
# can't be parsed.
CENTRE_LAT     =  51.613150     # scene centre lat (for reference / UTM auto-detect)
CENTRE_LON     = -0.143450      # scene centre lon (for reference / UTM auto-detect)

# Actual transmitter position - MUST match TX_LON/TX_LAT in
# sionna2_915mhz_dem_simulation_london.ipynb CELL 1 exactly. NOT the
# same as CENTRE_LAT/LON (which is just the bbox midpoint).
TX_LAT_REF     =  51.5305
TX_LON_REF     = -0.13399

SCENE_WEST  = -0.231017
SCENE_EAST  = -0.036963
SCENE_SOUTH = 51.47014
SCENE_NORTH = 51.59086

if AUTO_BBOX:
    _csv_root = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', 'london_ofcom_915mhz_dem'))
    _measurement_csv = os.path.join(_csv_root, 'london915.csv')
    if os.path.exists(_measurement_csv):
        import pandas as _pd, re as _re
        with open(_measurement_csv, 'r', encoding='utf-8', errors='replace') as _f:
            _all_lines = _f.readlines()
        _tx_lat_a = _tx_lon_a = None
        for _line in _all_lines[:40]:
            if 'latitude' in _line.lower() and _tx_lat_a is None:
                _m = _re.search(r'([-+]?\d+\.\d+)', _line)
                if _m: _tx_lat_a = float(_m.group(1))
            if 'longitude' in _line.lower() and _tx_lon_a is None:
                _m = _re.search(r'([-+]?\d+\.\d+)', _line)
                if _m: _tx_lon_a = float(_m.group(1))
        _hdr_idx_a = next((i for i, l in enumerate(_all_lines)
                            if 'Latitude' in l and 'Longitude' in l), None)
        if _hdr_idx_a is None:
            print(f'AUTO_BBOX: could not find header row in {_measurement_csv} — using fallback bbox.')
        else:
            _df_a = _pd.read_csv(_measurement_csv, skiprows=_hdr_idx_a, low_memory=False).head(AUTO_BBOX_NUM_RX)
            def _fcol_a(df, *kws):
                for c in df.columns:
                    if all(k.lower() in c.strip().lower() for k in kws):
                        return c
                return None
            _lat_col_a = _fcol_a(_df_a, 'latitude')
            _lon_col_a = _fcol_a(_df_a, 'longitude')
            if _lat_col_a and _lon_col_a:
                _df_a[_lat_col_a] = _pd.to_numeric(_df_a[_lat_col_a], errors='coerce')
                _df_a[_lon_col_a] = _pd.to_numeric(_df_a[_lon_col_a], errors='coerce')
                _lats_a = _df_a[_lat_col_a].dropna().tolist()
                _lons_a = _df_a[_lon_col_a].dropna().tolist()
                if _tx_lat_a is not None: _lats_a.append(_tx_lat_a)
                if _tx_lon_a is not None: _lons_a.append(_tx_lon_a)
                if _lats_a and _lons_a:
                    import numpy as _np_a
                    # Robust range (0.5-99.5 pct) - a handful of bad GPS
                    # fixes (0,0 / stray header rows / outliers) must not
                    # blow up the whole scene bbox.
                    _min_lat_a, _max_lat_a = _np_a.percentile(_lats_a, [0.5, 99.5])
                    _min_lon_a, _max_lon_a = _np_a.percentile(_lons_a, [0.5, 99.5])
                    _min_lat_a, _max_lat_a = float(_min_lat_a), float(_max_lat_a)
                    _min_lon_a, _max_lon_a = float(_min_lon_a), float(_max_lon_a)
                    _mid_lat_a = (_min_lat_a + _max_lat_a) / 2
                    _dlat_a = AUTO_BBOX_MARGIN_KM / 111.32
                    _dlon_a = AUTO_BBOX_MARGIN_KM / (111.32 * math.cos(math.radians(_mid_lat_a)))
                    SCENE_WEST  = _min_lon_a - _dlon_a
                    SCENE_EAST  = _max_lon_a + _dlon_a
                    SCENE_SOUTH = _min_lat_a - _dlat_a
                    SCENE_NORTH = _max_lat_a + _dlat_a
                    CENTRE_LAT  = (SCENE_NORTH + SCENE_SOUTH) / 2
                    CENTRE_LON  = (SCENE_EAST  + SCENE_WEST)  / 2
                    print(f'AUTO_BBOX: derived from first {len(_lats_a)} RX/TX points in london915.csv '
                          f'(+{AUTO_BBOX_MARGIN_KM} km margin)')
                    print(f'  SCENE_WEST/EAST  : {SCENE_WEST:.6f} / {SCENE_EAST:.6f}')
                    print(f'  SCENE_SOUTH/NORTH: {SCENE_SOUTH:.6f} / {SCENE_NORTH:.6f}')
                else:
                    print('AUTO_BBOX: no valid lat/lon rows found — using fallback bbox.')
            else:
                print(f'AUTO_BBOX: could not find lat/lon columns — using fallback bbox.')
    else:
        print(f'AUTO_BBOX: {_measurement_csv} not found — using fallback bbox.')

# ── Coordinate system — auto-detect UTM zone from scene centre ───────────────
def _auto_utm_epsg(lat, lon):
    zone = int((lon + 180) / 6) + 1
    return 32600 + zone if lat >= 0 else 32700 + zone

# ── Projection knob ─────────────────────────────────────────
# 'bng'    -> British National Grid, EPSG:27700 (default -- native UK
#             Ordnance Survey grid; matches EA LiDAR WCS tiles directly,
#             no extra reprojection needed for DTM/DSM downloads)
# 'utm30n' -> EPSG:32630 (UTM zone 30N, covers UK) -- kept as an
#             alternative for portability to non-OS-grid workflows.
# Must match PROJECTION_CRS in sionna2_915mhz_dem_simulation_london.ipynb
# CELL 1 exactly, or TX/RX positions will be computed in different CRSs.
PROJECTION_CRS = 'bng'    # 'bng' | 'utm30n'

_PROJECTION_EPSG_MAP = {'bng': 27700, 'utm30n': 32630}
if PROJECTION_CRS == 'utm30n':
    UTM_EPSG = _auto_utm_epsg(
        (SCENE_NORTH + SCENE_SOUTH) / 2,
        (SCENE_EAST  + SCENE_WEST)  / 2
    )
else:
    UTM_EPSG = _PROJECTION_EPSG_MAP.get(PROJECTION_CRS, 27700)
print(f'Projection CRS: {PROJECTION_CRS}  ->  EPSG:{UTM_EPSG}')

# ── Terrain ────────────────────────────────────────
FLAT_TERRAIN   = False
TERRAIN_SOURCE = 'auto'       # 'auto' = detect from coordinates (recommended)
                               # 'ea_lidar' (England) | 'srtm' (global) | 'usgs_1m' (USA)
TERRAIN_GRID_N = 500
TERRAIN_PAD_M  = 3000          # 3km beyond furthest receiver (9342m)
TILE_ZOOM      = 14

# ── LiDAR clutter heights (nDSM = DSM - DTM) ──────────────
# USE_NDSM_HEIGHTS = True  -> building heights from LiDAR nDSM (portable,
#                             fills OSM gaps; requires CELL 2d to run first)
# USE_NDSM_HEIGHTS = False -> use OSM height= tags only (original behaviour)
USE_NDSM_HEIGHTS    = True
NDSM_BUILDING_MIN_M = 2.0    # ignore nDSM below this (noise / ground return)
NDSM_PROVIDER       = 'auto' # 'auto' = detect from coordinates (recommended)
                              # 'ea' (England) | 'usgs' (USA) | 'opentopo' (global)

# ── Building parameters ───────────────────────────────
MIN_BUILDING_AREA_M2  =  0.0
CITY_MIN_HEIGHT_M     =  0.0
HEIGHT_PER_LEVEL_M    =  3.0
DEFAULT_HEIGHT_M      =  8.0

# ── Roof shape ───────────────────────────────────────
ROOF_PITCH_DEG        = 30.0
ROOF_MAX_RIDGE_M      =  6.0

# ── OSM extra features ────────────────────────────────
INCLUDE_ROADS       = True
INCLUDE_WATER       = True    # River Trent + lakes → itu_water PLY (specular reflection)
INCLUDE_VEGETATION  = True    # OSM forests/parks/scrub → itu_vegetation PLY + P.833 post-processing
INCLUDE_BRIDGES     = True    # road/rail bridges → itu_concrete PLY (hard diffraction edges)
INCLUDE_EMBANKMENTS      = False   # disabled: query fetches ALL railway lines, not just embankment=yes
INCLUDE_ROAD_EMBANKMENTS = False   # disabled: road embankment=yes tagging sparse in London OSM
INCLUDE_ROAD_CUTTINGS    = False   # highway cuttings — V-trench shadows near-TX receivers, disable
INCLUDE_HWY_BRIDGES      = True    # highway bridge decks (bridge=yes) → itu_concrete PLY
VEG_3D_GEOMETRY     = True   # True  -> hollow canopy shell (walls + roof cap) — blocks horizontal rays correctly
                              # False -> flat ground patches + ITU-R P.833 post-hoc (no horizontal ray blocking)
VEG_MIN_AREA_M2     = 50.0
VEG_CANOPY_M        = 8.0    # default canopy height (m) when OSM/nDSM absent — UK mixed forest avg
VEG_CLASS_HEIGHT_M  = {       # canopy height (m) by OSM tag — overrides VEG_CANOPY_M
    "forest" : 18.0,          # dense UK woodland (mature oak/ash)
    "wood"   : 16.0,          # mixed woodland
    "scrub"  :  4.0,          # scrubland / heath
    "orchard":  5.0,          # orchard trees
    "meadow" :  0.5,          # meadow — too low to extrude, will be skipped
    "park"             : 15.0,  # urban park (leisure=park) — mature London Plane/oak trees
    "nature_reserve"   : 16.0,  # nature reserve — mature woodland
    "garden"           :  8.0,  # formal garden / cemetery
    "recreation_ground": 10.0,  # sports ground with boundary trees
    "allotments"       :  3.0,  # allotments — low vegetation
}
# Internal vegetation grid for large parks (fills interior with canopy columns so
# rays crossing the park mid-section are blocked, not just the perimeter shell).
# Set to 0 to disable (perimeter-only hollow shell, same as before).
VEG_GRID_CELL_M    = 30.0   # grid cell size (m) for internal subdivision of large parks
VEG_GRID_MIN_AREA  = 5000.0 # only subdivide polygons larger than this (m²) — Regent's Park ~1.6M m²
INCLUDE_TREES       = True    # individual OSM trees (natural=tree) → itu_vegetation disks at canopy height (ITU-R P.833-10)
TREE_DEFAULT_HT_M   = 8.0    # canopy height when OSM height= tag and nDSM both absent
TREE_DISK_RADIUS_M  = 3.0    # disk radius for individual tree canopy representation
USE_BUILDING_AGE    = True    # OSM start_date= tag → pre-1940=brick, 1940-1980=concrete, post-1980=type-based (Jansen et al. 2019)
INCLUDE_RAILWAYS    = True    # railway=rail/tram tracks → itu_metal PLY (steel rail specular reflector)
INCLUDE_BARRIERS    = True    # barrier=wall/fence/noise_barrier → itu_concrete/itu_metal PLY
BARRIER_MIN_LEN_M   = 10.0   # skip barrier segments shorter than this (removes noise)
INCLUDE_PYLONS       = True   # power=tower/pole → itu_metal PLY (metal diffraction edges, [DeE04])
INCLUDE_MASTS        = True   # man_made=mast/communications_tower → itu_metal PLY
INCLUDE_CHIMNEYS     = True   # man_made=chimney → itu_concrete PLY (industrial stacks)
INCLUDE_WATER_TOWERS = True   # man_made=water_tower → itu_metal PLY
INCLUDE_STORAGE_TANKS= True   # man_made=storage_tank → itu_metal PLY (curved backscatterers [DeE04])
INCLUDE_STADIUMS     = True   # leisure=stadium → itu_metal PLY (large roof structures [Xia24])
INCLUDE_SUBSTATIONS  = True   # power=substation → itu_metal PLY (fenced transformer enclosures)
INCLUDE_CAR_PARKS    = True   # building=parking / parking=multi-storey → itu_concrete PLY
INCLUDE_COOLING_TOWERS = True # man_made=cooling_tower → itu_concrete PLY (large cylindrical reflectors)
EXCLUDE_TUNNELS       = True    # skip road/rail ways tagged tunnel=yes (underground — not surface obstacles)
INCLUDE_FUEL_CANOPIES = True    # amenity=fuel → itu_metal flat canopy roof at +5m (strong specular reflector)
INCLUDE_BUS_STATIONS  = True    # amenity=bus_station → itu_metal extruded 6m (large covered urban structure)
INCLUDE_SURFACE_PARKS = True   # surface car parks — destructive interference at long range, disable for 915 MHz
INCLUDE_GREENHOUSES   = True    # building=greenhouse/glasshouse → itu_glass extruded (RF-transparent vs brick)

# ── V3 / V4 / V5 high-frequency feature flags ──────────────────────────
# V3 (2.8-6 GHz) : INCLUDE_ROOFTOP_EQUIPMENT
# V4 (6-28 GHz)  : V3 + INCLUDE_PARKED_VEHICLES + INCLUDE_STREET_LAMPS
# V5 (28-60 GHz) : V4 + INCLUDE_BODY_PHANTOMS + INCLUDE_SHELTERS
INCLUDE_ROOFTOP_EQUIPMENT = False  # HVAC/plant-room +2.5m on commercial/industrial roofs
                                    # ITU-R P.2040-2: itu_metal (er=1, sigma=1e7)
ROOFTOP_EQUIP_HEIGHT_M    = 2.5    # height of HVAC box above roof level
ROOFTOP_EQUIP_FRACTION    = 0.25   # fraction of roof area covered (footprint scale)
ROOFTOP_BLDG_TYPES        = {'commercial','retail','industrial','office','supermarket',
                              'warehouse','factory','hospital','university','school'}

INCLUDE_PARKED_VEHICLES   = False  # box primitives along OSM parking lanes
                                    # ITU-R P.2040-2: itu_metal (steel body proxy)
VEHICLE_LENGTH_M          = 4.5
VEHICLE_WIDTH_M           = 1.8
VEHICLE_HEIGHT_M          = 1.4
VEHICLE_SPACING_M         = 6.0
VEHICLE_OFFSET_M          = 2.5

INCLUDE_STREET_LAMPS      = False  # cylinder lamp posts on highway edges
                                    # ITU-R P.2040-2: itu_metal (steel pole)
LAMP_HEIGHT_M             = 6.0
LAMP_RADIUS_M             = 0.05
LAMP_SPACING_M            = 35.0
LAMP_ROAD_OFFSET_M        = 2.5
LAMP_ROAD_TYPES           = {'primary','secondary','tertiary','residential','unclassified'}

INCLUDE_BODY_PHANTOMS     = False  # cylinder human-body proxy at each RX position
                                    # ITU-R P.1238 / P.2040-2: itu_concrete proxy
                                    # 15-20 dB blockage at 60 GHz (V5 only)
PHANTOM_HEIGHT_M          = 1.7
PHANTOM_RADIUS_M          = 0.2

INCLUDE_SHELTERS          = False  # bus shelters / kiosks
                                    # ITU-R P.2040-2: itu_metal roof + itu_glass sides
SHELTER_HEIGHT_M          = 2.5
SHELTER_DEPTH_M           = 1.5
SHELTER_WIDTH_M           = 3.0

# ── Exclude trivial building types ──────────────────────
EXCLUDE_BUILDING_TYPES = set()

# ── TX / RX parameters ──────────────────────────────
FREQUENCY_HZ         = 915.95e6
TX_AGL_M             = 25.0  # tx_center height, transmitter_positions.csv
RX_AGL_M             =  1.5
TX_CONDUCTED_DBM     = 49.0
TX_ANTENNA_GAIN_DBI  =  1.3
RX_EXTRA_GAIN_DB     = -7.8
SITE_CORRECTION_DB   =  0.0
ANTENNA_PATTERN      = 'donut'

# ── Auto-detect terrain/nDSM provider from scene centre ─────────────────────
def _detect_terrain_provider(lat, lon):
    """Return (TERRAIN_SOURCE, NDSM_PROVIDER) based on scene centre."""
    if 49.5 <= lat <= 55.9 and -6.5 <= lon <= 2.1:   # England — EA LiDAR
        return 'ea_lidar', 'ea'
    if 24.0 <= lat <= 50.0 and -125.0 <= lon <= -66.0:  # contiguous USA — USGS
        return 'usgs_1m', 'usgs'
    return 'srtm', 'opentopo'                           # global fallback — SRTM 30m

_scene_lat = (SCENE_NORTH + SCENE_SOUTH) / 2
_scene_lon = (SCENE_EAST  + SCENE_WEST)  / 2
if TERRAIN_SOURCE == 'auto' or NDSM_PROVIDER == 'auto':
    _ts, _np = _detect_terrain_provider(_scene_lat, _scene_lon)
    if TERRAIN_SOURCE == 'auto': TERRAIN_SOURCE = _ts
    if NDSM_PROVIDER  == 'auto': NDSM_PROVIDER  = _np
    print(f'  Terrain provider: {TERRAIN_SOURCE}  |  nDSM provider: {NDSM_PROVIDER}')

# ╔══════════════════════════════════════════════════════════╗
# ║                    PATH CONFIGURATION                    ║
# ╚══════════════════════════════════════════════════════════╝

_ROOT       = os.environ.get('RT_BASE_DIR', os.path.join(os.path.expanduser('~'), 'sionna_rt', 'london_ofcom_915mhz_dem'))
BASE_DIR    = _ROOT
# ── Scene version folders ─────────────────────────────────
# scene_v2_infra/    : baseline (buildings + roads + infra)  — 915 MHz
# scene_v3_enhanced/ : V3 (+ rooftop equipment, nDSM)        — 2.8-6 GHz
# scene_v4_enhanced/ : V4 (V3 + parked vehicles + lamps)     — 6-28 GHz
# scene_v5_enhanced/ : V5 (V4 + body phantoms + shelters)    — 28-60 GHz
# Scene directory — change suffix for ablation variants:
#   'scene_v4_full'    → all features (baseline)
#   'scene_v4_a1'      → ablation A1: no barriers/substations/surface_parks
_SCENE_SUFFIX = globals().get('SCENE_SUFFIX', 'v4_full')
SCENE_DIR   = os.path.join(_ROOT, f'scene_{_SCENE_SUFFIX}')
MESH_DIR    = os.path.join(_ROOT, f'scene_{_SCENE_SUFFIX}', 'meshes')
MERGED_DIR  = os.path.join(_ROOT, 'scene', 'meshes_roads')
# ── LiDAR file paths ─────────────────────────────────────
# Not used for flat terrain (FLAT_TERRAIN   = False)
LIDAR_DIR   = _ROOT
EA_DTM_TIFF = os.path.join(LIDAR_DIR, 'dem.tif')
EA_DSM_TIFF = os.path.join(LIDAR_DIR, 'lidar_dsm.tif')
NDSM_TIFF   = os.path.join(_ROOT, 'ndsm.tif')

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(MESH_DIR,   exist_ok=True)

# ── ABLATION FLAGS — set False to exclude feature groups from the scene ──────
# Ablation A1: disable features suspected to increase RMSE in v4_full
INCLUDE_BARRIERS       = globals().get('INCLUDE_BARRIERS',       True)   # metal/concrete walls
INCLUDE_SUBSTATIONS    = globals().get('INCLUDE_SUBSTATIONS',    True)   # 183 metal boxes
INCLUDE_SURFACE_PARKS  = globals().get('INCLUDE_SURFACE_PARKS',  True)   # 1185 asphalt surfaces
INCLUDE_CAR_PARKS      = globals().get('INCLUDE_CAR_PARKS',      True)   # multi-storey car parks
INCLUDE_EMBANKMENTS    = globals().get('INCLUDE_EMBANKMENTS',    True)   # railway embankments
# To run ablation A1 (disable top suspects), set these before running Cell 4:
#   INCLUDE_BARRIERS=False; INCLUDE_SUBSTATIONS=False; INCLUDE_SURFACE_PARKS=False

print('Config loaded.')
print(f'  Scenario    : {SCENARIO_NAME}')
print(f'  Scene bbox  : lon [{SCENE_WEST}, {SCENE_EAST}]')
print(f'                lat [{SCENE_SOUTH}, {SCENE_NORTH}]')
print(f'  Output      : {SCENE_DIR}')
print(f'  nDSM heights: {USE_NDSM_HEIGHTS}  (provider={NDSM_PROVIDER})')
print(f'  Veg 3D geom : {VEG_3D_GEOMETRY}')
if not FLAT_TERRAIN:
    print(f'  Terrain src : {TERRAIN_SOURCE}  ({TERRAIN_GRID_N}x{TERRAIN_GRID_N} grid)')


In [ ]:
# ============================================================
# CELL 1 — IMPORTS & DEPENDENCIES
# ============================================================
import os, math, time, json, struct, warnings
import numpy as np
import requests
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyproj import Transformer
import shapely.geometry as sg
import shapely.ops as so
from shapely.geometry import Polygon, MultiPolygon, box

# Optional: rasterio for GeoTIFF reading
try:
    import rasterio
    from rasterio.transform import from_bounds
    _HAS_RASTERIO = True
except ImportError:
    _HAS_RASTERIO = False
    print('⚠  rasterio not found — falling back to PIL for GeoTIFF tiles')

# PIL for fallback
try:
    from PIL import Image
    _HAS_PIL = True
except ImportError:
    _HAS_PIL = False

# osmnx for OSM data
try:
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.log_console = False
    _HAS_OSMNX = True
except ImportError:
    _HAS_OSMNX = False
    print('⚠  osmnx not found — install with: pip install osmnx')

# trimesh for PLY export
try:
    import trimesh
    _HAS_TRIMESH = True
except ImportError:
    _HAS_TRIMESH = False
    print('⚠  trimesh not found — install with: pip install trimesh')

# Coordinate transformers
to_utm   = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
to_wgs84 = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene bbox in UTM
sw_utm = to_utm.transform(SCENE_WEST,  SCENE_SOUTH)
ne_utm = to_utm.transform(SCENE_EAST,  SCENE_NORTH)
center_utm = ((sw_utm[0]+ne_utm[0])/2, (sw_utm[1]+ne_utm[1])/2)
center_lon, center_lat = to_wgs84.transform(*center_utm)

print(f'UTM SW : {sw_utm[0]:.1f}, {sw_utm[1]:.1f}')
print(f'UTM NE : {ne_utm[0]:.1f}, {ne_utm[1]:.1f}')
print(f'Center : ({center_lon:.5f}, {center_lat:.5f})')
print(f'Size   : {(ne_utm[0]-sw_utm[0])/1000:.2f} km × {(ne_utm[1]-sw_utm[1])/1000:.2f} km')

## GPS → Scene Coordinate Reference

All geographic coordinates are converted WGS84 (lon, lat) → projected CRS (set by `PROJECTION_CRS` in CELL 0) → scene-local (x, y) metres.

- **Projection:** `pyproj.Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)`
  - `PROJECTION_CRS = 'bng'` (default) → EPSG:27700 (British National Grid) — native UK Ordnance Survey grid, matches EA LiDAR WCS tiles directly
  - `PROJECTION_CRS = 'utm30n'` → EPSG:32630 (UTM Zone 30N) — auto-detected fallback for portability
- **Local origin:** Scene bounding-box centre (`center_utm`) in the projected CRS
- **x = East, y = North, z = Up**

Must match `PROJECTION_CRS` in `sionna2_915mhz_dem_simulation_london.ipynb` CELL 1 exactly — same TX coordinates, same projection method.

In [ ]:
# ============================================================
# CELL 2 — AWS ELEVATION TILES → HEIGHTMAP
# ============================================================
# Only needed when TERRAIN_SOURCE = 'aws_dem'
# Skip this cell if TERRAIN_SOURCE = 'ea_lidar' (use CELL 2b–2e instead)
# ============================================================
if globals().get('TERRAIN_SOURCE', 'aws_dem') != 'aws_dem':
    print(f"TERRAIN_SOURCE='{TERRAIN_SOURCE}' — skipping AWS terrain download.")
    print("For EA LiDAR terrain run CELL 2b → 2c → 2d → 2e → CELL 3.")
else:
 pass  # guard end — rest of cell runs only for aws_dem
# ============================================================
# Downloads GeoTIFF tiles from the public AWS elevation-tiles-prod bucket
# (same source used by Mapzen/Terrarium, Cesium, sionna-large-radio-maps).
# No credentials needed — anonymous public access.
# URL: s3://elevation-tiles-prod/geotiff/{z}/{x}/{y}.tif
# Values are direct metres ASL (float32 GeoTIFF, no conversion formula).

AWS_BASE = 'https://s3.amazonaws.com/elevation-tiles-prod/geotiff'

def _lon2tile(lon, z):
    return int(math.floor((lon + 180) / 360 * 2**z))

def _lat2tile(lat, z):
    lat_r = math.radians(lat)
    return int(math.floor((1 - math.log(math.tan(lat_r) + 1/math.cos(lat_r)) / math.pi) / 2 * 2**z))

def _tile2lon(x, z):
    return x / 2**z * 360 - 180

def _tile2lat(y, z):
    n = math.pi - 2 * math.pi * y / 2**z
    return math.degrees(math.atan(math.sinh(n)))

def _download_tile(z, x, y, retries=3):
    url = f'{AWS_BASE}/{z}/{x}/{y}.tif'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            buf = BytesIO(r.content)
            if _HAS_RASTERIO:
                with rasterio.open(buf) as ds:
                    data = ds.read(1).astype(np.float32)
            elif _HAS_PIL:
                # PIL may not read float GeoTIFF correctly — warn
                img = Image.open(buf)
                data = np.array(img, dtype=np.float32)
            else:
                raise RuntimeError('Need rasterio or PIL to read GeoTIFF tiles')
            # Resample to 512×512 if needed
            if data.shape != (512, 512):
                from scipy.ndimage import zoom as nd_zoom
                fx = 512 / data.shape[0]; fy = 512 / data.shape[1]
                data = nd_zoom(data, (fx, fy), order=1).astype(np.float32)
            return data
        except Exception as e:
            if attempt == retries - 1:
                print(f'  ⚠ tile {z}/{x}/{y} failed: {e} — using zeros')
                return np.zeros((512, 512), dtype=np.float32)
            time.sleep(2 ** attempt)

# Compute tile range for scene bbox
x0 = _lon2tile(SCENE_WEST,  TILE_ZOOM)
x1 = _lon2tile(SCENE_EAST,  TILE_ZOOM)
y0 = _lat2tile(SCENE_NORTH, TILE_ZOOM)  # north = smaller y in tile coords
y1 = _lat2tile(SCENE_SOUTH, TILE_ZOOM)
n_cols = x1 - x0 + 1
n_rows = y1 - y0 + 1
print(f'Tile range : x=[{x0},{x1}] y=[{y0},{y1}]  →  {n_cols}×{n_rows} = {n_cols*n_rows} tiles')

# Download in parallel
tile_mosaic = np.zeros((n_rows * 512, n_cols * 512), dtype=np.float32)
jobs = [(TILE_ZOOM, x0+col, y0+row, col, row)
        for row in range(n_rows) for col in range(n_cols)]

print(f'Downloading {len(jobs)} tiles ...')
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    futures = {ex.submit(_download_tile, z, x, y): (col, row)
               for z, x, y, col, row in jobs}
    for i, fut in enumerate(as_completed(futures)):
        col, row = futures[fut]
        tile_mosaic[row*512:(row+1)*512, col*512:(col+1)*512] = fut.result()
        if (i+1) % max(1, len(jobs)//4) == 0:
            print(f'  {i+1}/{len(jobs)} tiles done')

print(f'Download done in {time.time()-t0:.1f}s')

# Extent of the mosaic in WGS84
_map_min_lon = _tile2lon(x0,   TILE_ZOOM)
_map_max_lon = _tile2lon(x1+1, TILE_ZOOM)
_map_max_lat = _tile2lat(y0,   TILE_ZOOM)   # y0 is north
_map_min_lat = _tile2lat(y1+1, TILE_ZOOM)   # y1+1 is south
_mosaic_h, _mosaic_w = tile_mosaic.shape

print(f'Mosaic size : {_mosaic_w}×{_mosaic_h} px')
print(f'Mosaic lon  : [{_map_min_lon:.5f}, {_map_max_lon:.5f}]')
print(f'Mosaic lat  : [{_map_min_lat:.5f}, {_map_max_lat:.5f}]')
print(f'Elevation   : [{tile_mosaic.min():.1f}, {tile_mosaic.max():.1f}] m ASL')

def height_from_wgs84(lon, lat):
    """Bilinear interpolation from mosaic. Returns elevation in metres ASL."""
    u = (lon - _map_min_lon) / (_map_max_lon - _map_min_lon) * (_mosaic_w - 1)
    v = (1 - (lat - _map_min_lat) / (_map_max_lat - _map_min_lat)) * (_mosaic_h - 1)
    u = np.clip(u, 0, _mosaic_w - 1)
    v = np.clip(v, 0, _mosaic_h - 1)
    x0i, y0i = int(np.floor(u)), int(np.floor(v))
    x1i, y1i = min(x0i+1, _mosaic_w-1), min(y0i+1, _mosaic_h-1)
    fx, fy = u - x0i, v - y0i
    h = (tile_mosaic[y0i, x0i] * (1-fx) * (1-fy)
       + tile_mosaic[y0i, x1i] *    fx  * (1-fy)
       + tile_mosaic[y1i, x0i] * (1-fx) *    fy
       + tile_mosaic[y1i, x1i] *    fx  *    fy)
    return float(h)

def height_from_utm(easting, northing):
    """Height at UTM coordinates."""
    lon, lat = to_wgs84.transform(easting, northing)
    return height_from_wgs84(lon, lat)

# Scene centre elevation = local z=0 reference
origin_elev_asl = height_from_wgs84(center_lon, center_lat)
print(f'\nScene centre elevation : {origin_elev_asl:.2f} m ASL  (= local z=0)')

def local_z(lon, lat):
    """Returns local z in metres relative to scene centre elevation."""
    return height_from_wgs84(lon, lat) - origin_elev_asl

## CELL 2b — Download EA LiDAR DTM

Downloads the Environment Agency 1 m Composite DTM (bare-earth terrain) for the
scene bbox from the EA WCS service (free, no credentials, England only).

- Skips automatically if `FLAT_TERRAIN=True` or `dem.tif` already exists
- If you have pre-downloaded tiles, run **CELL 2e** to merge them first
- Coverage check: run **CELL 2c** after this cell


In [ ]:
# ============================================================
# CELL 2b — EA LiDAR DTM Auto-Download  (skip if FLAT_TERRAIN=True)
# ============================================================
# Downloads Environment Agency 1m Composite DTM for scene bbox.
# Source: EA WCS service (free, no credentials, England only).
# CRS: EPSG:27700 (British National Grid) for request, output GeoTIFF.
# Skip this cell entirely when FLAT_TERRAIN=True.
#
# The WCS server enforces a max image size per GetCoverage request, so
# large bboxes (anything beyond a couple of km per side at 1m res) 404.
# This cell auto-tiles the bbox into WCS_TILE_KM-sized chunks, downloads
# each separately, then mosaics them into EA_DTM_TIFF with rasterio.
# ============================================================

WCS_TILE_KM = 2.0   # max tile size per WCS GetCoverage request (km)

if globals().get('FLAT_TERRAIN', True):
    print('FLAT_TERRAIN=True — skipping EA LiDAR download.')
    print('Set FLAT_TERRAIN=False in CELL 0 and re-run this cell for real terrain.')
else:
    import requests, os
    from pyproj import Transformer

    print('=' * 60)
    print('CELL 2b — EA LiDAR DTM Download (1m resolution, tiled)')
    print('=' * 60)

    if os.path.exists(EA_DTM_TIFF):
        print(f'Already downloaded: {EA_DTM_TIFF}')
        print('Delete the file and re-run to force re-download.')
    else:
        # Convert scene bbox WGS84 -> BNG (EPSG:27700) for EA WCS request
        _wgs_to_bng = Transformer.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
        _e_min, _n_min = _wgs_to_bng.transform(SCENE_WEST,  SCENE_SOUTH)
        _e_max, _n_max = _wgs_to_bng.transform(SCENE_EAST,  SCENE_NORTH)

        # Add 200m buffer so terrain edge doesn't clip buildings
        _buf = 200
        _e_min -= _buf; _n_min -= _buf
        _e_max += _buf; _n_max += _buf

        print(f'BNG bbox: E[{_e_min:.0f}, {_e_max:.0f}]  N[{_n_min:.0f}, {_n_max:.0f}]')
        print(f'Area    : {(_e_max-_e_min)/1000:.1f} km x {(_n_max-_n_min)/1000:.1f} km')

        _tile_m = WCS_TILE_KM * 1000.0
        import math as _math
        _n_cols = max(1, _math.ceil((_e_max - _e_min) / _tile_m))
        _n_rows = max(1, _math.ceil((_n_max - _n_min) / _tile_m))
        print(f'Tiling  : {_n_cols} x {_n_rows} = {_n_cols*_n_rows} tiles of {WCS_TILE_KM} km')

        _WCS_BASE = (
            'https://environment.data.gov.uk/spatialdata/lidar-composite-dtm-1m/wcs'
            '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
            '&COVERAGEID=LIDAR_Composite_DTM_1m'
        )

        _tile_dir = os.path.join(os.path.dirname(EA_DTM_TIFF), '_dtm_tiles')
        os.makedirs(_tile_dir, exist_ok=True)
        _tile_paths = []
        _failed = []

        for _row in range(_n_rows):
            for _col in range(_n_cols):
                _te_min = _e_min + _col * _tile_m
                _te_max = min(_e_min + (_col + 1) * _tile_m, _e_max)
                _tn_min = _n_min + _row * _tile_m
                _tn_max = min(_n_min + (_row + 1) * _tile_m, _n_max)
                _tile_path = os.path.join(_tile_dir, f'dtm_tile_{_row}_{_col}.tif')
                _tile_paths.append(_tile_path)
                if os.path.exists(_tile_path) and os.path.getsize(_tile_path) > 0:
                    print(f'  tile [{_row},{_col}] already downloaded, skipping')
                    continue
                _url = (
                    _WCS_BASE +
                    f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_te_min:.0f},{_te_max:.0f})'
                    f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_tn_min:.0f},{_tn_max:.0f})'
                    '&FORMAT=image/tiff'
                )
                print(f'  tile [{_row},{_col}]  E[{_te_min:.0f},{_te_max:.0f}] N[{_tn_min:.0f},{_tn_max:.0f}] ...', end=' ', flush=True)
                try:
                    _r = requests.get(_url, timeout=120, stream=True)
                    _r.raise_for_status()
                    with open(_tile_path, 'wb') as _f:
                        for _chunk in _r.iter_content(chunk_size=1024 * 1024):
                            _f.write(_chunk)
                    print(f'OK ({os.path.getsize(_tile_path)//1024} KB)')
                except Exception as _e:
                    print(f'FAILED: {_e}')
                    _failed.append((_row, _col, str(_e)))

        if _failed:
            print(f'\n{len(_failed)}/{len(_tile_paths)} tiles failed:')
            for _row, _col, _e in _failed:
                print(f'  [{_row},{_col}]: {_e}')
            print('Re-run this cell to retry only the missing tiles (existing ones are skipped).')
            print('If a smaller WCS_TILE_KM still 404s, the service may be down - use CELL 2b-ZIP or CELL 2b-SCRIPT instead.')
        else:
            _ok_tiles = [p for p in _tile_paths if os.path.exists(p) and os.path.getsize(p) > 0]
            print(f'\nAll {len(_ok_tiles)} tiles downloaded. Mosaicking ...')
            import rasterio as _rio
            from rasterio.merge import merge as _rio_merge
            _srcs = [_rio.open(p) for p in _ok_tiles]
            _mosaic, _out_trans = _rio_merge(_srcs)
            _meta = _srcs[0].meta.copy()
            _meta.update({'height': _mosaic.shape[1], 'width': _mosaic.shape[2], 'transform': _out_trans})
            os.makedirs(os.path.dirname(EA_DTM_TIFF), exist_ok=True)
            with _rio.open(EA_DTM_TIFF, 'w', **_meta) as _dst:
                _dst.write(_mosaic)
            for _s in _srcs:
                _s.close()
            print(f'Saved: {EA_DTM_TIFF}  ({os.path.getsize(EA_DTM_TIFF)//1024//1024} MB)')

            if _HAS_RASTERIO:
                with _rio.open(EA_DTM_TIFF) as _ds:
                    print(f'CRS     : {_ds.crs}')
                    print(f'Shape   : {_ds.height} x {_ds.width} px')
                    print(f'Res     : {_ds.res[0]:.1f} m/px')
                    _data = _ds.read(1)
                    print(f'Z range : {float(_data.min()):.1f} - {float(_data.max()):.1f} m ASL')
            print('\nEA LiDAR DTM ready. Now run CELL 3 to build terrain mesh.')


## CELL 2c — DTM Coverage Check

Verifies the downloaded DTM fully encloses the scene bbox before expensive
mesh generation. Reports elevation range and NoData fraction.
Skips when `FLAT_TERRAIN=True`.


In [ ]:
# ============================================================
# CELL 2c — DTM COVERAGE CHECK  (verify dem.tif encloses the AOI)
# ============================================================
# Confirms the local EA/AWS GeoTIFF (EA_DTM_TIFF) actually covers the
# scene bbox BEFORE CELL 3 samples it into terrain.ply. Reprojects the
# raster bounds to WGS84 for an apples-to-apples comparison and reports
# elevation range + NoData fraction. Skip when FLAT_TERRAIN=True.
# ============================================================
if globals().get('FLAT_TERRAIN', True):
    print("FLAT_TERRAIN=True - no DTM needed, skipping coverage check.")
elif not os.path.exists(globals().get('EA_DTM_TIFF', '')):
    print(f"No DTM found at: {globals().get('EA_DTM_TIFF','')}")
    print("Copy your GeoTIFF there, e.g.:")
    print(f"  cp /path/to/your_terrain.tif {globals().get('EA_DTM_TIFF','dem.tif')}")
else:
    import rasterio
    from rasterio.warp import transform_bounds
    print("=" * 60)
    print("CELL 2c - DTM COVERAGE CHECK")
    print("=" * 60)
    with rasterio.open(EA_DTM_TIFF) as ds:
        print(f"File       : {EA_DTM_TIFF}")
        print(f"CRS        : {ds.crs}")
        print(f"Size       : {ds.width} x {ds.height} px   res {ds.res}")
        print(f"Bounds(raw): {tuple(round(b,1) for b in ds.bounds)}")
        w, s, e, n = transform_bounds(ds.crs, "EPSG:4326", *ds.bounds, densify_pts=21)
        print(f"Bounds WGS84: lon[{w:.6f}, {e:.6f}]  lat[{s:.6f}, {n:.6f}]")
        print(f"AOI    WGS84: lon[{SCENE_WEST:.6f}, {SCENE_EAST:.6f}]  "
              f"lat[{SCENE_SOUTH:.6f}, {SCENE_NORTH:.6f}]")
        _tol = 1e-4  # ~10m tolerance for floating point
        _covers = (w <= SCENE_WEST+_tol and e >= SCENE_EAST-_tol and
                   s <= SCENE_SOUTH+_tol and n >= SCENE_NORTH-_tol)
        if _covers:
            print("COVERS AOI : YES - DTM fully encloses the scene bbox.")
        else:
            _short = []
            if w > SCENE_WEST+_tol:  _short.append("West")
            if e < SCENE_EAST-_tol:  _short.append("East")
            if s > SCENE_SOUTH+_tol: _short.append("South")
            if n < SCENE_NORTH-_tol: _short.append("North")
            print(f"COVERS AOI : NO - DTM short on: {', '.join(_short)}")
            print("  -> use a larger tile or shrink the AOI bbox in CELL 0.")
        try:
            import numpy as np
            from rasterio.warp import transform_bounds as _tb
            from rasterio.windows import from_bounds as _wfb
            # Stats over the FULL raster are meaningless here -- the canvas
            # is usually far bigger than the actual AOI (e.g. covers ~3x
            # the area), so the rest reads as NoData and would dilute /
            # mask out everything. Crop to just the AOI window first.
            _aoi_x0, _aoi_y0, _aoi_x1, _aoi_y1 = _tb(
                "EPSG:4326", ds.crs, SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH)
            _win = _wfb(_aoi_x0, _aoi_y0, _aoi_x1, _aoi_y1, transform=ds.transform)
            _aoi_arr = ds.read(1, window=_win)
            _nd = ds.nodata
            _valid_mask = np.isfinite(_aoi_arr) & (np.abs(_aoi_arr) < 1e30)
            if _nd is not None:
                _valid_mask &= ~np.isclose(_aoi_arr, _nd)
            _valid = _aoi_arr[_valid_mask]
            _ndf = 100.0 * (1.0 - _valid.size / _aoi_arr.size) if _aoi_arr.size else 100.0
            print(f"NoData frac (within AOI): {_ndf:.1f}%")
            if _valid.size == 0:
                raise RuntimeError(
                    "DTM has ZERO valid elevation pixels inside the AOI -- the merge is "
                    "broken or empty. Delete dem.tif and re-run CELL 2b-SCRIPT/CELL 2e "
                    "before continuing; do not proceed to CELL 3 with this file.")
            print(f"Elev range (within AOI): {float(_valid.min()):.1f} .. {float(_valid.max()):.1f} m ASL")
            if _ndf > 5.0:
                print("  ! >5% NoData inside the AOI - terrain may have holes in the AOI.")
        except RuntimeError:
            raise
        except Exception as _e:
            print(f"  (elevation/NoData stats unavailable: {_e})")


## CELL 2b-ZIP — Bulk Zip-Tile Download + Auto-Extract (fallback for WCS failures / large areas)

The EA WCS GetCoverage request in CELL 2b has a server-side image-size limit, so it can fail or time out for large `RADIUS_KM` areas. This cell is the alternative path: it downloads 1km LiDAR tiles as `.zip` archives (the same files the [DEFRA Data Download portal](https://environment.data.gov.uk/DefraDataDownload/?Mode=survey) serves when you tick tiles by hand), extracts the `.tif` rasters from each zip straight into `LIDAR_DIR`, then deletes the zip.

**You provide the zip URLs** — browse the portal for your bbox, right-click each tile's download link, and paste the URLs into `ZIP_TILE_URLS` below (or point `ZIP_TILE_DIR` at a folder of zips you already downloaded by hand — no internet needed in that case).

After this cell, run **CELL 2e** to merge the extracted tiles into `dem.tif` / `lidar_dsm.tif` (it already globs `*_DTM_1m.tif` / `*_FZ_DSM_1m.tif` in `LIDAR_DIR`, so no extra config needed).

In [ ]:
# ============================================================
# CELL 2b-ZIP — Bulk zip-tile download + auto-extract
# ============================================================
# Fallback to CELL 2b/2d when the EA WCS GetCoverage request fails
# or the area is too large for a single WCS call. Downloads .zip
# tile archives, extracts the .tif files into LIDAR_DIR, then runs
# CELL 2e to merge them into dem.tif / lidar_dsm.tif.
# ============================================================
import os, glob, zipfile, requests

# ── Configure one or both of these ──────────────────────────
# A) Direct download: paste tile zip URLs from the DEFRA Data Download
#    portal (https://environment.data.gov.uk/DefraDataDownload/?Mode=survey
#    -> LIDAR Composite DTM/DSM -> 1m -> draw your bbox -> copy each tile link).
ZIP_TILE_URLS = [
    # 'https://environment.data.gov.uk/UserDownloads/.../TQ38_DTM_1m.zip',
    # 'https://environment.data.gov.uk/UserDownloads/.../TQ38_FZ_DSM_1m.zip',
]

# B) Already downloaded by hand: point this at the folder containing
#    the .zip files (skips network access entirely).
ZIP_TILE_DIR = None   # e.g. os.path.expanduser('~/Downloads/london_lidar_zips')

os.makedirs(LIDAR_DIR, exist_ok=True)
_zip_paths = []

# -- Download remote zips into LIDAR_DIR -----------------------------
for _url in ZIP_TILE_URLS:
    _zname = os.path.basename(_url.split('?')[0])
    _zpath = os.path.join(LIDAR_DIR, _zname)
    if os.path.exists(_zpath):
        print(f'Already downloaded: {_zname}')
    else:
        print(f'Downloading {_zname} ...')
        _r = requests.get(_url, timeout=300, stream=True)
        _r.raise_for_status()
        with open(_zpath, 'wb') as _f:
            for _chunk in _r.iter_content(chunk_size=1024 * 1024):
                _f.write(_chunk)
        print(f'  saved {os.path.getsize(_zpath) // 1024} KB')
    _zip_paths.append(_zpath)

# -- Collect already-local zips -----------------------------------------
# Only pick up actual EA LiDAR composite tile zips (DTM / first-return DSM).
# Filters out unrelated downloads that may sit in the same folder, e.g.
# OS Terrain50 ('terr50_gagg_gb.zip') which is a different, 50m product
# that would corrupt the 1m merge if extracted alongside the real tiles.
if ZIP_TILE_DIR and os.path.isdir(ZIP_TILE_DIR):
    _zip_paths += sorted(
        glob.glob(os.path.join(ZIP_TILE_DIR, 'lidar_composite_dtm-*.zip'))
        + glob.glob(os.path.join(ZIP_TILE_DIR, 'lidar_composite_first_return_dsm-*.zip'))
    )

if not _zip_paths:
    print('No zip URLs configured and no ZIP_TILE_DIR set — nothing to do.')
    print('Fill in ZIP_TILE_URLS or ZIP_TILE_DIR above, or use CELL 2b/2d (WCS) instead.')
else:
    # -- Extract every .tif from every zip straight into LIDAR_DIR ----
    _extracted = []
    for _zpath in _zip_paths:
        print(f'Extracting {os.path.basename(_zpath)} ...')
        try:
            with zipfile.ZipFile(_zpath, 'r') as _zf:
                for _member in _zf.namelist():
                    if not _member.lower().endswith('.tif'):
                        continue
                    _out_name = os.path.basename(_member)  # flatten — ignore zip's internal folders
                    _out_path = os.path.join(LIDAR_DIR, _out_name)
                    if os.path.exists(_out_path):
                        print(f'  skip (exists): {_out_name}')
                        continue
                    with _zf.open(_member) as _src, open(_out_path, 'wb') as _dst:
                        _dst.write(_src.read())
                    _extracted.append(_out_path)
                    print(f'  -> {_out_name}')
        except zipfile.BadZipFile:
            print(f'  [WARN] not a valid zip — skipping: {_zpath}')

    print(f'\nExtracted {len(_extracted)} new GeoTIFF tile(s) into {LIDAR_DIR}')
    print('Next: run CELL 2e to merge tiles into dem.tif / lidar_dsm.tif,')
    print('      then CELL 2d to compute nDSM (if not already done).')


## CELL 2b-SCRIPT — Run Standalone LiDAR Extractor/Merger

Runs `extract_london_lidar.py` (repo root) which extracts the DTM/DSM
tiles from your `~/Downloads` zips and merges them into `dem.tif` /
`lidar_dsm.tif` directly — no dependency on notebook variable state.
Use this when CELL 2b's WCS download 404s.


In [ ]:
# ============================================================
# CELL 2b-SCRIPT — Run standalone LiDAR zip extractor/merger
# ============================================================
# Extracts DTM/DSM tiles from ~/Downloads zips and merges them
# into dem.tif / lidar_dsm.tif. Use this instead of CELL 2b-ZIP
# when the WCS download (CELL 2b) 404s and you already have the
# zip tiles downloaded manually from the DEFRA Data Download portal.
#
# Self-healing: auto-deletes dem.tif / lidar_dsm.tif if they were
# built by an older buggy merge -- either containing -inf/nan pixels,
# or entirely empty (all-NoData, from the -n flag bug) -- so the
# extractor's merge step regenerates them cleanly on this run.
# ============================================================
import numpy as np

for _f in (globals().get('EA_DTM_TIFF'), os.path.join(LIDAR_DIR, 'lidar_dsm.tif')):
    if _f and os.path.exists(_f):
        try:
            import rasterio as _rio_check
            with _rio_check.open(_f) as _ds_check:
                _arr_check = _ds_check.read(1)
                _bad = (not np.isfinite(_arr_check).all()) or (_arr_check == -9999).all()
                if _bad:
                    print(f'[CELL 2b-SCRIPT] {_f} is corrupted/empty -- deleting for clean regeneration.')
                    os.remove(_f)
        except Exception as _e:
            print(f'[CELL 2b-SCRIPT] could not check {_f}: {_e}')

!python3 extract_london_lidar.py


## CELL 2d — DSM Download + nDSM

Downloads EA LiDAR DSM, computes `nDSM = DSM − DTM` (height of above-ground objects). Used in CELL 4 to replace sparse OSM `height=` tags with measured building heights. Provider-abstracted via `NDSM_PROVIDER`.

**Run after CELL 2b.** Skip when `USE_NDSM_HEIGHTS=False`.


In [ ]:
# ============================================================
# CELL 2d — DSM DOWNLOAD + nDSM = DSM - DTM
# ============================================================
# Downloads LiDAR DSM alongside the DTM already fetched in
# CELL 2b, computes nDSM = DSM - DTM (height of above-ground
# objects), saves ndsm.tif.  Building heights in CELL 4 are
# sampled from nDSM instead of (sparse) OSM height= tags,
# filling clutter gaps automatically.
#
# Provider abstraction (NDSM_PROVIDER in CELL 0):
#   'ea'        -> Environment Agency WCS (England, free)
#   'usgs'      -> USGS 3DEP WCS (USA, free)
#   'opentopo'  -> OpenTopography REST API (global, API key needed)
# ============================================================

if not USE_NDSM_HEIGHTS:
    print('USE_NDSM_HEIGHTS=False — skipping DSM download.')
elif globals().get('FLAT_TERRAIN', True):
    print('FLAT_TERRAIN=True — skipping DSM download.')
else:
    import requests, os
    import numpy as np

    def _dsm_wcs_url(provider, e_min, e_max, n_min, n_max):
        if provider == 'ea':
            return (
                'https://environment.data.gov.uk/spatialdata/lidar-composite-dsm-1m/wcs'
                '?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
                '&COVERAGEID=LIDAR_Composite_DSM_1m'
                f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({e_min:.0f},{e_max:.0f})'
                f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({n_min:.0f},{n_max:.0f})'
                '&FORMAT=image/tiff'
            )
        elif provider == 'usgs':
            raise NotImplementedError(
                'USGS 3DEP: use '
                'https://elevation.nationalmap.gov/arcgis/services/3DEPElevation'
                '/ImageServer/WCSServer with WGS84 bbox. Set NDSM_PROVIDER="usgs".'
            )
        elif provider == 'opentopo':
            raise NotImplementedError(
                'OpenTopography: get API key at https://opentopography.org, '
                'then use their REST API. Set NDSM_PROVIDER="opentopo".'
            )
        else:
            raise ValueError(f'Unknown NDSM_PROVIDER: {provider!r}')

    print('=' * 60)
    print(f'CELL 2d — LiDAR DSM + nDSM  (provider={NDSM_PROVIDER})')
    print('=' * 60)

    if os.path.exists(NDSM_TIFF):
        print(f'nDSM already computed: {NDSM_TIFF}')
        print('Delete ndsm.tif and dsm.tif to force recompute.')
    else:
        if not os.path.exists(EA_DTM_TIFF):
            raise FileNotFoundError(f'DTM not found: {EA_DTM_TIFF} — run CELL 2b first.')

        from pyproj import Transformer as _TrDSM
        _wgs_bng = _TrDSM.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
        _e0, _n0 = _wgs_bng.transform(SCENE_WEST,  SCENE_SOUTH)
        _e1, _n1 = _wgs_bng.transform(SCENE_EAST,  SCENE_NORTH)
        _buf = 200
        _e0 -= _buf; _n0 -= _buf; _e1 += _buf; _n1 += _buf

        if not os.path.exists(EA_DSM_TIFF):
            _url = _dsm_wcs_url(NDSM_PROVIDER, _e0, _e1, _n0, _n1)
            print(f'Downloading DSM ({NDSM_PROVIDER.upper()}) ...')
            _r = requests.get(_url, timeout=180, stream=True)
            _r.raise_for_status()
            _total = 0
            with open(EA_DSM_TIFF, 'wb') as _f:
                for _chunk in _r.iter_content(chunk_size=1024*1024):
                    _f.write(_chunk)
                    _total += len(_chunk)
                    print(f'  {_total//1024//1024} MB ...', end='\r')
            print(f'\nSaved: {EA_DSM_TIFF}  ({os.path.getsize(EA_DSM_TIFF)//1024//1024} MB)')
        else:
            print(f'DSM already on disk: {EA_DSM_TIFF}')

        # ── nDSM = DSM - DTM, reprojected onto DTM grid ─────────────────
        import rasterio
        from rasterio.warp import reproject, Resampling

        print('Computing nDSM = DSM - DTM ...')
        with rasterio.open(EA_DTM_TIFF) as _dtm_ds:
            _dtm     = _dtm_ds.read(1).astype(np.float32)
            _profile = _dtm_ds.profile.copy()
            _nodata  = _dtm_ds.nodata or -9999.0

        with rasterio.open(EA_DSM_TIFF) as _dsm_ds:
            _dsm_aligned = np.empty_like(_dtm)
            reproject(
                source      = rasterio.band(_dsm_ds, 1),
                destination = _dsm_aligned,
                src_transform = _dsm_ds.transform,
                src_crs       = _dsm_ds.crs,
                dst_transform = _profile['transform'],
                dst_crs       = _profile['crs'],
                resampling    = Resampling.bilinear,
            )

        _ndsm = np.clip(_dsm_aligned - _dtm, 0.0, None)
        _ndsm[_dtm == _nodata] = 0.0

        _profile.update(dtype=rasterio.float32, nodata=0.0)
        with rasterio.open(NDSM_TIFF, 'w', **_profile) as _out:
            _out.write(_ndsm.astype(np.float32), 1)

        print(f'nDSM saved: {NDSM_TIFF}')
        print(f'  Height range: {_ndsm.min():.1f} – {_ndsm.max():.1f} m')
        print(f'  Pixels > 2 m : {(_ndsm > 2).sum():,}  '
              f'({100*(_ndsm > 2).mean():.1f}% of scene)')
        print(f'  Pixels > 5 m : {(_ndsm > 5).sum():,}  '
              f'({100*(_ndsm > 5).mean():.1f}% of scene)')


## CELL 2e — Merge LiDAR Tiles

Merges individual EA LiDAR 1km tiles (DTM and DSM) into single
GeoTIFFs using GDAL. Skips if merged files already exist and are valid
(non-zero data). Run once per campaign — takes ~1 min per merge.

**Run before CELL 2d.**


In [ ]:
# ============================================================
# CELL 2e — MERGE LIDAR TILES (DTM + DSM)
# ============================================================
# Merges all individual 1km EA LiDAR tiles in LIDAR_DIR into
# two single GeoTIFFs (DTM + DSM). Skips if merged files already
# exist and contain valid (non-zero) data.
# Requires: GDAL (gdal_merge.py on PATH)
# ============================================================
import os, glob
import numpy as np

def _merged_valid(path):
    if not os.path.exists(path):
        return False
    try:
        import rasterio
        with rasterio.open(path) as ds:
            arr = ds.read(1)
            mask = arr != (ds.nodata or -9999)
            return bool(mask.any() and arr[mask].max() > 1.0)
    except Exception:
        return False

def _merge_tiles(pattern, out_path, label):
    if _merged_valid(out_path):
        print(f'{label}: already merged and valid -> {os.path.basename(out_path)}')
        return
    tiles = sorted(glob.glob(os.path.join(LIDAR_DIR, pattern)))
    if not tiles:
        print(f'{label}: no tiles found matching {pattern} in {LIDAR_DIR}')
        return
    print(f'{label}: merging {len(tiles)} tiles -> {os.path.basename(out_path)} ...')
    tile_args = ' '.join(f'"{t}"' for t in tiles)
    # Deliberately no -n / -a_nodata here: passing -n with the source
    # tiles' extreme float32-min NoData sentinel made gdal_merge write
    # an entirely empty (all-NoData) output instead of the real tile
    # data. Plain gdal_merge correctly copies real elevation values, so
    # we sanitize the NoData sentinel ourselves afterwards in Python.
    cmd = (f'gdal_merge.py -o "{out_path}" -of GTiff '
           f'-co COMPRESS=LZW -co TILED=YES '
           f'{tile_args}')
    ret = os.system(cmd)
    if ret != 0:
        raise RuntimeError(f'gdal_merge.py failed (exit {ret}). Is GDAL installed?')
    import numpy as np
    import rasterio
    with rasterio.open(out_path) as ds:
        profile = ds.profile
        arr = ds.read(1)
    bad = ~np.isfinite(arr) | (np.abs(arr) > 1e30)
    n_bad = int(bad.sum())
    arr[bad] = -9999.0
    profile.update(nodata=-9999.0)
    with rasterio.open(out_path, 'w', **profile) as ds:
        ds.write(arr, 1)
    print(f'  sanitized {n_bad} bad/sentinel pixel(s) -> -9999, nodata set to -9999')
    valid = arr[arr != -9999.0]
    if valid.size == 0:
        raise RuntimeError(f'Merge produced empty file (no valid data): {out_path}')
    print(f'  -> {os.path.basename(out_path)}  '
         f'min={valid.min():.1f}  max={valid.max():.1f}  '
         f'mean={valid.mean():.1f} m  ({100*valid.size/arr.size:.1f}% coverage)')

print('=' * 60)
print('CELL 2e — Merge LiDAR Tiles')
print('=' * 60)
print(f'LIDAR_DIR: {LIDAR_DIR}')

_merge_tiles('*_DTM_1m.tif',    EA_DTM_TIFF, 'DTM')
_merge_tiles('*_FZ_DSM_1m.tif', EA_DSM_TIFF, 'DSM')

print('Done. Proceed to CELL 2d to compute nDSM.')


## CELL 2d-vom — EA LiDAR Vegetation Object Model (VOM) Auto-Download
Downloads the EA Vegetation Object Model via WCS. VOM pixels = canopy height above ground (vegetation only — buildings excluded).
Used in CELL 2h for auto-derived `VEG_HEIGHT_CAP_M` and in CELL 4 `_veg_height()` for per-polygon canopy heights.
Skip if `vom.tif` already exists.

In [ ]:
# ============================================================
# CELL 2d-vom — EA LiDAR VOM Auto-Download
# ============================================================
# Downloads EA Vegetation Object Model (VOM) via WCS.
# VOM pixels = height of top of canopy above ground for
# vegetation objects > 2.5 m (buildings excluded by design).
# Saves to LIDAR_DIR/vom.tif. Skip if already exists.
# ============================================================
import requests, os, re as _re_vom, math as _math_vom
import numpy as np
from pyproj import Transformer as _TrVOM

VOM_TIFF = os.path.join(LIDAR_DIR, 'vom.tif')
_VOM_WCS  = 'https://environment.data.gov.uk/spatialdata/vegetation-object-model/wcs'
_VOM_TILE_KM = 2.0

if os.path.exists(VOM_TIFF):
    print(f'VOM already downloaded: {VOM_TIFF}')
elif globals().get('FLAT_TERRAIN', False):
    print('FLAT_TERRAIN=True — skipping VOM download.')
    VOM_TIFF = None
else:
    # Auto-discover coverage ID from GetCapabilities
    _vom_cid = None
    try:
        _gc = requests.get(_VOM_WCS,
                           params={'SERVICE':'WCS','VERSION':'2.0.1','REQUEST':'GetCapabilities'},
                           timeout=30)
        _all_ids = _re_vom.findall(r'<(?:wcs:)?Identifier>(.*?)</(?:wcs:)?Identifier>', _gc.text)
        _vids = [x for x in _all_ids if any(k in x.lower() for k in ('vegetation','vom','ecae'))]
        if _vids:
            _vom_cid = _vids[0]
            print(f'VOM coverage ID: {_vom_cid}')
    except Exception as _ex_cap:
        print(f'GetCapabilities: {_ex_cap}')

    if _vom_cid is None:
        _vom_cid = 'ecae3bef-1e1d-4051-887b-9dc613c928ec__Vegetation_Object_Model_Elevation_2022'
        print(f'Using fallback coverage ID: {_vom_cid}')

    # BNG bbox
    _tr_vom = _TrVOM.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
    _e0v, _n0v = _tr_vom.transform(SCENE_WEST,  SCENE_SOUTH)
    _e1v, _n1v = _tr_vom.transform(SCENE_EAST,  SCENE_NORTH)
    _e0v -= 200; _n0v -= 200; _e1v += 200; _n1v += 200

    _tm_v   = _VOM_TILE_KM * 1000
    _ncol_v = max(1, _math_vom.ceil((_e1v - _e0v) / _tm_v))
    _nrow_v = max(1, _math_vom.ceil((_n1v - _n0v) / _tm_v))
    print(f'BNG bbox E[{_e0v:.0f},{_e1v:.0f}] N[{_n0v:.0f},{_n1v:.0f}]')
    print(f'Tiling: {_ncol_v}x{_nrow_v} = {_ncol_v*_nrow_v} tiles of {_VOM_TILE_KM} km')

    _tdir_v = os.path.join(LIDAR_DIR, '_vom_tiles')
    os.makedirs(_tdir_v, exist_ok=True)
    _tpaths_v, _ok_v, _fail_v = [], 0, 0

    for _rv in range(_nrow_v):
        for _cv in range(_ncol_v):
            _te0 = _e0v + _cv * _tm_v
            _te1 = min(_e0v + (_cv+1) * _tm_v, _e1v)
            _tn0 = _n0v + _rv * _tm_v
            _tn1 = min(_n0v + (_rv+1) * _tm_v, _n1v)
            _tp  = os.path.join(_tdir_v, f'vom_{_rv}_{_cv}.tif')
            if os.path.exists(_tp):
                _tpaths_v.append(_tp); continue
            _url_v = (
                f'{_VOM_WCS}?SERVICE=WCS&VERSION=2.0.1&REQUEST=GetCoverage'
                f'&COVERAGEID={_vom_cid}'
                f'&SUBSET=E,http://www.opengis.net/def/crs/EPSG/0/27700({_te0:.0f},{_te1:.0f})'
                f'&SUBSET=N,http://www.opengis.net/def/crs/EPSG/0/27700({_tn0:.0f},{_tn1:.0f})'
                f'&FORMAT=image/tiff'
            )
            try:
                _r_v = requests.get(_url_v, timeout=60)
                ct = _r_v.headers.get('Content-Type', '')
                if _r_v.status_code == 200 and 'image' in ct:
                    with open(_tp, 'wb') as _fv: _fv.write(_r_v.content)
                    _tpaths_v.append(_tp); _ok_v += 1
                    print(f'  [{_rv},{_cv}] OK  ({len(_r_v.content)//1024} KB)')
                else:
                    print(f'  [{_rv},{_cv}] HTTP {_r_v.status_code}  {ct[:40]}')
                    _fail_v += 1
            except Exception as _ev:
                print(f'  [{_rv},{_cv}] Error: {_ev}')
                _fail_v += 1

    if _tpaths_v:
        import rasterio as _rio_vom
        from rasterio.merge import merge as _merge_vom
        _srcs_v = [_rio_vom.open(p) for p in _tpaths_v]
        _mosaic_v, _tfm_v = _merge_vom(_srcs_v)
        _prof_v = _srcs_v[0].profile.copy()
        _prof_v.update({'height': _mosaic_v.shape[1], 'width': _mosaic_v.shape[2],
                        'transform': _tfm_v, 'compress': 'lzw', 'tiled': True, 'nodata': 0.0})
        for _sv in _srcs_v: _sv.close()
        _mosaic_v = _mosaic_v.astype(np.float32)
        _nd_v = float(_prof_v.get('nodata') or -9999)
        _mosaic_v[~np.isfinite(_mosaic_v) | (_mosaic_v == _nd_v) | (_mosaic_v < 0)] = 0.0
        with _rio_vom.open(VOM_TIFF, 'w', **_prof_v) as _dst_v:
            _dst_v.write(_mosaic_v)
        _vx = _mosaic_v[_mosaic_v > 2.5]
        print(f'\nVOM saved: {VOM_TIFF}')
        print(f'  Tiles: {len(_tpaths_v)} ok / {_fail_v} failed')
        if _vx.size > 0:
            print(f'  Vegetation coverage: {100*_vx.size/_mosaic_v.size:.1f}%  '
                  f'height {_vx.min():.1f}-{_vx.max():.1f} m  p50={np.percentile(_vx,50):.1f} m')
        import shutil; shutil.rmtree(_tdir_v, ignore_errors=True)
    else:
        print('No VOM tiles downloaded. VEG heights will fall back to nDSM.')
        VOM_TIFF = None


## CELL 2d-bha — OS Building Height Attribute + Microsoft Fallback
Auto-downloads building heights per footprint. Primary: OS MasterMap BHA via WFS (requires `OS_API_KEY`). Fallback: Microsoft Global Building Footprints with heights (free). Saves to `bha_heights.gpkg`.

In [ ]:
# ============================================================
# CELL 2d-bha — BUILDING HEIGHT ATTRIBUTE AUTO-DOWNLOAD
# ============================================================
# Primary  : OS MasterMap BHA via WFS API (OS_API_KEY required)
#            Returns RelativeHeightMaximum per building footprint
# Fallback : Microsoft Global Building Footprints with heights
#            Free, no auth, 1.5M height estimates for UK (2024)
# Output   : bha_heights.gpkg — spatial index of building heights
#            Used in CELL 4 _roof_height() priority 1
# Skip     : if bha_heights.gpkg already exists
# ============================================================
import os, json, math, requests
import numpy as np
from pyproj import Transformer as _TrBHA

BHA_GPKG = os.path.join(LIDAR_DIR, 'bha_heights.gpkg')
_OS_BHA_WFS = 'https://api.os.uk/features/v1/wfs'
_MS_BF_BASE = 'https://minedbuildings.blob.core.windows.net/global-buildings'

_os_key = globals().get('OS_API_KEY', '').strip()

if os.path.exists(BHA_GPKG):
    print(f'BHA already downloaded: {BHA_GPKG}')
else:
    import geopandas as _gpd_bha
    from shapely.geometry import box as _box_bha, shape as _shape_bha
    _bha_gdf = None

    # -- PRIMARY: OS MasterMap BHA via WFS
    if _os_key:
        print('Querying OS MasterMap Building Height Attribute via WFS...')
        try:
            _tr_bha = _TrBHA.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
            _be0, _bn0 = _tr_bha.transform(SCENE_WEST, SCENE_SOUTH)
            _be1, _bn1 = _tr_bha.transform(SCENE_EAST, SCENE_NORTH)
            _page, _feats, _offset = 100, [], 0
            while True:
                _p = {
                    'service': 'WFS', 'version': '2.0.0', 'request': 'GetFeature',
                    'typeNames': 'Topography_TopographicArea',
                    'outputFormat': 'GEOJSON',
                    'srsName': 'urn:ogc:def:crs:EPSG::27700',
                    'count': str(_page), 'startIndex': str(_offset),
                    'bbox': f'{_be0:.0f},{_bn0:.0f},{_be1:.0f},{_bn1:.0f},urn:ogc:def:crs:EPSG::27700',
                    'key': _os_key
                }
                _r = requests.get(_OS_BHA_WFS, params=_p, timeout=60)
                _js = _r.json()
                _batch = _js.get('features', [])
                _feats.extend(_batch)
                print(f'  fetched {len(_feats)} features...')
                if len(_batch) < _page:
                    break
                _offset += _page
            _rows = []
            for _f in _feats:
                _prop = _f.get('properties', {})
                _rh = _prop.get('RelativeHeightMaximum') or _prop.get('relativeheightmaximum')
                if _rh and float(_rh) > 0:
                    _rows.append({'geometry': _shape_bha(_f['geometry']), 'height': float(_rh)})
            if _rows:
                _bha_gdf = _gpd_bha.GeoDataFrame(_rows, crs='EPSG:27700').to_crs('EPSG:4326')
                print(f'OS BHA: {len(_bha_gdf)} buildings with heights')
        except Exception as _e_os:
            print(f'OS BHA failed: {_e_os}')

    # -- FALLBACK: Microsoft Global Building Footprints
    if _bha_gdf is None or len(_bha_gdf) == 0:
        print('Using Microsoft Global Building Footprints (free fallback)...')
        try:
            import mercantile as _mct
            _tiles = list(_mct.tiles(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH, zooms=9))
            _qkeys = list({_mct.quadkey(t) for t in _tiles})
            _rows_ms = []
            for _qk in _qkeys[:4]:
                _url_ms = f'{_MS_BF_BASE}/dataset-links.csv'
                _r_idx = requests.get(_url_ms, timeout=30)
                _lines = _r_idx.text.strip().split('\n')
                _link = next((l.split(',')[1] for l in _lines[1:] if 'United Kingdom' in l), None)
                if _link:
                    import pandas as _pd_ms
                    _df_ms = _pd_ms.read_csv(_link.strip(), compression='gzip')
                    _bbox_ms = (_df_ms['longitude'] >= SCENE_WEST) & (_df_ms['longitude'] <= SCENE_EAST) & \
                               (_df_ms['latitude'] >= SCENE_SOUTH) & (_df_ms['latitude'] <= SCENE_NORTH)
                    _sub = _df_ms[_bbox_ms]
                    for _, _row_ms in _sub.iterrows():
                        if 'height' in _row_ms and not np.isnan(_row_ms['height']) and _row_ms['height'] > 0:
                            _geom_ms = _shape_bha(json.loads(_row_ms['geometry']))
                            _rows_ms.append({'geometry': _geom_ms, 'height': float(_row_ms['height'])})
                break
            if _rows_ms:
                _bha_gdf = _gpd_bha.GeoDataFrame(_rows_ms, crs='EPSG:4326')
                print(f'Microsoft BF: {len(_bha_gdf)} buildings with heights')
            else:
                print('Microsoft BF: no heights found for this area')
        except ImportError:
            print('mercantile not installed -- pip install mercantile')
            print('BHA unavailable -- building heights will use nDSM footprint method')
        except Exception as _e_ms:
            print(f'Microsoft BF failed: {_e_ms}')
            print('Building heights will use nDSM footprint method')

    if _bha_gdf is not None and len(_bha_gdf) > 0:
        _bha_gdf.to_file(BHA_GPKG, driver='GPKG', layer='bha')
        print(f'BHA saved: {BHA_GPKG}  ({len(_bha_gdf)} records)')
        _hvals = _bha_gdf['height'].values
        print(f'  Height range: {_hvals.min():.1f} - {_hvals.max():.1f} m  median={np.median(_hvals):.1f} m')
    else:
        BHA_GPKG = None
        print('No BHA data -- building heights will use nDSM footprint percentile')

## CELL 2d-flood — EA Flood Defence Structures Auto-Download
Auto-downloads EA flood embankments, flood walls and levees via ArcGIS REST API. Free, no auth needed. Used in CELL 4 to extrude physical flood defence structures as blocking geometry.

In [ ]:
# ============================================================
# CELL 2d-flood — EA FLOOD DEFENCE STRUCTURES AUTO-DOWNLOAD
# ============================================================
# Downloads Environment Agency flood defence structures
# (embankments, flood walls, levees, tidal barriers) via
# ArcGIS REST API. Free, no authentication required.
# Output: flood_defences.geojson
# Used in CELL 4 to extrude flood embankments as PLY geometry.
# ============================================================
import os, json, requests
from pyproj import Transformer as _TrFld

FLOOD_DEF_JSON = os.path.join(LIDAR_DIR, 'flood_defences.geojson')

_FLOOD_URLS = [
    'https://environment.data.gov.uk/arcgis/rest/services/EA/FloodDefenceStructures/FeatureServer/0/query',
    'https://environment.data.gov.uk/arcgis/rest/services/EA/FloodAlertAreas/FeatureServer/0/query',
]

if os.path.exists(FLOOD_DEF_JSON):
    print(f'Flood defences already downloaded: {FLOOD_DEF_JSON}')
else:
    _tr_fld = _TrFld.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
    _fe0, _fn0 = _tr_fld.transform(SCENE_WEST, SCENE_SOUTH)
    _fe1, _fn1 = _tr_fld.transform(SCENE_EAST, SCENE_NORTH)
    _bbox_fld = f'{_fe0:.0f},{_fn0:.0f},{_fe1:.0f},{_fn1:.0f}'

    _flood_feats = []
    for _furl in _FLOOD_URLS:
        try:
            _params_fld = {
                'geometry': _bbox_fld,
                'geometryType': 'esriGeometryEnvelope',
                'inSR': '27700',
                'outSR': '4326',
                'spatialRel': 'esriSpatialRelIntersects',
                'outFields': '*',
                'f': 'geojson',
                'returnGeometry': 'true',
                'resultRecordCount': '5000'
            }
            _r_fld = requests.get(_furl, params=_params_fld, timeout=30)
            _js_fld = _r_fld.json()
            _batch_fld = _js_fld.get('features', [])
            if _batch_fld:
                _flood_feats = _batch_fld
                print(f'Flood defence: {len(_flood_feats)} features from {_furl.split("/")[-3]}')
                break
        except Exception as _ef:
            print(f'  {_furl.split("/")[-3]}: {_ef}')
            continue

    if _flood_feats:
        _geojson_fld = {'type': 'FeatureCollection', 'features': _flood_feats}
        with open(FLOOD_DEF_JSON, 'w') as _ff:
            json.dump(_geojson_fld, _ff)
        print(f'Saved: {FLOOD_DEF_JSON}  ({len(_flood_feats)} flood defence features)')
    else:
        print('No flood defence data found -- check EA ArcGIS availability')
        print('Flood embankments will use OSM data only')
        FLOOD_DEF_JSON = None

## CELL 2d-nfi — National Forest Inventory Auto-Download
Auto-downloads Forestry Commission NFI woodland polygons for the scene bbox. Free, no registration, no API key. Provides field-surveyed canopy heights and woodland type (broadleaved/conifer/mixed). Used in CELL 4 `_veg_height()` as priority source for woodland >0.5ha. Saves to `nfi_woodland.gpkg`.

In [ ]:
# ============================================================
# CELL 2d-nfi — NATIONAL FOREST INVENTORY AUTO-DOWNLOAD
# ============================================================
# Downloads Forestry Commission NFI woodland polygons for
# the scene bbox. Free, no registration, no API key needed.
# Source: data-forestry.opendata.arcgis.com (ArcGIS REST)
# Fields: IFT_IOA (woodland type), PRISPECIES (main species)
# Output: nfi_woodland.gpkg
# Used in CELL 4 _veg_height() priority 1 for woodland >0.5ha
# Skip: if nfi_woodland.gpkg already exists
# ============================================================
import os, json, requests
from pyproj import Transformer as _TrNFI

NFI_GPKG = os.path.join(LIDAR_DIR, 'nfi_woodland.gpkg')

# ArcGIS REST — NFI England 2023, no auth required
_NFI_URLS = [
    'https://services2.arcgis.com/mHXjdsZyvWJyFog1/arcgis/rest/services/NFI_England_2023/FeatureServer/0/query',
    'https://opendata.arcgis.com/datasets/0682c7cb180e4abe9dee7e4d5cc35784_0/FeatureServer/0/query',
]

if os.path.exists(NFI_GPKG):
    print(f'NFI already downloaded: {NFI_GPKG}')
else:
    # Convert scene bbox to EPSG:27700 for query
    _tr_nfi = _TrNFI.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
    _ne0, _nn0 = _tr_nfi.transform(SCENE_WEST, SCENE_SOUTH)
    _ne1, _nn1 = _tr_nfi.transform(SCENE_EAST, SCENE_NORTH)
    _bbox_nfi = f'{_ne0:.0f},{_nn0:.0f},{_ne1:.0f},{_nn1:.0f}'

    _nfi_feats = []
    _active_url = None

    # Find working endpoint
    for _nurl in _NFI_URLS:
        try:
            _tr = requests.get(_nurl, params={
                'where': '1=1', 'resultRecordCount': 1,
                'f': 'json', 'outFields': 'IFT_IOA'
            }, timeout=15)
            if _tr.status_code == 200 and 'features' in _tr.json():
                _active_url = _nurl
                print(f'NFI endpoint: {_nurl.split("/arcgis")[0]}')
                break
        except Exception:
            continue

    if _active_url is None:
        # Try ArcGIS Hub direct download
        print('ArcGIS REST not accessible -- trying Hub API...')
        try:
            _hub_url = 'https://data-forestry.opendata.arcgis.com/api/download/v1/items/0682c7cb180e4abe9dee7e4d5cc35784/geojson'
            _params_hub = {
                'where': '1=1',
                'geometry': f'{SCENE_WEST},{SCENE_SOUTH},{SCENE_EAST},{SCENE_NORTH}',
                'geometryType': 'esriGeometryEnvelope',
                'inSR': '4326',
                'spatialRel': 'esriSpatialRelIntersects',
                'f': 'geojson'
            }
            _r_hub = requests.get(_hub_url, params=_params_hub, timeout=60)
            if _r_hub.status_code == 200:
                _nfi_feats = _r_hub.json().get('features', [])
                print(f'Hub API: {len(_nfi_feats)} features')
        except Exception as _eh:
            print(f'Hub API failed: {_eh}')

    if _active_url:
        _offset = 0
        while True:
            _params_nfi = {
                'geometry': _bbox_nfi,
                'geometryType': 'esriGeometryEnvelope',
                'inSR': '27700',
                'outSR': '4326',
                'spatialRel': 'esriSpatialRelIntersects',
                'outFields': 'IFT_IOA,PRISPECIES,QUALITY,Shape__Area',
                'where': '1=1',
                'resultOffset': str(_offset),
                'resultRecordCount': '1000',
                'f': 'geojson',
                'returnGeometry': 'true'
            }
            try:
                _r_nfi = requests.get(_active_url, params=_params_nfi, timeout=60)
                _js_nfi = _r_nfi.json()
                _batch = _js_nfi.get('features', [])
                _nfi_feats.extend(_batch)
                print(f'  fetched {len(_nfi_feats)} woodland polygons...')
                if len(_batch) < 1000:
                    break
                _offset += 1000
            except Exception as _en:
                print(f'NFI query failed: {_en}')
                break

    if _nfi_feats:
        import geopandas as _gpd_nfi
        from shapely.geometry import shape as _shape_nfi
        _rows_nfi = []
        for _f in _nfi_feats:
            try:
                _prop = _f.get('properties', {})
                _geom = _shape_nfi(_f['geometry'])
                _area = _geom.area * (111320 ** 2)  # rough m2 from degrees
                _ift  = str(_prop.get('IFT_IOA', '') or '').lower()
                _spp  = str(_prop.get('PRISPECIES', '') or '').lower()
                # Derive canopy height from woodland type (NFI field-surveyed averages)
                if 'conifer' in _ift or 'conif' in _spp:
                    _cht = 20.0   # mature UK conifer (Sitka spruce avg)
                elif 'broadlea' in _ift or 'oak' in _spp or 'ash' in _spp:
                    _cht = 18.0   # mature UK broadleaf
                elif 'mixed' in _ift:
                    _cht = 17.0
                elif 'young' in _ift or 'restock' in _ift:
                    _cht = 5.0
                elif 'shrub' in _ift or 'scrub' in _ift:
                    _cht = 4.0
                else:
                    _cht = 15.0   # default woodland
                _rows_nfi.append({
                    'geometry': _geom,
                    'ift_ioa': _ift,
                    'species': _spp,
                    'canopy_h': _cht,
                    'area_m2': _area
                })
            except Exception:
                pass
        if _rows_nfi:
            _nfi_gdf = _gpd_nfi.GeoDataFrame(_rows_nfi, crs='EPSG:4326')
            _nfi_gdf.to_file(NFI_GPKG, driver='GPKG', layer='nfi')
            print(f'NFI saved: {NFI_GPKG}  ({len(_nfi_gdf)} woodland polygons)')
            _types = _nfi_gdf['ift_ioa'].value_counts()
            for _t, _n in _types.items():
                print(f'  {_t}: {_n} polygons')
        else:
            print('No NFI polygons found in scene bbox')
            NFI_GPKG = None
    else:
        print('No NFI data retrieved -- vegetation heights from nDSM only')
        NFI_GPKG = None


## CELL 2d-epc — EPC Wall/Roof/Glazing Material Download
Auto-downloads Energy Performance Certificate construction data per building. Provides real wall material (brick/concrete/curtain-wall/cavity), glazing type and roof description. Used in CELL 4 to replace age heuristic with measured construction data. Requires free EPC account (`EPC_EMAIL` + `EPC_API_KEY` in CELL 0).

In [ ]:
# ============================================================
# CELL 2d-epc — EPC BUILDING MATERIAL DATA AUTO-DOWNLOAD
# ============================================================
# Downloads MHCLG Energy Performance Certificate data for
# buildings in the scene. Provides real construction materials:
#   WALLS_DESCRIPTION  -> brick / concrete / curtain wall / cavity
#   GLAZING_TYPE       -> single / double / triple glazed
#   ROOF_DESCRIPTION   -> flat / pitched / thatched
#   CONSTRUCTION_AGE_BAND -> more precise than OSM start_date
# Output: epc_materials.gpkg
# Requires: EPC_EMAIL + EPC_API_KEY set in CELL 0
#   Register free at: get-energy-performance-data.communities.gov.uk
# Skip: if epc_materials.gpkg already exists
# ============================================================
import os, json, math, time, base64, requests
import numpy as np
from pyproj import Transformer as _TrEPC

EPC_GPKG = os.path.join(LIDAR_DIR, 'epc_materials.gpkg')

_epc_email   = globals().get('EPC_EMAIL',   '').strip()
_epc_api_key = globals().get('EPC_API_KEY', '').strip()

if os.path.exists(EPC_GPKG):
    print(f'EPC data already downloaded: {EPC_GPKG}')
elif not _epc_email or not _epc_api_key:
    print('EPC_EMAIL or EPC_API_KEY not set in CELL 0 -- skipping EPC download')
    print('Set both to enable real construction material data')
    EPC_GPKG = None
else:
    import geopandas as _gpd_epc
    from shapely.geometry import Point as _PtEPC

    # Basic auth header
    _creds = base64.b64encode(f'{_epc_email}:{_epc_api_key}'.encode()).decode()
    _hdrs  = {'Authorization': f'Basic {_creds}', 'Accept': 'application/json'}

    # Try new endpoint first, fall back to old
    _EPC_ENDPOINTS = [
        'https://get-energy-performance-data.communities.gov.uk/api/v1/domestic/search',
        'https://epc.opendatacommunities.org/api/v1/domestic/search',
    ]

    # Derive postcode sectors covering scene bbox
    # Use postcodes.io bulk lookup to find all postcodes in bbox
    print('Fetching postcodes in scene bbox...')
    _tr_epc = _TrEPC.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
    _bbox_str = f'{SCENE_WEST},{SCENE_SOUTH},{SCENE_EAST},{SCENE_NORTH}'

    # postcodes.io bbox lookup (free, no auth)
    _pc_url = f'https://api.postcodes.io/postcodes?lon={( SCENE_WEST+SCENE_EAST)/2}&lat={(SCENE_SOUTH+SCENE_NORTH)/2}&radius=8000&limit=100&widesearch=true'
    _pc_r = requests.get(_pc_url, timeout=30)
    _pc_sectors = set()
    if _pc_r.status_code == 200:
        _pc_data = _pc_r.json().get('result') or []
        for _pc in _pc_data:
            if _pc and _pc.get('postcode'):
                # Get district (e.g. "SW1A 1AA" -> "SW1A")
                _pcs = _pc['postcode'].split(' ')[0]
                _pc_sectors.add(_pcs)
    if not _pc_sectors:
        # Hardcoded fallback for London scene bbox
        _pc_sectors = {'N1','N4','N5','N6','N7','N8','N10','N15','N16',
                       'NW1','NW3','NW5','NW6','NW8','NW10',
                       'W1','W2','W9','W10','W11','W12',
                       'WC1','WC2','EC1','EC2','EC3','EC4',
                       'SW1','SW3','SW5','SW6','SW7','SW8','SW9','SW10',
                       'SE1','SE5','SE11','SE15','SE17'}
    print(f'  Querying {len(_pc_sectors)} postcode districts: {sorted(_pc_sectors)[:10]}...')

    # EPC fields we need
    _FIELDS = ['uprn','address','postcode','property-type',
               'walls-description','glazing-type','roof-description',
               'construction-age-band','local-authority']

    _epc_rows = []
    _active_endpoint = None

    for _ep in _EPC_ENDPOINTS:
        try:
            _tr = requests.get(_ep, headers=_hdrs,
                               params={'size': 1}, timeout=15)
            if _tr.status_code in (200, 400):
                _active_endpoint = _ep
                print(f'  Using endpoint: {_ep}')
                break
        except Exception:
            continue

    if _active_endpoint is None:
        print('ERROR: Could not connect to any EPC endpoint')
        print('Check EPC_EMAIL and EPC_API_KEY are correct')
        EPC_GPKG = None
    else:
        for _pcd in sorted(_pc_sectors):
            _after = None
            _sector_rows = 0
            while True:
                _params = {'postcode': _pcd, 'size': 5000}
                if _after:
                    _params['search-after'] = _after
                try:
                    _r = requests.get(_active_endpoint, headers=_hdrs,
                                      params=_params, timeout=60)
                    if _r.status_code == 401:
                        print('ERROR: Authentication failed -- check EPC_EMAIL and EPC_API_KEY')
                        _active_endpoint = None
                        break
                    if _r.status_code != 200:
                        break
                    _js = _r.json()
                    _rows = _js.get('rows', [])
                    for _rec in _rows:
                        _walls = str(_rec.get('walls-description', '') or '').lower()
                        _glaz  = str(_rec.get('glazing-type', '') or '').lower()
                        _roof  = str(_rec.get('roof-description', '') or '').lower()
                        _age   = str(_rec.get('construction-age-band', '') or '').lower()
                        _addr  = str(_rec.get('address', '') or '')
                        _pc    = str(_rec.get('postcode', '') or '')
                        _uprn  = str(_rec.get('uprn', '') or '')
                        _lat   = _rec.get('latitude')
                        _lon   = _rec.get('longitude')
                        if _lat and _lon:
                            try:
                                _lat, _lon = float(_lat), float(_lon)
                                if (SCENE_SOUTH <= _lat <= SCENE_NORTH and
                                    SCENE_WEST  <= _lon <= SCENE_EAST):
                                    _epc_rows.append({
                                        'geometry': _PtEPC(_lon, _lat),
                                        'uprn': _uprn, 'address': _addr,
                                        'postcode': _pc,
                                        'walls': _walls, 'glazing': _glaz,
                                        'roof': _roof, 'age_band': _age
                                    })
                                    _sector_rows += 1
                            except Exception:
                                pass
                    _after = _r.headers.get('X-Next-Search-After')
                    if not _rows or not _after:
                        break
                    time.sleep(0.1)
                except Exception as _ee:
                    print(f'  {_pcd}: {_ee}')
                    break
            if _active_endpoint is None:
                break
            if _sector_rows > 0:
                print(f'  {_pcd}: {_sector_rows} records')

        if _epc_rows:
            _epc_gdf = _gpd_epc.GeoDataFrame(_epc_rows, crs='EPSG:4326')
            _epc_gdf.to_file(EPC_GPKG, driver='GPKG', layer='epc')
            print(f'\nEPC saved: {EPC_GPKG}  ({len(_epc_gdf):,} records)')
            # Summary
            _wall_vals = [r['walls'] for r in _epc_rows if r['walls']]
            _brick  = sum(1 for w in _wall_vals if 'brick' in w)
            _conc   = sum(1 for w in _wall_vals if 'concrete' in w or 'cavity' in w)
            _curt   = sum(1 for w in _wall_vals if 'curtain' in w or 'glass' in w)
            print(f'  Walls: brick={_brick:,}  concrete/cavity={_conc:,}  curtain/glass={_curt:,}')
        else:
            print('No EPC records found in scene bbox')
            EPC_GPKG = None


## CELL 2d-greenspace — OS/OSM Greenspace Auto-Download

Downloads greenspace polygons (parks, playing fields, allotments, cemeteries, golf courses, meadows, heath) for the scene bbox via Overpass API. Free, no API key, England-wide. Saves to `greenspace.gpkg`. Used by CELL 4 to supplement OSM vegetation with richer boundary coverage.

In [ ]:
# ============================================================
# CELL 2d-greenspace — GREENSPACE AUTO-DOWNLOAD (Overpass)
# ============================================================
import os, requests as _rqs
import geopandas as gpd
from shapely.geometry import Polygon as _GSPoly

_gs_out = os.path.join(LIDAR_DIR, 'greenspace.gpkg')

# Overpass bbox: (south, west, north, east) WGS84
_ov_bbox = f"{SCENE_SOUTH},{SCENE_WEST},{SCENE_NORTH},{SCENE_EAST}"

_ov_q = f"""[out:json][timeout:90];
(
  way["leisure"~"^(park|garden|playing_fields|pitch|golf_course|nature_reserve)$"]({_ov_bbox});
  way["landuse"~"^(grass|allotments|cemetery|meadow|forest|recreation_ground|village_green|greenfield|orchard)$"]({_ov_bbox});
  way["natural"~"^(wood|scrub|heath|grassland)$"]({_ov_bbox});
  relation["leisure"~"^(park|garden|playing_fields|pitch|golf_course|nature_reserve)$"]({_ov_bbox});
  relation["landuse"~"^(grass|allotments|cemetery|meadow|forest|recreation_ground|village_green|greenfield|orchard)$"]({_ov_bbox});
  relation["natural"~"^(wood|scrub|heath|grassland)$"]({_ov_bbox});
);
out body;
>;
out skel qt;"""

print('Downloading greenspace polygons via Overpass ...')
try:
    _resp = _rqs.post(
        'https://overpass-api.de/api/interpreter',
        data={'data': _ov_q}, timeout=120
    )
    _resp.raise_for_status()
    _ov_data = _resp.json()

    # Node id -> (lon, lat) lookup
    _nodes = {e['id']: (e['lon'], e['lat'])
              for e in _ov_data['elements'] if e['type'] == 'node'}

    _features = []
    for _el in _ov_data['elements']:
        if _el['type'] not in ('way', 'relation'): continue
        _tags = _el.get('tags', {})
        _func = (_tags.get('leisure') or _tags.get('landuse') or
                 _tags.get('natural') or 'unknown')
        _name = _tags.get('name', '')

        if _el['type'] == 'way':
            _nids = _el.get('nodes', [])
            if len(_nids) < 4: continue
            _coords = [_nodes[n] for n in _nids if n in _nodes]
            if len(_coords) < 4: continue
            _geom = _GSPoly(_coords)
            if not _geom.is_valid or _geom.is_empty: continue
            _features.append({
                'geometry': _geom, 'gs_function': _func, 'name': _name,
                'leisure': _tags.get('leisure',''),
                'landuse': _tags.get('landuse',''),
                'natural': _tags.get('natural','')
            })

        elif _el['type'] == 'relation':
            for _m in _el.get('members', []):
                if _m['type'] != 'way' or _m['role'] != 'outer': continue
                _wn = next((e['nodes'] for e in _ov_data['elements']
                            if e['type'] == 'way' and e['id'] == _m['ref']), None)
                if not _wn: continue
                _coords = [_nodes[n] for n in _wn if n in _nodes]
                if len(_coords) < 4: continue
                _geom = _GSPoly(_coords)
                if not _geom.is_valid or _geom.is_empty: continue
                _features.append({
                    'geometry': _geom, 'gs_function': _func, 'name': _name,
                    'leisure': _tags.get('leisure',''),
                    'landuse': _tags.get('landuse',''),
                    'natural': _tags.get('natural','')
                })

    if _features:
        _gs_gdf = gpd.GeoDataFrame(_features, crs='EPSG:4326')
        _gs_gdf = _gs_gdf[_gs_gdf.geometry.notna()].drop_duplicates(subset='geometry')
        _gs_gdf.to_file(_gs_out, driver='GPKG', layer='greenspace')
        print(f'Greenspace saved: {len(_gs_gdf)} polygons -> {_gs_out}')
        GS_GPKG = _gs_out
    else:
        print('No greenspace polygons found in bbox.')
        GS_GPKG = None

except Exception as _e:
    print(f'Greenspace download failed: {_e}')
    GS_GPKG = None


## CELL 2d-conservation — Historic England Conservation Areas Auto-Download

Downloads conservation area polygons for the scene bbox via Historic England ArcGIS REST (primary) with OSM Overpass fallback. Buildings inside conservation areas are assigned itu_brick in CELL 4. Free, no API key, England-wide. Saves to `conservation_areas.gpkg`.

In [ ]:
# ============================================================
# CELL 2d-conservation — CONSERVATION AREAS AUTO-DOWNLOAD
# ============================================================
import os, requests as _rqs
import geopandas as gpd
from shapely.geometry import shape as _cshp
from pyproj import Transformer as _TrCA

_ca_out = os.path.join(LIDAR_DIR, 'conservation_areas.gpkg')
_to_wgs_ca = _TrCA.from_crs('EPSG:27700', 'EPSG:4326', always_xy=True)
_to_bng_ca = _TrCA.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)

# Scene bbox in BNG (EPSG:27700)
_bng_sw = _to_bng_ca.transform(SCENE_WEST, SCENE_SOUTH)
_bng_ne = _to_bng_ca.transform(SCENE_EAST, SCENE_NORTH)
_bbox_bng = f"{_bng_sw[0]},{_bng_sw[1]},{_bng_ne[0]},{_bng_ne[1]}"

print('Downloading conservation areas ...')
_ca_features = []

# Primary: Historic England ArcGIS REST
try:
    _he_url = 'https://services.historicengland.org.uk/arcgis/rest/services/Planning/Conservation_Areas/FeatureServer/0/query'
    _he_params = {
        'geometry': _bbox_bng,
        'geometryType': 'esriGeometryEnvelope',
        'inSR': '27700', 'spatialRel': 'esriSpatialRelIntersects',
        'outFields': 'NAME,DESIG_DATE',
        'returnGeometry': 'true', 'outSR': '4326',
        'resultRecordCount': '2000', 'f': 'json'
    }
    _r = _rqs.get(_he_url, params=_he_params, timeout=30)
    _r.raise_for_status()
    _jd = _r.json()
    for _feat in _jd.get('features', []):
        try:
            _geom = _cshp(_feat['geometry'])
            if not _geom.is_valid: _geom = _geom.buffer(0)
            if _geom.is_empty: continue
            _ca_features.append({'geometry': _geom,
                                  'name': _feat.get('attributes', {}).get('NAME', ''),
                                  'source': 'historic_england'})
        except Exception: pass
    print(f'  Historic England: {len(_ca_features)} conservation areas')
except Exception as _e_he:
    print(f'  Historic England failed: {_e_he}')

# Fallback: OSM Overpass
if not _ca_features:
    try:
        _ov_bbox = f"{SCENE_SOUTH},{SCENE_WEST},{SCENE_NORTH},{SCENE_EAST}"
        _ov_q = f"""[out:json][timeout:60];
(
  way[\"boundary\"=\"conservation_area\"]({_ov_bbox});
  relation[\"boundary\"=\"conservation_area\"]({_ov_bbox});
);
out body;>;out skel qt;"""
        _r2 = _rqs.post('https://overpass-api.de/api/interpreter', data={'data': _ov_q}, timeout=90)
        _r2.raise_for_status()
        _od = _r2.json()
        _onodes = {e['id']: (e['lon'], e['lat']) for e in _od['elements'] if e['type'] == 'node'}
        from shapely.geometry import Polygon as _CAPoly
        for _el in _od['elements']:
            if _el['type'] != 'way': continue
            _nids = _el.get('nodes', [])
            _coords = [_onodes[n] for n in _nids if n in _onodes]
            if len(_coords) < 4: continue
            _geom = _CAPoly(_coords)
            if not _geom.is_valid: _geom = _geom.buffer(0)
            if _geom.is_empty: continue
            _ca_features.append({'geometry': _geom,
                                  'name': _el.get('tags', {}).get('name', ''),
                                  'source': 'osm'})
        print(f'  OSM fallback: {len(_ca_features)} conservation areas')
    except Exception as _e_ov:
        print(f'  OSM fallback failed: {_e_ov}')

if _ca_features:
    _ca_gdf_dl = gpd.GeoDataFrame(_ca_features, crs='EPSG:4326')
    _ca_gdf_dl = _ca_gdf_dl[_ca_gdf_dl.geometry.notna()]
    _ca_gdf_dl.to_file(_ca_out, driver='GPKG', layer='conservation_areas')
    print(f'Conservation areas saved: {len(_ca_gdf_dl)} -> {_ca_out}')
    CA_GPKG = _ca_out
else:
    print('No conservation areas found.')
    CA_GPKG = None


## CELL 2d-ms-footprints — Microsoft Global Building Footprints Auto-Download

Downloads Microsoft ML-detected building footprints for the scene bbox via
quadkey-partitioned GeoJSON.gz tiles (zoom level 9, ~70x80 km per tile at UK latitude).
Fills gaps where OSM coverage is sparse.  Output saved to `ms_buildings.gpkg`.

In [ ]:
# ============================================================
# CELL 2d-ms-footprints — Microsoft Global Building Footprints
# ============================================================
import math, gzip, io, os, json as _json_ms
import requests
import geopandas as gpd
from shapely.geometry import box as sg_box, shape as sg_shape

_MS_GPKG_PATH = os.path.join(GPKG_DIR, 'ms_buildings.gpkg')
_MS_BASE_URL  = 'https://minedbuildings.blob.core.windows.net/global-buildings/'

def _ms_tile(lat, lng, z):
    n = 1 << z
    tx = int((lng + 180.0) / 360.0 * n)
    sin_lat = math.sin(math.radians(max(-85.05, min(85.05, lat))))
    ty = int((0.5 - math.log((1 + sin_lat) / max(1e-12, 1 - sin_lat)) / (4 * math.pi)) * n)
    return max(0, min(n-1, tx)), max(0, min(n-1, ty))

def _ms_quadkey(tx, ty, z):
    qk = []
    for i in range(z, 0, -1):
        d = 0; m = 1 << (i - 1)
        if tx & m: d += 1
        if ty & m: d += 2
        qk.append(str(d))
    return ''.join(qk)

def _scene_quadkeys(z=9):
    tx0, ty0 = _ms_tile(SCENE_NORTH, SCENE_WEST, z)
    tx1, ty1 = _ms_tile(SCENE_SOUTH, SCENE_EAST, z)
    return [_ms_quadkey(tx, ty, z) for tx in range(tx0, tx1+1) for ty in range(ty0, ty1+1)]

MS_GPKG = None
gdf_ms_raw = None

if os.path.exists(_MS_GPKG_PATH):
    print('MS footprints: loading from cache ...')
    try:
        gdf_ms_raw = gpd.read_file(_MS_GPKG_PATH)
        MS_GPKG = _MS_GPKG_PATH
        print(f'  {len(gdf_ms_raw)} MS buildings (cached)')
    except Exception as _e:
        print(f'  Cache load failed: {_e}')

if gdf_ms_raw is None:
    _qks = _scene_quadkeys(9)
    print(f'MS footprints: fetching {len(_qks)} tile(s) ...')
    _scene_box_wgs = sg_box(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH)
    _ms_rows = []
    for _qk in _qks:
        _url = f'{_MS_BASE_URL}{_qk}.geojson.gz'
        try:
            _resp = requests.get(_url, timeout=60,
                                 headers={'User-Agent': 'sionna-rt-scene-builder/1.0'})
            if _resp.status_code == 404:
                print(f'  {_qk}: no tile (404)')
                continue
            _resp.raise_for_status()
            with gzip.open(io.BytesIO(_resp.content)) as _gz:
                _fc = _json_ms.loads(_gz.read().decode('utf-8'))
            _feats = _fc.get('features', [])
            _n_in = 0
            for _feat in _feats:
                try:
                    _geom = sg_shape(_feat['geometry'])
                    if not _scene_box_wgs.intersects(_geom):
                        continue
                    _props = _feat.get('properties') or {}
                    _ms_rows.append({'geometry': _geom,
                                     'ms_height': _props.get('height', None),
                                     'ms_conf':   _props.get('confidence', None)})
                    _n_in += 1
                except Exception:
                    pass
            print(f'  {_qk}: {_n_in}/{len(_feats)} in scene bbox')
        except Exception as _e:
            print(f'  {_qk}: failed: {_e}')
    if _ms_rows:
        _gdf_ms = gpd.GeoDataFrame(_ms_rows, geometry='geometry', crs='EPSG:4326')
        _gdf_ms = _gdf_ms.to_crs('EPSG:27700')
        _gdf_ms.to_file(_MS_GPKG_PATH, driver='GPKG')
        gdf_ms_raw = _gdf_ms
        MS_GPKG = _MS_GPKG_PATH
        print(f'MS footprints: saved {len(gdf_ms_raw)} buildings')
    else:
        print('MS footprints: no buildings in this bbox')
        gdf_ms_raw = gpd.GeoDataFrame(
            columns=['geometry','ms_height','ms_conf'], geometry='geometry', crs='EPSG:27700')


## CELL 2f — EA LiDAR Point Cloud Auto-Download

Auto-downloads EA National LiDAR Programme 1m point cloud tiles (.laz) for the exact scene bbox. Uses a **single ArcGIS bbox query** to fetch all tile URLs at once — no per-tile requests, no timeouts. Shows estimated download size before starting. Tiles already on disk are skipped. Output: `LIDAR_DIR/pointcloud_tiles/`. Used by CELL 2g for roof shape extraction.

In [ ]:
# ============================================================
# CELL 2f — EA LIDAR POINT CLOUD AUTO-DOWNLOAD
# ============================================================
# Downloads EA National LiDAR Programme 1m point cloud tiles
# (.laz) for the exact scene bbox — fully auto, no predefined
# tile names or paths.
# Strategy: single ArcGIS bbox query returns ALL tile URLs at
# once — avoids per-tile timeout that plagued previous approach.
# Shows estimated download size before starting.
# Output: LIDAR_DIR/pointcloud_tiles/*.laz
# Used in CELL 2g (roof shape) and optional V5 features.
# Skip: if tiles already present for this bbox.
# ============================================================
import os, math, zipfile, time, glob as _glob_pc
import requests
from pyproj import Transformer as _TrPC

PC_DIR = os.path.join(LIDAR_DIR, 'pointcloud_tiles')
os.makedirs(PC_DIR, exist_ok=True)

# ── Convert scene bbox to EPSG:27700 ─────────────────────────
_tr_pc = _TrPC.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)
_pe0, _pn0 = _tr_pc.transform(SCENE_WEST,  SCENE_SOUTH)
_pe1, _pn1 = _tr_pc.transform(SCENE_EAST,  SCENE_NORTH)
_bbox_pc = f'{_pe0:.0f},{_pn0:.0f},{_pe1:.0f},{_pn1:.0f}'
print(f'Scene bbox BNG: E {_pe0:.0f}-{_pe1:.0f}  N {_pn0:.0f}-{_pn1:.0f}')

# ── Single ArcGIS bbox query — all tiles at once ──────────────
_ARCGIS_PC = ('https://environment.data.gov.uk/arcgis/rest/services/'
              'EA/LiDAR_Point_Cloud_NLP/MapServer/0/query')

_ARCGIS_FALLBACK = ('https://environment.data.gov.uk/arcgis/rest/services/'
                    'EA/LiDAR_Time_Stamped_Point_Cloud/MapServer/0/query')

def _fetch_tile_index(bbox_27700):
    """Single request returns all tile URLs for the bbox."""
    _params = {
        'geometry': bbox_27700,
        'geometryType': 'esriGeometryEnvelope',
        'inSR': '27700',
        'spatialRel': 'esriSpatialRelIntersects',
        'outFields': 'TILE_NAME,DOWNLOAD_URL,FILE_SIZE_MB,YEAR',
        'where': '1=1',
        'returnGeometry': 'false',
        'resultRecordCount': '5000',
        'f': 'json'
    }
    for _url in [_ARCGIS_PC, _ARCGIS_FALLBACK]:
        try:
            _r = requests.get(_url, params=_params, timeout=30)
            _r.raise_for_status()
            _js = _r.json()
            _feats = _js.get('features', [])
            if _feats:
                print(f'  Tile index from: {_url.split("/EA/")[1].split("/")[0]}')
                return _feats
        except Exception as _e:
            print(f'  {_url.split("/EA/")[1].split("/")[0]}: {_e}')
    return []

print('Querying EA tile index...')
_tile_feats = _fetch_tile_index(_bbox_pc)

if not _tile_feats:
    print('ERROR: Could not fetch tile index from EA ArcGIS')
    print('Check network connection or try later')
else:
    # ── Build tile download list ──────────────────────────────
    _tile_list = []
    for _tf in _tile_feats:
        _attr = _tf.get('attributes', {})
        _name = str(_attr.get('TILE_NAME', '') or '')
        _url  = str(_attr.get('DOWNLOAD_URL', '') or '')
        _size = float(_attr.get('FILE_SIZE_MB', 0) or 0)
        if not _url or not _name:
            # Derive direct URL from tile name
            _url = (f'https://environment.data.gov.uk/UserDownloads/'
                    f'interactive/NLPPC1m/{_name}_PointCloud_1m.zip')
        _tile_list.append((_name, _url, _size))

    # ── Show size estimate before downloading ─────────────────
    _total_mb = sum(s for _, _, s in _tile_list if s > 0)
    _known    = sum(1 for _, _, s in _tile_list if s > 0)
    if _known < len(_tile_list):
        # Estimate unknown sizes (~50MB per urban tile)
        _total_mb += (len(_tile_list) - _known) * 50
    print(f'\nTiles found: {len(_tile_list)}')
    print(f'Estimated download size: {_total_mb:.0f} MB  ({_total_mb/1024:.1f} GB)')
    print(f'Output: {PC_DIR}')
    print()

    # ── Check which tiles already exist ──────────────────────
    _existing = set()
    for _laz in (_glob_pc.glob(os.path.join(PC_DIR, '*.laz')) +
                 _glob_pc.glob(os.path.join(PC_DIR, '*.las'))):
        _existing.add(os.path.basename(_laz).split('_')[0].upper())

    _to_download = [t for t in _tile_list if t[0].upper() not in _existing]
    print(f'Already cached: {len(_existing)}  |  To download: {len(_to_download)}')

    if not _to_download:
        print('All tiles already downloaded.')
    else:
        # ── Download tiles ────────────────────────────────────
        _sess_pc = requests.Session()
        _sess_pc.headers['User-Agent'] = 'sionna-rt-scene-builder/1.0'
        _ok = _err = _miss = 0

        for _i, (_name, _url, _size) in enumerate(_to_download, 1):
            _laz_found = _glob_pc.glob(os.path.join(PC_DIR, f'{_name}*.laz'))
            if _laz_found:
                _ok += 1
                print(f'  [{_i:3d}/{len(_to_download)}] = {_name} (cached)')
                continue

            _zip = os.path.join(PC_DIR, f'{_name}.zip')
            _downloaded = False
            for _attempt in range(3):
                try:
                    _r = _sess_pc.get(_url, timeout=120, stream=True)
                    if _r.status_code == 404:
                        _miss += 1
                        print(f'  [{_i:3d}/{len(_to_download)}] - {_name} (not available)')
                        _downloaded = True
                        break
                    _r.raise_for_status()
                    with open(_zip, 'wb') as _fz:
                        for _chunk in _r.iter_content(65_536):
                            _fz.write(_chunk)
                    _downloaded = True
                    break
                except Exception as _de:
                    if _attempt < 2:
                        time.sleep(2 ** _attempt)
                    else:
                        print(f'  [{_i:3d}/{len(_to_download)}] x {_name}: {_de}')
                        _err += 1
                        _downloaded = True

            if os.path.exists(_zip):
                try:
                    with zipfile.ZipFile(_zip, 'r') as _zf:
                        for _m in _zf.namelist():
                            if _m.lower().endswith(('.laz', '.las')):
                                _zf.extract(_m, PC_DIR)
                    os.remove(_zip)
                    _ok += 1
                    print(f'  [{_i:3d}/{len(_to_download)}] + {_name}  ({_size:.0f} MB)')
                except Exception as _ze:
                    print(f'  [{_i:3d}/{len(_to_download)}] x {_name} unzip: {_ze}')
                    try: os.remove(_zip)
                    except: pass
                    _err += 1
            time.sleep(0.3)

        _sess_pc.close()

        _all_laz = (_glob_pc.glob(os.path.join(PC_DIR, '*.laz')) +
                    _glob_pc.glob(os.path.join(PC_DIR, '*.las')))
        _disk_mb = sum(os.path.getsize(f) for f in _all_laz) / 1e6
        print(f'\nDone: ok={_ok}  missing={_miss}  errors={_err}')
        print(f'Point cloud tiles on disk: {len(_all_laz)} files  ({_disk_mb:.0f} MB)')


## CELL 2g — Point Cloud Roof Shape Detector

Indexes downloaded `.laz` tiles and defines `_pc_roof_shape(geom)` used by CELL 4.

- **Flat**: roof point height range < 2 m
- **Pitched**: height variation > 2 m with sloped distribution
- **Stepped**: bimodal height histogram (podium + tower)

Requires `laspy` (`pip install laspy[lazrs]`). Falls back to OSM/nDSM if unavailable.

In [ ]:
# ============================================================
# CELL 2g — POINT CLOUD ROOF SHAPE DETECTOR SETUP
# ============================================================
import glob as _glob_pc2, re as _re_pc
import numpy as np
from pyproj import Transformer as _TrPC2

try:
    import laspy as _laspy
    _HAS_LASPY = True
    print(f'laspy {_laspy.__version__} available — roof shape detection enabled')
except ImportError:
    _HAS_LASPY = False
    print('[WARN] laspy not installed — roof shape detection disabled')
    print('       pip install laspy[lazrs]')

# Reverse BNG tile ref → (E, N) SW corner in metres
_BNG_500K_INV_PC = {'S':(0,0),'T':(1,0),'N':(0,1),'O':(1,1),'H':(0,2),'J':(1,2)}
_BNG_LTRS_PC2 = 'ABCDEFGHJKLMNOPQRSTUVWXYZ'

def _ostile_to_bng_pc(tile_ref):
    maj, minor = tile_ref[0], tile_ref[1]
    ew, nw = int(tile_ref[2:4]), int(tile_ref[4:6])
    e500k, n500k = _BNG_500K_INV_PC.get(maj, (0, 0))
    min_idx = _BNG_LTRS_PC2.index(minor)
    e100 = e500k * 5 + (min_idx % 5)
    n100 = n500k * 5 + (4 - min_idx // 5)
    return e100 * 100_000 + ew * 1_000, n100 * 100_000 + nw * 1_000

# Build tile index: (E_km, N_km) → file path
_PC_TILE_IDX = {}
_pc_dir_search = globals().get('PC_DIR', '')
if _HAS_LASPY and _pc_dir_search:
    _pc_files2 = (_glob_pc2.glob(os.path.join(_pc_dir_search, '**/*.laz'), recursive=True) +
                  _glob_pc2.glob(os.path.join(_pc_dir_search, '**/*.las'), recursive=True))
    for _pf in _pc_files2:
        _base = os.path.splitext(os.path.basename(_pf))[0].upper()
        _m2 = _re_pc.match(r'([A-Z]{2}\d{4})', _base)
        if _m2:
            try:
                _em2, _nm2 = _ostile_to_bng_pc(_m2.group(1))
                _PC_TILE_IDX[(_em2 // 1_000, _nm2 // 1_000)] = _pf
            except Exception:
                pass
    print(f'Point cloud tiles indexed: {len(_PC_TILE_IDX)}')
else:
    print('_PC_TILE_IDX empty — _pc_roof_shape() will return None')

_wgs84_to_bng_pc = Transformer.from_crs('EPSG:4326', 'EPSG:27700', always_xy=True)

def _pc_roof_shape(geom_wgs84, min_points=8):
    """
    Classify building roof from LiDAR point cloud.
    Returns dict {shape, pitch_deg, step_height} or None.
    shape: 'flat' | 'pitched' | 'stepped'
    """
    if not _HAS_LASPY or not _PC_TILE_IDX:
        return None
    try:
        from shapely.ops import transform as _shp_tr2
        geom_bng = _shp_tr2(_wgs84_to_bng_pc.transform, geom_wgs84)
        minx, miny, maxx, maxy = geom_bng.bounds
        if (maxx - minx) < 3 or (maxy - miny) < 3:
            return None

        e0k, e1k = int(minx // 1_000), int(maxx // 1_000)
        n0k, n1k = int(miny // 1_000), int(maxy // 1_000)

        Z_parts = []
        for _ek2 in range(e0k, e1k + 1):
            for _nk2 in range(n0k, n1k + 1):
                _tile2 = _PC_TILE_IDX.get((_ek2, _nk2))
                if _tile2 is None:
                    continue
                try:
                    with _laspy.open(_tile2) as _lf2:
                        for _chunk2 in _lf2.chunk_iterator(50_000):
                            _cx2 = np.array(_chunk2.x)
                            _cy2 = np.array(_chunk2.y)
                            _cz2 = np.array(_chunk2.z)
                            _bm2 = ((_cx2>=minx)&(_cx2<=maxx)&(_cy2>=miny)&(_cy2<=maxy))
                            if not np.any(_bm2):
                                continue
                            _cx2, _cy2, _cz2 = _cx2[_bm2], _cy2[_bm2], _cz2[_bm2]
                            try:
                                from shapely import contains_xy as _cxy2
                                _inp2 = _cxy2(geom_bng, _cx2, _cy2)
                            except Exception:
                                _inp2 = np.ones(len(_cx2), dtype=bool)
                            Z_parts.append(_cz2[_inp2])
                except Exception:
                    continue

        if not Z_parts:
            return None
        Z = np.concatenate(Z_parts)
        Z = Z[(Z > 1.5) & (Z < 150.0)]
        if len(Z) < min_points:
            return None

        z_p10 = float(np.percentile(Z, 10))
        z_p90 = float(np.percentile(Z, 90))
        z_range = z_p90 - z_p10

        # Stepped: bimodal Z distribution
        _nb = min(30, max(10, len(Z) // 5))
        _hist, _edges = np.histogram(Z, bins=_nb)
        _step_h = None
        for _bi in range(2, len(_hist) - 2):
            if (_hist[_bi] < 0.15 * np.max(_hist) and
                    np.max(_hist[:_bi]) > 0.2 * np.max(_hist) and
                    np.max(_hist[_bi+1:]) > 0.2 * np.max(_hist)):
                _step_h = float(_edges[_bi])
                break
        if _step_h is not None:
            return {'shape': 'stepped', 'pitch_deg': 0.0, 'step_height': _step_h}

        # Pitched: significant height range
        if z_range > 2.0:
            _half = max((maxx-minx)/2.0, (maxy-miny)/2.0, 1.0)
            _pdeg = float(np.clip(np.degrees(np.arctan(z_range/2.0/_half)), 8, 55))
            return {'shape': 'pitched', 'pitch_deg': _pdeg, 'step_height': None}

        return {'shape': 'flat', 'pitch_deg': 0.0, 'step_height': None}
    except Exception:
        return None

print('_pc_roof_shape() ready')


## CELL 2h — Auto-Derive Height Caps from nDSM
Derives `VEG_HEIGHT_CAP_M` and `CITY_MAX_HEIGHT_M` automatically from the nDSM raster.
Makes the scene builder portable across any city with no manual tuning.
Run after CELL 2d (ndsm.tif must exist). Falls back to config defaults if nDSM is unavailable.

In [ ]:
# ============================================================
# CELL 2h — AUTO-DERIVE HEIGHT CAPS FROM nDSM + VOM
# ============================================================
# Derives VEG_HEIGHT_CAP_M and CITY_MAX_HEIGHT_M automatically.
#
#   VEG_HEIGHT_CAP_M  : p95 of VOM pixels > 2.5 m (vegetation only)
#                       Falls back to p95 of nDSM [1.5-35 m] if VOM unavailable
#   CITY_MAX_HEIGHT_M : p99.9 of nDSM pixels > 3 m (all above-ground objects)
#
# Portable — works for any city, zero manual tuning.
# ============================================================
import numpy as np
import os

_ndsm_path_2h = os.path.join(LIDAR_DIR, 'ndsm.tif')
_vom_path_2h  = globals().get('VOM_TIFF', os.path.join(LIDAR_DIR, 'vom.tif'))

def _sample_raster_flat(path, downsample=4):
    """Read raster at 1/downsample resolution, return valid float32 values."""
    import rasterio
    with rasterio.open(path) as _s:
        _h = max(1, _s.height // downsample)
        _w = max(1, _s.width  // downsample)
        _d = _s.read(1, out_shape=(_h, _w)).astype(np.float32)
        _nd = _s.nodata
    _v = _d.ravel()
    if _nd is not None:
        _v = _v[_v != float(_nd)]
    return _v[np.isfinite(_v)]

# ── VEG_HEIGHT_CAP_M ──────────────────────────────────────────────────────
_veg_src = 'config default'
if _vom_path_2h and os.path.exists(_vom_path_2h):
    try:
        _vom_px = _sample_raster_flat(_vom_path_2h)
        _vom_veg = _vom_px[_vom_px > 2.5]
        if len(_vom_veg) >= 200:
            VEG_HEIGHT_CAP_M = float(np.clip(np.percentile(_vom_veg, 95), 8.0, 35.0))
            _veg_src = f'VOM p95 of {len(_vom_veg):,} vegetation pixels (buildings excluded)'
        else:
            _veg_src = f'VOM insufficient ({len(_vom_veg)} pixels) — trying nDSM'
    except Exception as _ev2h:
        _veg_src = f'VOM error ({_ev2h}) — trying nDSM'

if _veg_src.startswith('VOM') and 'insufficient' not in _veg_src and 'error' not in _veg_src:
    pass  # already set from VOM
elif os.path.exists(_ndsm_path_2h):
    try:
        _ndsm_px = _sample_raster_flat(_ndsm_path_2h)
        _veg_px  = _ndsm_px[(_ndsm_px >= 1.5) & (_ndsm_px <= 35.0)]
        if len(_veg_px) >= 200:
            VEG_HEIGHT_CAP_M = float(np.clip(np.percentile(_veg_px, 95), 8.0, 30.0))
            _veg_src = f'nDSM p95 of {len(_veg_px):,} pixels in [1.5-35 m]'
    except Exception as _en2h:
        _veg_src = f'nDSM error ({_en2h}) — using config default'

# ── CITY_MAX_HEIGHT_M ─────────────────────────────────────────────────────
_bld_src = 'config default'
if os.path.exists(_ndsm_path_2h):
    try:
        if '_ndsm_px' not in dir():
            _ndsm_px = _sample_raster_flat(_ndsm_path_2h)
        _bld_px = _ndsm_px[_ndsm_px >= 3.0]
        if len(_bld_px) >= 200:
            CITY_MAX_HEIGHT_M = float(np.clip(np.percentile(_bld_px, 99.9), 20.0, 250.0))
            _bld_src = f'nDSM p99.9 of {len(_bld_px):,} pixels > 3 m'
    except Exception as _eb2h:
        _bld_src = f'nDSM error ({_eb2h}) — using config default'

print('Auto-derived height caps:')
print(f'  VEG_HEIGHT_CAP_M  = {VEG_HEIGHT_CAP_M:.1f} m  ({_veg_src})')
print(f'  CITY_MAX_HEIGHT_M = {CITY_MAX_HEIGHT_M:.1f} m  ({_bld_src})')


## CELL 3 — Build Terrain PLY

Samples the DTM GeoTIFF onto a regular `TERRAIN_GRID_N × TERRAIN_GRID_N` grid
and writes `terrain.ply`. All building and vegetation base heights are referenced
to this terrain surface so TX/RX positions sit exactly on the loaded scene.

Skip when `FLAT_TERRAIN=True`.


In [ ]:
# ============================================================
# CELL 3 — BUILD TERRAIN PLY
# ============================================================
# FLAT_TERRAIN=True          : flat z=0 plane — no DEM needed
# TERRAIN_SOURCE='aws_dem'   : DEM from AWS tiles  (run CELL 2 first)
# TERRAIN_SOURCE='ea_lidar'  : DEM from EA LiDAR  (run CELL 2b–2e first)
# Skips automatically if terrain.ply already exists on disk.
# ============================================================
_terrain_ply = os.path.join(MESH_DIR, 'terrain.ply')
if os.path.exists(_terrain_ply):
    print(f"terrain.ply already exists — skipping rebuild.")
    print(f"  {_terrain_ply}")
    print("Delete the file and re-run this cell to force a rebuild.")
    # Still define local_z from EA DTM so CELL 4 base_z is correct
    _src = globals().get('TERRAIN_SOURCE', 'aws_dem')
    if not FLAT_TERRAIN and _src == 'ea_lidar' and os.path.exists(globals().get('EA_DTM_TIFF','')):
        import rasterio as _rio2
        from pyproj import Transformer as _Tr2
        _utm_to_bng2 = _Tr2.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)
        _ds2 = _rio2.open(EA_DTM_TIFF)
        _dem2 = _ds2.read(1).astype(__import__('numpy').float32)
        _tf2  = _ds2.transform
        _nd2  = _ds2.nodata
        _elev_path2 = os.path.join(SCENE_DIR, 'origin_elev2.json')
        if os.path.exists(_elev_path2):
            import json as _j2
            origin_elev_asl = float(_j2.load(open(_elev_path2))['origin_elev_asl_m'])
        else:
            _ox2, _oy2 = _utm_to_bng2.transform(center_utm[0], center_utm[1])
            _oc2, _or2 = ~_tf2 * (_ox2, _oy2)
            _or2, _oc2 = int(_or2), int(_oc2)
            origin_elev_asl = float(_dem2[_or2, _oc2]) if 0<=_or2<_dem2.shape[0] and 0<=_oc2<_dem2.shape[1] else 0.0
        def _ea_z2(ux, uy):
            bx, by = _utm_to_bng2.transform(ux, uy)
            cf, rf = ~_tf2 * (bx, by)
            r, c = int(rf), int(cf)
            H, W = _dem2.shape
            if 0<=r<H-1 and 0<=c<W-1:
                dr, dc = rf-r, cf-c
                z = ((1-dr)*(1-dc)*_dem2[r,c]+(1-dr)*dc*_dem2[r,c+1]+dr*(1-dc)*_dem2[r+1,c]+dr*dc*_dem2[r+1,c+1])
                if _nd2 is None or not __import__('numpy').isclose(float(z), _nd2): return float(z)
            return origin_elev_asl
        def local_z(lon, lat):
            ux, uy = to_utm.transform(lon, lat)
            return _ea_z2(ux, uy) - origin_elev_asl
        print(f'  local_z restored from EA DTM  (origin={origin_elev_asl:.1f} m ASL)')
    elif FLAT_TERRAIN:
        origin_elev_asl = 0.0
        def local_z(lon, lat): return 0.0
        print('  local_z = 0.0 (FLAT_TERRAIN)')
    else:
        print('[WARN] EA_DTM_TIFF not found — local_z will fall back to 0.0 in CELL 4')
else:
    import struct, os
    import numpy as np

    N = TERRAIN_GRID_N

    if FLAT_TERRAIN:
        print(f'Building FLAT terrain mesh: {N}×{N} grid ...')
        # Derive terrain extent from merged PLY bounds so terrain covers full scene
        _mdir = globals().get('MERGED_DIR', os.path.join(SCENE_DIR, 'meshes_merged'))
        _xmin, _xmax, _ymin, _ymax = 1e9, -1e9, 1e9, -1e9
        if os.path.isdir(_mdir):
            for _pf in os.listdir(_mdir):
                if not _pf.endswith('.ply') or 'terrain' in _pf: continue
                try:
                    import struct as _st
                    with open(os.path.join(_mdir, _pf), 'rb') as _f:
                        _hdr, _nv = [], 0
                        while True:
                            _l = _f.readline().decode('ascii','ignore').strip()
                            _hdr.append(_l)
                            if _l.startswith('element vertex'): _nv = int(_l.split()[-1])
                            if _l == 'end_header': break
                        if _nv > 0 and any('binary' in h for h in _hdr):
                            _raw = np.frombuffer(_f.read(_nv*12), dtype=np.float32).reshape(-1,3)
                            _xmin=min(_xmin,float(_raw[:,0].min())); _xmax=max(_xmax,float(_raw[:,0].max()))
                            _ymin=min(_ymin,float(_raw[:,1].min())); _ymax=max(_ymax,float(_raw[:,1].max()))
                except Exception: pass
        if _xmin < _xmax:
            _pad  = float(globals().get("TERRAIN_PAD_M", 5000.0))
            x_span = (_xmax - _xmin) + 2*_pad
            y_span = (_ymax - _ymin) + 2*_pad
            _ox = (_xmin + _xmax) / 2
            _oy = (_ymin + _ymax) / 2
            print(f'  Terrain from PLY bounds: {x_span:.0f} x {y_span:.0f} m  (pad={_pad}m)')
        else:
            _ox, _oy = 0.0, 0.0
            x_span = (SCENE_EAST - SCENE_WEST) * 111000 * np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
            y_span = (SCENE_NORTH - SCENE_SOUTH) * 111000
            print(f'  Terrain from bbox: {x_span:.0f} x {y_span:.0f} m')
        xs = np.linspace(_ox - x_span/2, _ox + x_span/2, N, dtype=np.float32)
        ys = np.linspace(_oy - y_span/2, _oy + y_span/2, N, dtype=np.float32)
        XX, YY = np.meshgrid(xs, ys, indexing='ij')
        ZZ = np.zeros((N, N), dtype=np.float32)
        origin_elev_asl = 0.0
        def local_z(lon, lat): return 0.0
    else:
        _src = globals().get('TERRAIN_SOURCE', 'aws_dem')
        if _src == 'ea_lidar' and os.path.exists(globals().get('EA_DTM_TIFF','')):
            print(f'Building terrain mesh from EA LiDAR DTM: {N}×{N} grid ...')
            import rasterio as _rio
            from pyproj import Transformer as _Tr
            _bng_to_utm = _Tr.from_crs('EPSG:27700', f'EPSG:{UTM_EPSG}', always_xy=True)
            _utm_to_bng = _Tr.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:27700', always_xy=True)
            _ds = _rio.open(EA_DTM_TIFF)
            _dem_data = _ds.read(1).astype(np.float32)
            _dem_tf   = _ds.transform
            _dem_nd   = _ds.nodata

            # scene origin elevation from EA DTM
            _ox_bng, _oy_bng = _utm_to_bng.transform(center_utm[0], center_utm[1])
            _oc, _or = ~_dem_tf * (_ox_bng, _oy_bng)
            _or, _oc = int(_or), int(_oc)
            if 0 <= _or < _dem_data.shape[0] and 0 <= _oc < _dem_data.shape[1]:
                origin_elev_asl = float(_dem_data[_or, _oc])
            else:
                origin_elev_asl = 0.0
            print(f'  Origin elevation: {origin_elev_asl:.1f} m ASL')

            def _ea_z(utm_x, utm_y):
                bx, by = _utm_to_bng.transform(utm_x, utm_y)
                col_f, row_f = ~_dem_tf * (bx, by)
                r, c = int(row_f), int(col_f)
                H, W = _dem_data.shape
                if 0 <= r < H-1 and 0 <= c < W-1:
                    dr, dc = row_f-r, col_f-c
                    z = ((1-dr)*(1-dc)*_dem_data[r,c] + (1-dr)*dc*_dem_data[r,c+1] +
                         dr*(1-dc)*_dem_data[r+1,c] + dr*dc*_dem_data[r+1,c+1])
                    if _dem_nd is None or not np.isclose(float(z), _dem_nd):
                        return float(z)
                return origin_elev_asl

            x_span = (SCENE_EAST-SCENE_WEST)*111000*np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2))
            y_span = (SCENE_NORTH-SCENE_SOUTH)*111000
            xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
            ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
            XX, YY = np.meshgrid(xs, ys, indexing='ij')
            ZZ = np.zeros((N, N), dtype=np.float32)
            import time as _t; t0 = _t.time()
            for i in range(N):
                for j in range(N):
                    ZZ[i,j] = _ea_z(center_utm[0]+XX[i,j], center_utm[1]+YY[i,j]) - origin_elev_asl
                if (i+1) % max(1,N//5)==0:
                    print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')
            def local_z(lon, lat):
                ux,uy = to_utm.transform(lon,lat)
                return _ea_z(ux,uy) - origin_elev_asl
        else:
            print(f'Building terrain mesh from AWS DEM: {N}×{N} grid ...')
            x_span = ne_utm[0] - sw_utm[0]
            y_span = ne_utm[1] - sw_utm[1]
            xs = np.linspace(-x_span/2, x_span/2, N, dtype=np.float32)
            ys = np.linspace(-y_span/2, y_span/2, N, dtype=np.float32)
            XX, YY = np.meshgrid(xs, ys, indexing='ij')
            ZZ = np.zeros((N, N), dtype=np.float32)
            import time as _t; t0 = _t.time()
            for i in range(N):
                for j in range(N):
                    utm_x = center_utm[0] + XX[i, j]
                    utm_y = center_utm[1] + YY[i, j]
                    ZZ[i, j] = height_from_utm(utm_x, utm_y) - origin_elev_asl
                if (i+1) % max(1, N//5) == 0:
                    print(f'  row {i+1}/{N}  ({_t.time()-t0:.0f}s)')

    print(f'Terrain Z range: [{ZZ.min():.1f}, {ZZ.max():.1f}] m')

    verts = np.stack([XX.ravel(), YY.ravel(), ZZ.ravel()], axis=1).astype(np.float32)
    faces = []
    for i in range(N-1):
        for j in range(N-1):
            a = i*N+j; b = a+1; c = a+N; d = c+1
            faces.append([a,d,b]); faces.append([a,c,d])  # normals point UP (+z)
    faces = np.array(faces, dtype=np.int32)

    # Write binary PLY
    ply_path = os.path.join(MESH_DIR, 'terrain.ply')  # always save alongside other PLYs
    with open(ply_path, 'wb') as f:
        hdr = (f'ply\nformat binary_little_endian 1.0\n'
               f'element vertex {len(verts)}\nproperty float x\nproperty float y\nproperty float z\n'
               f'element face {len(faces)}\nproperty list uchar int vertex_indices\nend_header\n')
        f.write(hdr.encode())
        f.write(verts.tobytes())
        for fc in faces:
            f.write(struct.pack('<B3i', 3, *fc))

    kb = os.path.getsize(ply_path)//1024
    print(f'terrain.ply: {len(verts):,} verts  {len(faces):,} faces  {kb} KB  → {ply_path}')

    # Save origin elevation for main notebook compatibility
    import json as _json
    _elev_path = os.path.join(SCENE_DIR, 'origin_elev2.json')
    _json.dump({'origin_elev_asl_m': float(origin_elev_asl), 'flat_terrain': FLAT_TERRAIN}, open(_elev_path, 'w'))
    print(f'origin_elev2.json: {_elev_path}')


## CELL 3b — Enriched Building Material Classification

Fetches full OSM building tags via Overpass API (free, no key) and applies a
priority-chain classifier to assign the most accurate ITU-R P.2040-2 material
for every building in the scene. Output saved to `building_materials_vlm.json`.
Cell 4 reads this as Priority 0 — overrides age/type heuristics.

**Priority chain:** OSM material tag → colour tag → use type → building age → geometry rules → fallback

**Run order:** Cell 3 → **Cell 3b** → Cell 4 → Cell B3 → Cell B1

In [ ]:
# ============================================================
# CELL 3b — ENRICHED BUILDING MATERIAL CLASSIFICATION
# Overpass API (free, no key) → per-building ITU-R P.2040-2 material
# Output: building_materials_vlm.json  {osm_id: itu_material}
# Cell 4 reads this as Priority 0 override.
# Compatible: Sionna 0.19 AND Sionna 2.0 (Cell 4 internal keys)
# ============================================================
import os, json, math, time, re

# ── 1. Download building tags via OSMnx — calibration zone only ─────────────
# Use 1.5 km radius around TX centre (covers all calibration receivers)
# Much smaller than full scene bbox — avoids sub-query fragmentation
import osmnx as ox
import math as _math

_TX_LAT   = TX_LAT_REF   # actual transmitter position (NOT scene bbox centre)
_TX_LON   = TX_LON_REF
_CALIB_KM = 1.5          # calibration radius — all 173 mat_calib_receivers
_dlat     = _CALIB_KM / 111.320
_dlon     = _CALIB_KM / (111.320 * _math.cos(_math.radians(_TX_LAT)))
_C_SOUTH  = _TX_LAT - _dlat
_C_NORTH  = _TX_LAT + _dlat
_C_WEST   = _TX_LON - _dlon
_C_EAST   = _TX_LON + _dlon

print('=' * 70)
print('CELL 3b — Enriched Building Material Classification')
print('=' * 70)
print(f'Calibration zone: {_CALIB_KM} km radius around TX')
print(f'  bbox: S={_C_SOUTH:.4f} W={_C_WEST:.4f} N={_C_NORTH:.4f} E={_C_EAST:.4f}')
print(f'Full scene bbox : S={SCENE_SOUTH:.4f} W={SCENE_WEST:.4f} '
      f'N={SCENE_NORTH:.4f} E={SCENE_EAST:.4f}')
print()

_BLD_TAGS = {}
_GDF_CACHE = os.path.join(BASE_DIR, 'buildings_gdf_cache.pkl')

# Strategy 1: read Cell 4 cache (full 77k buildings, no network call)
if os.path.exists(_GDF_CACHE):
    import pickle as _pkl
    print('Strategy 1: reading Cell 4 buildings cache (full scene coverage)')
    try:
        with open(_GDF_CACHE, 'rb') as _pf:
            _gdf = _pkl.load(_pf)
        print(f'  Loaded {len(_gdf):,} buildings from cache (full scene)')
        # Clip to the 1.5 km calibration zone — CELL 3b only needs buildings
        # near the TX, not the whole scene (full-scene cache made this loop
        # iterate 185k+ buildings and take hours).
        try:
            from shapely.geometry import box as _sbox
            _calib_box = _sbox(_C_WEST, _C_SOUTH, _C_EAST, _C_NORTH)
            _gdf = _gdf[_gdf.intersects(_calib_box)]
            print(f'  Clipped to calibration zone: {len(_gdf):,} buildings')
        except Exception as _e:
            print(f'  Clip failed ({_e}) — proceeding with full-scene set')
    except Exception as _e:
        print(f'  Cache read failed: {_e}')
        _gdf = None
else:
    # Strategy 2: OSMnx calibration zone (1.5 km) — no cache yet
    print('Strategy 2: Cell 4 cache not found — downloading 1.5 km zone via OSMnx')
    print('  Tip: run Cell 4 first to build cache for full 77k coverage')
    _gdf = None
    _ox_version = tuple(int(x) for x in ox.__version__.split('.')[:2])
    try:
        if _ox_version >= (2, 0):
            _gdf = ox.features_from_bbox(
                bbox=(_C_WEST, _C_SOUTH, _C_EAST, _C_NORTH),
                tags={'building': True}
            )
        elif _ox_version >= (1, 3):
            _gdf = ox.features_from_bbox(
                bbox=(_C_NORTH, _C_SOUTH, _C_EAST, _C_WEST),
                tags={'building': True}
            )
        else:
            _gdf = ox.features_from_bbox(
                north=_C_NORTH, south=_C_SOUTH,
                east=_C_EAST,   west=_C_WEST,
                tags={'building': True}
            )
        print(f'  Downloaded {len(_gdf):,} buildings (1.5 km zone)')
    except Exception as _e:
        print(f'  [ERROR] OSMnx download failed: {_e}')
        print('  Cell 3b skipped — Cell 4 uses existing heuristics')

if _gdf is not None:
    _tag_cols = [c for c in _gdf.columns
                 if ':' in c or c in ('building','amenity','shop',
                    'office','start_date','year_built','colour')]
    print(f'  Tag columns available: {_tag_cols}')
    for _idx, _row in _gdf.iterrows():
        try:
            _eid = str(_idx[1]) if isinstance(_idx, tuple) else str(_idx)
        except Exception:
            _eid = str(_idx)
        _tags = {c: str(_row[c]) for c in _gdf.columns
                 if c != 'geometry' and _row[c] is not None
                 and str(_row[c]) not in ('nan', 'None', '')}
        _BLD_TAGS[_eid] = _tags
    print(f'  Tag dict built: {len(_BLD_TAGS):,} buildings')

# ── 2. Material lookup tables ────────────────────────────────────────────────
# OSM building:material → ITU-R P.2040-2 key  (Cell 4 internal names)
_OSM_MAT_MAP = {
    'concrete': 'itu_concrete', 'reinforced_concrete': 'itu_concrete',
    'cement': 'itu_concrete',   'precast': 'itu_concrete',
    'stone': 'itu_brick',       'sandstone': 'itu_brick',
    'limestone': 'itu_brick',   'granite': 'itu_brick',
    'cobblestone': 'itu_brick', 'flint': 'itu_brick',
    'brick': 'itu_brick',       'clay': 'itu_brick',
    'glass': 'itu_glass',       'glazed': 'itu_glass',
    'metal': 'itu_metal',       'steel': 'itu_metal',
    'aluminium': 'itu_metal',   'aluminum': 'itu_metal',
    'zinc': 'itu_metal',        'copper': 'itu_metal',
    'corrugated_iron': 'itu_metal', 'iron': 'itu_metal', 'tin': 'itu_metal',
    'wood': 'itu_wood',         'timber': 'itu_wood',
    'wooden': 'itu_wood',       'thatch': 'itu_wood',
}

# Colour → material
_COLOUR_MAP = {
    'red': 'itu_brick',         'brown': 'itu_brick',
    'orange': 'itu_brick',      'salmon': 'itu_brick',
    'tan': 'itu_brick',         'terracotta': 'itu_brick',
    'rust': 'itu_brick',        'maroon': 'itu_brick',
    'grey': 'itu_concrete',     'gray': 'itu_concrete',
    'white': 'itu_concrete',    'cream': 'itu_concrete',
    'beige': 'itu_concrete',    'ivory': 'itu_concrete',
    'light_grey': 'itu_concrete','off-white': 'itu_concrete',
    'silver': 'itu_metal',      'aluminium': 'itu_metal',
    'aluminum': 'itu_metal',    'steel': 'itu_metal',
    'copper': 'itu_metal',      'bronze': 'itu_metal',
    'transparent': 'itu_glass', 'blue': 'itu_glass',
    'teal': 'itu_glass',        'yellow': 'itu_wood',
}

# Use type → material
_USE_GLASS    = {'greenhouse','glasshouse','winter_garden','conservatory'}
_USE_METAL    = {'industrial','warehouse','shed','hangar','barn',
                  'storage_tank','transformer_tower','service'}
_USE_CONCRETE = {'office','commercial','retail','supermarket',
                  'university','school','hospital','hotel',
                  'apartments','tower','civic'}
_USE_BRICK    = {'house','detached','semidetached_house','terrace',
                  'residential','bungalow','church','chapel',
                  'cathedral','mosque','synagogue'}

def _classify_building(tags):
    bmat    = tags.get('building:material', '').lower().strip()
    fmat    = tags.get('building:facade:material', '').lower().strip()
    colour  = (tags.get('building:colour','') or
               tags.get('building:color','')  or
               tags.get('colour','')).lower().strip()
    btype   = tags.get('building','').lower().strip()
    amenity = tags.get('amenity','').lower().strip()
    shop    = tags.get('shop','').lower().strip()
    office  = tags.get('office','').lower().strip()
    date    = tags.get('start_date', tags.get('year_built','')).strip()
    lvl_s   = tags.get('building:levels', tags.get('levels','')).strip()

    # P0: OSM building:material tag
    for _tag in (bmat, fmat):
        if _tag in _OSM_MAT_MAP:
            return _OSM_MAT_MAP[_tag], 'P0_osm_material'
        for k, v in _OSM_MAT_MAP.items():
            if k in _tag and _tag:
                return v, 'P0_osm_material_partial'

    # P1: building:colour tag
    if colour:
        for k, v in _COLOUR_MAP.items():
            if k in colour:
                return v, 'P1_colour'
        if colour.startswith('#') and len(colour) == 7:
            try:
                r = int(colour[1:3], 16)
                g = int(colour[3:5], 16)
                b = int(colour[5:7], 16)
                if r > 160 and r > g * 1.3 and r > b * 1.3:
                    return 'itu_brick',    'P1_colour_hex_red'
                if abs(r-g) < 30 and abs(g-b) < 30:
                    return 'itu_concrete', 'P1_colour_hex_grey'
            except Exception:
                pass

    # P2: use type
    if btype in _USE_GLASS or amenity in _USE_GLASS or shop in _USE_GLASS:
        return 'itu_glass',    'P2_use_glass'
    if btype in _USE_METAL:
        return 'itu_metal',    'P2_use_metal'
    if btype in _USE_BRICK:
        return 'itu_brick',    'P2_use_brick'
    if btype in _USE_CONCRETE or office or shop in ('supermarket','mall','department_store'):
        return 'itu_concrete', 'P2_use_concrete'

    # P3: building age
    _year = None
    if date:
        _m = re.search(r'(\d{4})', date)
        if _m:
            _year = int(_m.group(1))
    if _year:
        if _year < 1940:  return 'itu_brick',    'P3_age_pre1940'
        if _year < 1980:  return 'itu_concrete',  'P3_age_1940_1980'
        return                    'itu_brick',    'P3_age_post1980'

    # P4: geometry (levels)
    _lvl = 0
    try:    _lvl = int(float(lvl_s)) if lvl_s else 0
    except: pass
    if _lvl >= 8:
        return 'itu_concrete', 'P4_geom_tower'
    if _lvl <= 2 and btype in ('', 'yes', 'residential', 'house', 'terrace'):
        return 'itu_brick',    'P4_geom_low_residential'

    return None, None   # no override — Cell 4 fallback keeps existing logic

# ── 3. Classify all buildings ────────────────────────────────────────────────
_overrides       = {}
_priority_counts = {}

for _osm_id, _tags in _BLD_TAGS.items():
    _mat, _pri = _classify_building(_tags)
    if _mat is not None:
        _overrides[_osm_id] = _mat
        _priority_counts[_pri] = _priority_counts.get(_pri, 0) + 1

print(f'\nClassification results:')
print(f'  Total buildings queried : {len(_BLD_TAGS):,}')
print(f'  Overrides generated     : {len(_overrides):,}  '
      f'({100*len(_overrides)/max(len(_BLD_TAGS),1):.1f}%)')
print(f'  No override (fallback)  : {len(_BLD_TAGS) - len(_overrides):,}')
print()
print('  By priority level:')
for _p in sorted(_priority_counts):
    print(f'    {_p:<32} {_priority_counts[_p]:>6,}')
print()
print('  Material distribution:')
_mat_counts = {}
for _m in _overrides.values():
    _mat_counts[_m] = _mat_counts.get(_m, 0) + 1
for _m, _c in sorted(_mat_counts.items(), key=lambda x: -x[1]):
    print(f'    {_m:<22} {_c:>6,}  ({100*_c/len(_overrides):.1f}%)')

# ── 4. Save to building_materials_vlm.json ───────────────────────────────────
_OUT_PATH = os.path.join(BASE_DIR, 'building_materials_vlm.json')
with open(_OUT_PATH, 'w') as _fout:
    json.dump(_overrides, _fout, indent=2)

print(f'\nSaved : {_OUT_PATH}')
print(f'        {len(_overrides):,} building overrides')
print()
print('Next: re-run Cell 4 — it will load building_materials_vlm.json as Priority 0')


## CELL 4 — Build OSM Feature PLYs

Downloads and builds PLY meshes for all scene features:

**Buildings** — height priority:
1. LiDAR nDSM at footprint centroid (`USE_NDSM_HEIGHTS=True`)
2. OSM `height=` tag
3. OSM `building:levels × HEIGHT_PER_LEVEL_M`
4. `DEFAULT_HEIGHT_M` fallback

**Roads / Water** — flat polygons at terrain height with appropriate materials.

**Vegetation** — controlled by `VEG_3D_GEOMETRY`:
- `True` (recommended): hollow canopy shell — perimeter walls + flat roof cap at
  canopy height (open bottom). Height capped at `VEG_HEIGHT_CAP_M` (default 10 m).
  Horizontal rays at RX height diffract/scatter at forest edge — physically correct.
  Material: itu_vegetation (S=0.40, εr=1.5, σ=0.0019 S/m).
- `False` (legacy): flat ground patches at terrain height — horizontal rays pass
  through unattenuated. Use with P.833 Weissberger post-hoc in DEM notebook.


In [ ]:
# ============================================================
# CELL 4 — OSM BUILDINGS → BUILDING PLYS
# ============================================================

# Guard: local_z may not be defined if terrain cells were skipped
if 'local_z' not in dir():
    if 'height_from_wgs84' in dir() and 'origin_elev_asl' in dir():
        def local_z(lon, lat):
            return height_from_wgs84(lon, lat) - origin_elev_asl
    else:
        def local_z(lon, lat):
            return 0.0
        print("[WARN] local_z fallback: returning 0.0 (terrain cells not run)")

assert _HAS_OSMNX, 'osmnx required: pip install osmnx'

import osmnx as _ox_ver
_ox_version = tuple(int(x) for x in _ox_ver.__version__.split('.')[:2])

print(f'Downloading OSM buildings (osmnx {_ox_ver.__version__}) ...')
t0 = time.time()
try:
    if _ox_version >= (2, 0):
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
            tags={'building': True}
        )
    elif _ox_version >= (1, 3):
        gdf_bld = ox.features_from_bbox(
            bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
            tags={'building': True}
        )
    else:
        gdf_bld = ox.features_from_bbox(
            north=SCENE_NORTH, south=SCENE_SOUTH,
            east=SCENE_EAST,   west=SCENE_WEST,
            tags={'building': True}
        )
except Exception as e:
    print(f'osmnx error: {e}')
    raise
print(f'  {len(gdf_bld)} raw building features  ({time.time()-t0:.1f}s)')
# ── Cache gdf_bld for Cell 3b (full coverage, no extra network call) ─────
import pickle as _pkl
_GDF_CACHE = os.path.join(BASE_DIR, 'buildings_gdf_cache.pkl')
try:
    with open(_GDF_CACHE, 'wb') as _pf: _pkl.dump(gdf_bld, _pf)
    print(f'  buildings_gdf_cache.pkl saved ({len(gdf_bld):,} buildings)')
except Exception as _pe:
    print(f'  [WARN] cache save failed: {_pe}')

# ── Download building:part (OSM Simple 3D Buildings) ────────────────────────
print('Downloading OSM building parts (S3DB) ...')
try:
    if _ox_version >= (2, 0):
        gdf_parts = ox.features_from_bbox(
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
            tags={'building:part': True})
    elif _ox_version >= (1, 3):
        gdf_parts = ox.features_from_bbox(
            bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
            tags={'building:part': True})
    else:
        gdf_parts = ox.features_from_bbox(
            north=SCENE_NORTH, south=SCENE_SOUTH,
            east=SCENE_EAST, west=SCENE_WEST, tags={'building:part': True})
    _parts_poly = gdf_parts[gdf_parts.geometry.geom_type.isin(['Polygon','MultiPolygon'])]
    _parts_sindex = _parts_poly.sindex if len(_parts_poly) else None
    print(f'  {len(_parts_poly)} building part polygons')
except Exception as _ep:
    print(f'  Building parts download failed: {_ep}')
    _parts_poly = None; _parts_sindex = None

# ── Microsoft Global Building Footprints merge ──────────────────────────────
# Add MS-detected buildings not already covered by OSM footprints (>40% overlap = duplicate)
import pandas as _pd_ms
_gdf_ms_merge = None
if 'MS_GPKG' in dir() and MS_GPKG and os.path.exists(str(MS_GPKG)):
    try:
        _gdf_ms_merge = gpd.read_file(MS_GPKG)
    except Exception:
        pass
elif 'gdf_ms_raw' in dir() and gdf_ms_raw is not None and len(gdf_ms_raw):
    _gdf_ms_merge = gdf_ms_raw

if _gdf_ms_merge is not None and len(_gdf_ms_merge):
    _osm_si = gdf_bld.sindex
    _ms_add = []
    for _, _mr in _gdf_ms_merge.iterrows():
        _mg = _mr.geometry
        if _mg is None or _mg.is_empty:
            continue
        _hits = list(_osm_si.intersection(_mg.bounds))
        _dup = False
        for _hi in _hits[:10]:
            _og = gdf_bld.iloc[_hi].geometry
            if _og is not None and not _og.is_empty:
                try:
                    if _og.intersection(_mg).area > 0.4 * _mg.area:
                        _dup = True; break
                except Exception:
                    pass
        if not _dup:
            _ms_add.append({'geometry': _mg, 'building': 'yes',
                            'height': _mr.get('ms_height', None)})
    if _ms_add:
        _gdf_add = gpd.GeoDataFrame(_ms_add, geometry='geometry', crs='EPSG:27700')
        gdf_bld = _pd_ms.concat([gdf_bld, _gdf_add], ignore_index=True)
        gdf_bld = gpd.GeoDataFrame(gdf_bld, geometry='geometry', crs='EPSG:27700')
        print(f'MS footprints: +{len(_ms_add)} new buildings added to gdf_bld')
    else:
        print('MS footprints: all covered by OSM')
else:
    print('MS footprints: not available (run CELL 2d-ms-footprints first)')

# ── Material helpers ───────────────────────────────────────────────────────
def _parse_year(date_str):
    """Extract 4-digit year from OSM start_date= tag. Returns int or None."""
    import re
    if not date_str or str(date_str).lower() in ('nan', 'none', ''):
        return None
    m = re.search(r'\b(1[0-9]{3}|20[0-2][0-9])\b', str(date_str))
    return int(m.group(1)) if m else None

def _bld_mat(row, _height=None):
    # ── Priority 0: VLM/Overpass enriched override (Cell 3b output) ────────
    _osm_id = str(row.get('osmid', row.get('element_id', '')))
    if _BLD_MAT_OVERRIDE and _osm_id in _BLD_MAT_OVERRIDE:
        return _BLD_MAT_OVERRIDE[_osm_id]

    mat  = str(row.get('building:material', '')).lower()
    fmat = str(row.get('building:facade:material', '')).lower()
    tag  = str(row.get('building', '')).lower()
    amen = str(row.get('amenity', '')).lower()
    shop = str(row.get('shop', '')).lower()
    off  = str(row.get('office', '')).lower()
    # ── Priority 0: EPC wall description (measured construction data) ──────────
    if _epc_sindex is not None:
        try:
            _epc_mat = _epc_material(row.geometry)
            if _epc_mat is not None:
                return _epc_mat
        except Exception:
            pass
    # ── Priority 0.5: Conservation area → Victorian/Edwardian → itu_brick ────
    if _ca_sindex is not None:
        try:
            _cen_ca = row.geometry.centroid
            _ca_hits = list(_ca_sindex.intersection(_cen_ca.buffer(0.0001).bounds))
            if _ca_hits and any(_ca_gdf.iloc[_h].geometry.contains(_cen_ca) for _h in _ca_hits):
                return 'itu_brick'
        except Exception:
            pass
    # ── Priority 1: OSM building:material= tag (highest confidence, universal) ─
    _MAT_TAG_MAP = {
        # Concrete/masonry
        'concrete': 'itu_concrete', 'reinforced_concrete': 'itu_concrete',
        'cement': 'itu_concrete',  'precast': 'itu_concrete',
        # Brick/stone (similar EM properties)
        'brick': 'itu_brick', 'stone': 'itu_brick', 'sandstone': 'itu_brick',
        'limestone': 'itu_brick', 'granite': 'itu_brick', 'flint': 'itu_brick',
        'cobblestone': 'itu_brick', 'clay': 'itu_brick',
        # Wood
        'wood': 'itu_wood', 'timber': 'itu_wood', 'wooden': 'itu_wood',
        'logs': 'itu_wood', 'bamboo': 'itu_wood', 'thatch': 'itu_wood',
        'log': 'itu_wood',
        # Glass
        'glass': 'itu_glass', 'glazed': 'itu_glass',
        # Metal
        'metal': 'itu_metal', 'steel': 'itu_metal', 'aluminium': 'itu_metal',
        'aluminum': 'itu_metal', 'tin': 'itu_metal', 'copper': 'itu_metal',
        'zinc': 'itu_metal', 'iron': 'itu_metal', 'corrugated_iron': 'itu_metal',
    }
    for _mkey, _mval in _MAT_TAG_MAP.items():
        if _mkey in mat or _mkey in fmat: return _mval

    # ── Priority 1.5: height-based for unlabelled tall buildings ─────────────
    if _height is not None and not mat and not fmat:
        if _height > 40.0 and tag in ('commercial', 'office', 'retail', 'yes', ''):
            return 'itu_glass'
        if _height > 15.0 and (tag in ('commercial', 'office') or off):
            return 'itu_concrete'
    # ── Priority 2: building use-type → material (global mapping) ────────────
    # Glazed/curtain-wall structures
    if tag in ('greenhouse', 'glasshouse', 'winter_garden'):    return 'itu_glass'
    if amen in ('shopping_centre', 'mall'):                     return 'itu_glass'
    if shop in ('mall', 'supermarket', 'department_store'):     return 'itu_glass'

    # Metal-frame structures
    _METAL_TYPES = {'industrial','warehouse','factory','shed','barn','hangar',
                    'storage_tank','silo','container','quonset_hut','stable'}
    if tag in _METAL_TYPES: return 'itu_metal'

    # Residential — use age if available, else brick (universal default)
    _RESID_TYPES = {'residential','house','detached','semidetached_house',
                    'semi_detached','terrace','terrace_house','bungalow','hut',
                    'farm','farmhouse','dormitory','apartments','block','cabin',
                    'villa','cottage','chalet','mobile_home'}
    if tag in _RESID_TYPES: _type_mat = 'itu_brick'

    # Retail/commercial
    elif tag in ('retail','commercial','supermarket','kiosk','shop',
                 'convenience','mall'):  _type_mat = 'itu_brick'

    # Religious (stone/brick globally)
    elif tag in ('cathedral','church','chapel','mosque','temple','shrine',
                 'synagogue','monastery','convent'):  _type_mat = 'itu_brick'

    # Civic/institutional → concrete frame (post-1950 global standard)
    elif tag in ('school','university','hospital','civic','public','college',
                 'library','museum','government','courthouse','prison',
                 'police','fire_station','train_station','bus_station',
                 'airport_terminal','sports_centre','stadium'):
        _type_mat = 'itu_concrete'

    # Office — curtain wall or concrete
    elif off: _type_mat = 'itu_concrete'

    # Default — concrete (most common globally for unlabelled buildings)
    else: _type_mat = 'itu_concrete'

    # ── Priority 3: age-based override (Jansen et al. 2019) ──────────────────
    # Pre-1940: solid brick  |  1940-1980: concrete panel  |  post-1980: type-based
    if globals().get('USE_BUILDING_AGE', True):
        _yr = _parse_year(str(row.get('start_date', '') or ''))
        if _yr is not None:
            if _yr < 1940: return 'itu_brick'
            if _yr < 1980: return 'itu_concrete'
    # Building age override — Jansen et al. 2019 wall material classification
    # Pre-1940: Victorian/Edwardian solid brick (30 cm walls, high transmission loss)
    # 1940-1980: post-war concrete panel construction
    # Post-1980: modern construction — fall back to type-based above
    if globals().get('USE_BUILDING_AGE', False):
        _yr = _parse_year(str(row.get('start_date', '') or ''))
        if _yr is not None:
            if _yr < 1940:  return 'itu_brick'
            if _yr < 1980:  return 'itu_concrete'
    return _type_mat

def _roof_mat(row):
    roof_tag = str(row.get('roof:material', '')).lower()
    btag     = str(row.get('building', '')).lower()
    if any(k in roof_tag for k in ['metal','steel','zinc','aluminium','copper','tin']):
        return 'itu_metal'
    if 'glass' in roof_tag:                              return 'itu_glass'
    if any(k in roof_tag for k in ['wood','timber','thatch']): return 'itu_wood'
    if any(k in roof_tag for k in ['tile','concrete','slate','terracotta']):
        return 'itu_concrete'
    if btag in ('industrial','warehouse','factory','shed','barn',
                'retail','supermarket','commercial','garage','garages'):
        return 'itu_metal'
    return 'itu_metal'   # fallback: open-land sheds/barns/houses -> metal roofs

# ── nDSM sampler (open once, reuse for all buildings) ─────────────────────
_ndsm_ds  = None
_ndsm_arr = None
_ndsm_tfm = None
_ndsm_crs = None
_ndsm_nd  = None
_ndsm_to_raster = None
if USE_NDSM_HEIGHTS and os.path.exists(NDSM_TIFF):
    import rasterio as _rio_n
    from rasterio.windows import from_bounds as _win_from_bounds
    from pyproj import Transformer as _TrNcrop
    _ndsm_ds  = _rio_n.open(NDSM_TIFF)
    _ndsm_crs = _ndsm_ds.crs
    _ndsm_nd  = _ndsm_ds.nodata
    _to_nds_crop = _TrNcrop.from_crs('EPSG:4326', _ndsm_crs.to_epsg() or 27700, always_xy=True)
    _bx0, _by0 = _to_nds_crop.transform(SCENE_WEST, SCENE_SOUTH)
    _bx1, _by1 = _to_nds_crop.transform(SCENE_EAST, SCENE_NORTH)
    _buf_n = 200
    _crop_win = _win_from_bounds(min(_bx0,_bx1)-_buf_n, min(_by0,_by1)-_buf_n,
                                  max(_bx0,_bx1)+_buf_n, max(_by0,_by1)+_buf_n,
                                  transform=_ndsm_ds.transform).round_offsets().round_lengths()
    _ndsm_arr = _ndsm_ds.read(1, window=_crop_win).astype(float)
    _ndsm_tfm = _ndsm_ds.window_transform(_crop_win)
    _ndsm_to_raster = _to_nds_crop
    print(f'  nDSM loaded: {NDSM_TIFF}  (cropped to scene window: {_ndsm_arr.shape}, '
          f'{_ndsm_arr.nbytes/1e9:.2f} GB)')
elif USE_NDSM_HEIGHTS:
    print(f'  WARNING: ndsm.tif not found — run CELL 2d first. Falling back to OSM tags.')

# ── Conservation area spatial index ─────────────────────────────────────────
_ca_gdf = None; _ca_sindex = None
_ca_path = globals().get('CA_GPKG', os.path.join(LIDAR_DIR, 'conservation_areas.gpkg'))
if _ca_path and os.path.exists(str(_ca_path)):
    try:
        import geopandas as _gpd_ca
        _ca_gdf = _gpd_ca.read_file(str(_ca_path))
        _ca_sindex = _ca_gdf.sindex
        print(f'  Conservation areas loaded: {len(_ca_gdf)}')
    except Exception as _eca:
        print(f'  Conservation areas load failed: {_eca}')

def _ndsm_height(lon, lat):
    if _ndsm_ds is None:
        return None
    from rasterio.transform import rowcol as _rc
    try:
        ex, ny = _ndsm_to_raster.transform(lon, lat)
        r, c = int(_rc(_ndsm_tfm, ex, ny)[0]), int(_rc(_ndsm_tfm, ex, ny)[1])
        if 0 <= r < _ndsm_arr.shape[0] and 0 <= c < _ndsm_arr.shape[1]:
            v = float(_ndsm_arr[r, c])
            if _ndsm_nd is not None and np.isclose(v, float(_ndsm_nd)): return None
            if v >= NDSM_BUILDING_MIN_M:
                return float(np.clip(v, max(CITY_MIN_HEIGHT_M, NDSM_BUILDING_MIN_M), 500.0))
    except Exception:
        pass
    return None

def _ndsm_height_footprint(geom, percentile=90):
    if _ndsm_ds is None:
        return None
    try:
        from rasterio.transform import rowcol as _rc2
        from shapely.ops import transform as _shp_tr
        geom_r = _shp_tr(_ndsm_to_raster.transform, geom)
        minx, miny, maxx, maxy = geom_r.bounds
        rows_mm = sorted([int(_rc2(_ndsm_tfm, minx, maxy)[0]), int(_rc2(_ndsm_tfm, maxx, miny)[0])])
        cols_mm = sorted([int(_rc2(_ndsm_tfm, minx, maxy)[1]), int(_rc2(_ndsm_tfm, maxx, miny)[1])])
        H, W = _ndsm_arr.shape
        r0, r1 = max(0, rows_mm[0]-1), min(H, rows_mm[1]+2)
        c0, c1 = max(0, cols_mm[0]-1), min(W, cols_mm[1]+2)
        if r0 >= r1 or c0 >= c1: return None
        patch = _ndsm_arr[r0:r1, c0:c1]
        if patch.size <= 2:
            cen = geom.centroid
            return _ndsm_height(cen.x, cen.y)
        from rasterio.features import geometry_mask as _gmask
        from affine import Affine as _Aff
        sub_tfm = _Aff(_ndsm_tfm.a, _ndsm_tfm.b, _ndsm_tfm.c + c0 * _ndsm_tfm.a,
                       _ndsm_tfm.d, _ndsm_tfm.e, _ndsm_tfm.f + r0 * _ndsm_tfm.e)
        in_poly = ~_gmask([geom_r.__geo_interface__], transform=sub_tfm,
                          out_shape=patch.shape, invert=False)
        vals = patch[in_poly].astype(float)
        if _ndsm_nd is not None:
            vals = vals[~np.isclose(vals, float(_ndsm_nd))]
        vals = vals[vals >= NDSM_BUILDING_MIN_M]
        if len(vals) == 0: return None
        h = float(np.percentile(vals, percentile))
        return float(np.clip(h, max(CITY_MIN_HEIGHT_M, NDSM_BUILDING_MIN_M), 500.0))
    except Exception:
        try:
            cen = geom.centroid
            return _ndsm_height(cen.x, cen.y)
        except Exception:
            return None

def _bld_height(row):
    # 1. OSM height= tag (surveyed, most accurate)
    try:
        h = float(str(row.get('height','0')).replace('m','').strip())
        if h > 1: return np.clip(h, CITY_MIN_HEIGHT_M, 500.0)
    except: pass
    # 2. OSM building:levels
    try:
        lvl = float(str(row.get('building:levels','0')).strip())
        if lvl > 0: return np.clip(lvl * HEIGHT_PER_LEVEL_M, CITY_MIN_HEIGHT_M, 500.0)
    except: pass
    # 3. nDSM footprint percentile — robust against rooftop artefacts
    if USE_NDSM_HEIGHTS:
        try:
            h = _ndsm_height_footprint(row.geometry)
            if h is not None:
                return h
        except Exception:
            pass
    return DEFAULT_HEIGHT_M

def _roof_height(row, coords_utm):
    """Ridge height (m) above the eave + roof shape string.
    Priority: roof:shape=flat -> flat; roof:height tag; roof:levels;
    else derive from ROOF_PITCH_DEG and footprint short dimension."""
    shape = str(row.get('roof:shape', '')).lower().strip()
    if shape == 'flat':
        return 0.0, 'flat'
    try:
        rh = float(str(row.get('roof:height', '0')).replace('m', '').strip())
        if rh > 0.2:
            return float(np.clip(rh, 0.0, ROOF_MAX_RIDGE_M)), (shape or 'pyramidal')
    except Exception:
        pass
    try:
        rl = float(str(row.get('roof:levels', '0')).strip())
        if rl > 0:
            return float(np.clip(rl * 1.5, 0.0, ROOF_MAX_RIDGE_M)), (shape or 'pyramidal')
    except Exception:
        pass
    # LiDAR point cloud roof shape (when laspy + tiles available, no OSM tag)
    if not shape and '_pc_roof_shape' in dir():
        try:
            _pc = _pc_roof_shape(row.geometry)
            if _pc is not None:
                if _pc['shape'] == 'flat':
                    return 0.0, 'flat'
                elif _pc['shape'] == 'pitched':
                    _pa_pc = np.asarray(coords_utm, dtype=float)
                    _span_pc = min(_pa_pc[:,0].max()-_pa_pc[:,0].min(),
                                   _pa_pc[:,1].max()-_pa_pc[:,1].min())
                    _rh_pc = (_span_pc/2.0)*math.tan(math.radians(_pc['pitch_deg']))
                    return float(np.clip(_rh_pc, 0.0, ROOF_MAX_RIDGE_M)), 'pyramidal'
        except Exception:
            pass
    try:
        pa = np.asarray(coords_utm, dtype=float)
        span = min(pa[:, 0].max() - pa[:, 0].min(), pa[:, 1].max() - pa[:, 1].min())
        rh = (span / 2.0) * math.tan(math.radians(ROOF_PITCH_DEG))
        return float(np.clip(rh, 0.0, ROOF_MAX_RIDGE_M)), (shape or 'pyramidal')
    except Exception:
        return 0.0, 'flat'

def _write_ply(verts, faces, path):
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    if _HAS_TRIMESH:
        trimesh.Trimesh(vertices=verts, faces=faces, process=False).export(path)
        return
    with open(path, 'w') as f:
        f.write('ply\nformat ascii 1.0\n')
        f.write(f'element vertex {len(verts)}\n')
        f.write('property float x\nproperty float y\nproperty float z\n')
        f.write(f'element face {len(faces)}\n')
        f.write('property list uchar int vertex_indices\nend_header\n')
        for v in verts:   f.write(f'{v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n')
        for fc in faces:  f.write(f'3 {fc[0]} {fc[1]} {fc[2]}\n')

def _triangulate_roof(pts_2d, top_z):
    """
    Robust Delaunay roof triangulation via Shapely.
    Handles convex AND concave footprints — no degenerate triangles.
    """
    poly = sg.Polygon(pts_2d)
    if not poly.is_valid:
        poly = poly.buffer(0)   # auto-repair self-intersections
    if poly.is_empty or poly.area < 0.1:
        return None

    try:
        tris = so.triangulate(poly)
    except Exception:
        return None

    v_idx = {}
    roof_v = []
    roof_f = []

    for tri in tris:
        # Keep only triangles whose centroid lies inside the footprint
        if not poly.contains(tri.centroid):
            continue
        coords = list(tri.exterior.coords)[:3]
        face_idxs = []
        for cx, cy in coords:
            key = (round(cx, 3), round(cy, 3))
            if key not in v_idx:
                v_idx[key] = len(roof_v)
                roof_v.append([cx, cy, top_z])
            face_idxs.append(v_idx[key])
        # Skip zero-area triangles (cross product check)
        a = np.array(roof_v[face_idxs[1]][:2]) - np.array(roof_v[face_idxs[0]][:2])
        b = np.array(roof_v[face_idxs[2]][:2]) - np.array(roof_v[face_idxs[0]][:2])
        if abs(a[0]*b[1] - a[1]*b[0]) > 1e-6:
            roof_f.append(face_idxs)

    if not roof_v or not roof_f:
        return None
    return np.array(roof_v, np.float32), np.array(roof_f, np.int32)

def _make_sloped_roof(pts_2d, top_z, roof_h, shape='gabled'):
    """Gabled/hipped roof via PCA long-axis ridge line. Works for any convex/concave footprint."""
    _pts = np.array(pts_2d, dtype=float)
    _n = len(_pts)
    if _n < 3: return None
    _poly_sr = sg.Polygon(_pts)
    if not _poly_sr.is_valid: _poly_sr = _poly_sr.buffer(0)
    if _poly_sr.is_empty: return None
    try:
        _mrr = _poly_sr.minimum_rotated_rectangle
        _mc = np.array(_mrr.exterior.coords[:-1])
        _d0 = np.linalg.norm(_mc[1] - _mc[0])
        _d1 = np.linalg.norm(_mc[2] - _mc[1])
        _lv = (_mc[1] - _mc[0]) if _d0 >= _d1 else (_mc[2] - _mc[1])
        _lu = _lv / (np.linalg.norm(_lv) + 1e-9)
    except Exception:
        return None
    _cen = np.array([_poly_sr.centroid.x, _poly_sr.centroid.y])
    _rel = _pts - _cen
    _prj = _rel @ _lu
    _pmin, _pmax = _prj.min(), _prj.max()
    if shape == 'hipped':
        _ins = min((_pmax - _pmin) * 0.2, 2.5)
        _pmin += _ins; _pmax -= _ins
        if _pmin >= _pmax: _pmin, _pmax = _pmin - 0.5, _pmin + 0.5
    _rz = top_z + roof_h
    _verts = [[x, y, top_z] for x, y in _pts]
    for _i in range(_n):
        _pc = float(np.clip(_prj[_i], _pmin, _pmax))
        _rpt = _cen + _pc * _lu
        _verts.append([float(_rpt[0]), float(_rpt[1]), _rz])
    _faces = []
    for _i in range(_n):
        _j = (_i + 1) % _n
        _ri = _n + _i; _rj = _n + _j
        _rv_i = _verts[_ri]; _rv_j = _verts[_rj]
        if abs(_rv_i[0] - _rv_j[0]) < 1e-3 and abs(_rv_i[1] - _rv_j[1]) < 1e-3:
            _faces.append([_i, _j, _ri])
        else:
            _faces.append([_i, _j, _rj])
            _faces.append([_i, _rj, _ri])
    if not _faces: return None
    return np.array(_verts, np.float32), np.array(_faces, np.int32)

def _extrude_building(poly_utm, base_z, height, roof_h=0.0, roof_shape=''):
    pts = np.array(poly_utm, dtype=np.float32)
    if len(pts) < 3:
        return None
    if np.allclose(pts[0], pts[-1]):
        pts = pts[:-1]
    n = len(pts)
    top_z = base_z + height
    bot_z = base_z

    # ── Walls ──────────────────────────────────────────────────────────────
    wall_v, wall_f = [], []
    for i in range(n):
        j = (i+1) % n
        v_base = len(wall_v)
        wall_v += [
            [pts[i,0], pts[i,1], bot_z],
            [pts[j,0], pts[j,1], bot_z],
            [pts[j,0], pts[j,1], top_z],
            [pts[i,0], pts[i,1], top_z],
        ]
        wall_f += [
            [v_base,   v_base+1, v_base+2],
            [v_base,   v_base+2, v_base+3],
        ]

    # ── Roof ──────────────────────────────────────────────────────────────
    # Pitched (pyramidal/hip) roof when roof_h>0 and shape!='flat'; else a
    # flat Delaunay cap. The apex sits at the footprint representative point
    # raised by roof_h, so the roof is gap-free for any footprint and
    # presents sloped faces for realistic diffuse scattering.
    if roof_h > 0.2 and roof_shape in ('gabled', 'hipped'):
        _sr = _make_sloped_roof([(pts[i,0], pts[i,1]) for i in range(n)], top_z, roof_h, roof_shape)
        if _sr is not None:
            rv, rf = _sr
        else:
            roof_result = _triangulate_roof([(pts[i,0], pts[i,1]) for i in range(n)], top_z)
            if roof_result is None: return None
            rv, rf = roof_result
    elif roof_h > 0.2 and roof_shape not in ('flat',):
        cx, cy = float(pts[:,0].mean()), float(pts[:,1].mean())
        try:
            _c = sg.Polygon([(pts[i,0], pts[i,1]) for i in range(n)]).representative_point()
            cx, cy = float(_c.x), float(_c.y)
        except Exception:
            pass
        rv = [[pts[i,0], pts[i,1], top_z] for i in range(n)]
        apex = len(rv)
        rv.append([cx, cy, top_z + roof_h])
        rf = [[i, (i+1) % n, apex] for i in range(n)]
        rv = np.array(rv, np.float32); rf = np.array(rf, np.int32)
    else:
        roof_result = _triangulate_roof([(pts[i,0], pts[i,1]) for i in range(n)], top_z)
        if roof_result is None:
            return None
        rv, rf = roof_result
    return (np.array(wall_v, np.float32), np.array(wall_f, np.int32), rv, rf)

# ── Load building material overrides from Cell 3b (if available) ────────────
_bld_mat_path   = os.path.join(BASE_DIR, 'building_materials_vlm.json')
_BLD_MAT_OVERRIDE = {}
if os.path.exists(_bld_mat_path):
    with open(_bld_mat_path) as _bmf:
        _BLD_MAT_OVERRIDE = json.load(_bmf)
    print(f'[Cell 3b] Building material overrides loaded: {len(_BLD_MAT_OVERRIDE):,}')
else:
    print('[Cell 3b] building_materials_vlm.json not found — using heuristics only')

# ── Process buildings — accumulate geometry per material ────────────────────
mat_geom = {}   # mat -> {'verts': [], 'faces': [], 'offset': 0}
n_ok = n_skip = 0
_n_total = len(gdf_bld)
t0 = time.time()

for idx, (oid, row) in enumerate(gdf_bld.iterrows()):
    geom = row.geometry
    if geom is None or geom.is_empty:
        n_skip += 1; continue

    if isinstance(geom, MultiPolygon):
        sub_geoms = list(geom.geoms)
    elif isinstance(geom, Polygon):
        sub_geoms = [geom]
    else:
        n_skip += 1; continue
    geom = sub_geoms[0]

    # Skip buildings fully covered by building:part features
    if _parts_sindex is not None:
        try:
            _ph = list(_parts_sindex.intersection(geom.bounds))
            if _ph and any(_parts_poly.iloc[_pi].geometry.intersects(geom.centroid.buffer(1.0)) for _pi in _ph[:5]):
                n_skip += 1; continue
        except Exception: pass

    btag = str(row.get('building','')).lower()
    if btag in EXCLUDE_BUILDING_TYPES:
        n_skip += 1; continue

    coords_wgs = list(geom.exterior.coords)
    coords_utm = []
    for lon, lat in coords_wgs:
        ex, ny = to_utm.transform(lon, lat)
        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))

    area_m2 = sg.Polygon(coords_utm).area
    if area_m2 < MIN_BUILDING_AREA_M2:
        n_skip += 1; continue

    # Inward buffer: shrink footprint to prevent GPS-drifted receivers
    # landing inside building meshes (urban-canyon GPS error ~5-15 m).

    c_lon, c_lat = geom.centroid.x, geom.centroid.y
    base_z_local = local_z(c_lon, c_lat)
    h = _bld_height(row)
    roof_h, roof_shape = _roof_height(row, coords_utm)

    result = _extrude_building(coords_utm, base_z_local, h, roof_h, roof_shape)
    if result is None:
        n_skip += 1; continue
    wv, wf, rv, rf = result

    w_mat = _bld_mat(row, h)
    r_mat = _roof_mat(row)

    for mat, verts, faces in [(w_mat, wv, wf), (r_mat, rv, rf)]:
        if mat not in mat_geom:
            mat_geom[mat] = {'verts': [], 'faces': [], 'offset': 0}
        g = mat_geom[mat]
        g['faces'].append(faces + g['offset'])
        g['verts'].append(verts)
        g['offset'] += len(verts)

    n_ok += 1
    if n_ok % 100 == 0:
        print(f'  {n_ok}/{_n_total} buildings  ({time.time()-t0:.0f}s elapsed) ...')

print(f'\nBuildings : {n_ok} exported, {n_skip} skipped')
print(f'Materials : {list(mat_geom.keys())}')

# ── Process building:part features (S3DB multi-height volumes) ──────────────
if _parts_poly is not None and len(_parts_poly):
    print(f'Processing {len(_parts_poly)} building parts ...')
    _n_parts_ok = 0
    for _, _prow in _parts_poly.iterrows():
        _pgeom = _prow.geometry
        if _pgeom is None or _pgeom.is_empty: continue
        _psubs = list(_pgeom.geoms) if isinstance(_pgeom, MultiPolygon) else [_pgeom]
        for _pg in _psubs:
            if not isinstance(_pg, Polygon): continue
            _pc_utm = []
            for _plon, _plat in _pg.exterior.coords:
                _ex, _ny = to_utm.transform(_plon, _plat)
                _pc_utm.append((_ex - center_utm[0], _ny - center_utm[1]))
            if sg.Polygon(_pc_utm).area < MIN_BUILDING_AREA_M2: continue
            _plon_c, _plat_c = _pg.centroid.x, _pg.centroid.y
            _pbz = local_z(_plon_c, _plat_c)
            _ph = _bld_height(_prow)
            _prh, _prs = _roof_height(_prow, _pc_utm)
            _pres = _extrude_building(_pc_utm, _pbz, _ph, _prh, _prs)
            if _pres is None: continue
            _pwv, _pwf, _prv, _prf = _pres
            _pwm = _bld_mat(_prow, _ph); _prm = _roof_mat(_prow)
            for _pm, _pverts, _pfaces in [(_pwm, _pwv, _pwf), (_prm, _prv, _prf)]:
                if _pm not in mat_geom:
                    mat_geom[_pm] = {'verts': [], 'faces': [], 'offset': 0}
                _pg2 = mat_geom[_pm]
                _pg2['faces'].append(_pfaces + _pg2['offset'])
                _pg2['verts'].append(_pverts)
                _pg2['offset'] += len(_pverts)
            _n_parts_ok += 1
    print(f'  Building parts: {_n_parts_ok} exported')

# ── Write one merged PLY per material ───────────────────────────────────────
mat_plys = {}
for mat, g in mat_geom.items():
    all_v = np.concatenate(g['verts'], axis=0).astype(np.float32)
    all_f = np.concatenate(g['faces'], axis=0).astype(np.int32)
    ply_name = f'bld_{mat}.ply'
    ply_path = os.path.join(MESH_DIR, ply_name)
    _write_ply(all_v, all_f, ply_path)
    mat_plys[mat] = [('meshes/' + ply_name, 'buildings')]
    print(f'  {mat}: {len(all_v)} verts, {len(all_f)} faces → {ply_name}')

print(f'\nPLY files written: {len(mat_plys)} (one per material)')

# ── Roads → per-material PLYs (terrain-following, bridge-aware) ──────────────
if INCLUDE_ROADS:
    print('\nDownloading OSM roads ...')
    try:
        _road_tags = {'highway': True}
        if _ox_version >= (2, 0):
            gdf_roads = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_road_tags)
        elif _ox_version >= (1, 3):
            gdf_roads = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_road_tags)
        else:
            gdf_roads = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_road_tags)

        # ── Width: width= tag > lanes x lane-width > highway-type default ──────
        _LANE_W = {'motorway':3.65,'trunk':3.65,'primary':3.0,'secondary':3.0,
                   'tertiary':2.75,'residential':2.75,'unclassified':2.75}
        _HW_W   = {'motorway':14.6,'trunk':11.0,'primary':9.0,'secondary':7.3,
                   'tertiary':6.5,'residential':5.5,'unclassified':5.0,
                   'service':3.5,'living_street':4.5,'pedestrian':6.0,
                   'footway':2.0,'cycleway':2.5,'path':1.5,'track':3.5,'road':5.0}

        def _road_width(row):
            _hw = str(row.get('highway','') or '').lower()
            # Priority 1: explicit measured width tag
            _wt = str(row.get('width','') or '').split(';')[0].strip().replace('m','').replace(' ','')
            if _wt:
                try: return float(_wt)
                except ValueError: pass
            # Priority 2: lanes x UK standard lane width
            _lt = str(row.get('lanes','') or '').split(';')[0].strip()
            if _lt:
                try:
                    _nl = int(float(_lt))
                    if _nl > 0:
                        return _nl * _LANE_W.get(_hw, 2.75)
                except ValueError: pass
            # Priority 3: highway-type default
            return _HW_W.get(_hw, 4.5)

        # ── Material: surface= tag > highway type ─────────────────────────────
        _SURF_MAT = {
            'asphalt':'itu_asphalt','paved':'itu_asphalt','tar':'itu_asphalt',
            'concrete':'itu_concrete','cement':'itu_concrete',
            'cobblestone':'itu_concrete','sett':'itu_concrete',
            'paving_stones':'itu_concrete','unhewn_cobblestone':'itu_concrete',
            'gravel':'itu_medium_dry_ground','fine_gravel':'itu_medium_dry_ground',
            'compacted':'itu_medium_dry_ground','pebblestone':'itu_medium_dry_ground',
            'unpaved':'itu_medium_dry_ground','dirt':'itu_medium_dry_ground',
            'ground':'itu_medium_dry_ground','grass':'itu_medium_dry_ground',
            'earth':'itu_medium_dry_ground','mud':'itu_medium_dry_ground',
            'sand':'itu_medium_dry_ground','wood':'itu_wood','metal':'itu_metal',
        }
        _FOOT_HW = {'footway','path','bridleway','steps','track'}

        def _road_material(row):
            _surf = str(row.get('surface','') or '').lower().strip()
            if _surf in _SURF_MAT: return _SURF_MAT[_surf]
            _hw = str(row.get('highway','') or '').lower()
            if _hw in _FOOT_HW: return 'itu_medium_dry_ground'
            return 'itu_asphalt'

        # ── Bridge elevation: layer= tag x 5m clearance per layer ────────────
        _BRIDGE_CLEAR = 5.0  # metres per OSM layer above ground

        def _bridge_z(row, cx_scene, cy_scene):
            if str(row.get('bridge','') or '').lower() not in ('yes','viaduct','aqueduct'):
                return None
            try:
                _lon, _lat = to_wgs84.transform(
                    cx_scene + center_utm[0], cy_scene + center_utm[1])
                _terrain = local_z(_lon, _lat)
            except Exception:
                _terrain = 0.0

            # Priority 1: nDSM at bridge centroid — actual LiDAR deck height
            if _ndsm_arr is not None:
                try:
                    _bh = _ndsm_height(_lon, _lat)
                    if _bh is not None and _bh > 1.5:
                        return _terrain + float(_bh)
                except Exception:
                    pass

            # Priority 2: OSM ele= tag — absolute elevation (ASL)
            _ele = str(row.get('ele','') or '').strip().replace('m','')
            if _ele:
                try:
                    return float(_ele) - (_terrain + float(globals().get('origin_elev_asl', 0.0)))
                except (ValueError, TypeError):
                    pass

            # Priority 3: layer= x standard clearance (fallback only)
            try:
                _layer = int(float(str(row.get('layer','1') or '1')))
            except (ValueError, TypeError):
                _layer = 1
            return _terrain + max(1, _layer) * _BRIDGE_CLEAR

        from shapely.geometry import LineString as _LS, MultiLineString as _MLS
        _road_mats = {}  # mat -> {'verts':[], 'faces':[], 'off':0}

        for _, row in gdf_roads.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty: continue
            if globals().get('EXCLUDE_TUNNELS', True) and \
               str(row.get('tunnel','') or '').lower() in ('yes','building_passage','avalanche_protector'):
                continue

            _hw   = str(row.get('highway','') or '').lower()
            _half = _road_width(row) / 2.0
            _mat  = _road_material(row)
            _is_bridge = str(row.get('bridge','') or '').lower() in ('yes','viaduct','aqueduct')

            if _mat not in _road_mats:
                _road_mats[_mat] = {'verts': [], 'faces': [], 'off': 0}
            _rd = _road_mats[_mat]

            segs = geom.geoms if isinstance(geom, _MLS) else [geom]
            for seg in segs:
                if not isinstance(seg, _LS): continue
                coords_utm = []
                for lon, lat in seg.coords:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                if len(coords_utm) < 2: continue

                poly = sg.LineString(coords_utm).buffer(_half, cap_style=2, join_style=2)
                if poly.is_empty: continue
                _road_polys = list(poly.geoms) if poly.geom_type == 'MultiPolygon' else [poly]

                for _rpoly in _road_polys:
                    if _rpoly.is_empty: continue
                    pts2d = np.array(_rpoly.exterior.coords[:-1], np.float32)
                    n = len(pts2d)
                    if n < 3: continue

                    # Terrain-following z per vertex (DTM lookup in WGS84)
                    zs = np.zeros(n, np.float32)
                    if not FLAT_TERRAIN:
                        for _vi, (vx, vy) in enumerate(pts2d):
                            try:
                                _vlon, _vlat = to_wgs84.transform(
                                    vx + center_utm[0], vy + center_utm[1])
                                zs[_vi] = local_z(_vlon, _vlat)
                            except Exception:
                                pass

                    # Bridge: override z to bridge deck elevation
                    if _is_bridge:
                        cx2 = float(pts2d[:,0].mean())
                        cy2 = float(pts2d[:,1].mean())
                        _bz = _bridge_z(row, cx2, cy2)
                        if _bz is not None:
                            zs[:] = _bz

                    verts = np.hstack([pts2d, zs.reshape(-1,1)])
                    cx2, cy2 = pts2d[:,0].mean(), pts2d[:,1].mean()
                    cz2 = float(zs.mean())
                    base = _rd['off']
                    _rd['verts'].append(verts)
                    _rd['verts'].append(np.array([[cx2, cy2, cz2]], np.float32))
                    for _fi in range(n):
                        _rd['faces'].append([base+_fi, base+(_fi+1)%n, base+n])
                    _rd['off'] += n + 1

        _road_total_v = 0
        for _mat, _rd in _road_mats.items():
            if not _rd['verts']: continue
            _rv = np.concatenate(_rd['verts']).astype(np.float32)
            _rf = np.array(_rd['faces'], np.int32)
            _pname = f'road_{_mat}.ply'
            _write_ply(_rv, _rf, os.path.join(MESH_DIR, _pname))
            _entry = ('meshes/' + _pname, 'roads')
            if _mat in mat_plys:
                mat_plys[_mat].append(_entry)
            else:
                mat_plys[_mat] = [_entry]
            _road_total_v += len(_rv)
            print(f'  Roads [{_mat}]: {len(_rv):,} verts, {len(_rf):,} faces -> {_pname}')
        if not _road_total_v:
            print('  Roads: no geometry generated')
    except Exception as _e:
        import traceback; traceback.print_exc()
        print(f'  Roads download failed: {_e}')
else:
    print('Roads: skipped (INCLUDE_ROADS=False)')

# ── Water → itu_water PLY ────────────────────────────────────────────────────
if INCLUDE_WATER:
    print('\nDownloading OSM water ...')
    try:
        _water_tags = {'natural': ['water','river','stream','lake'],
                       'waterway': ['river','stream','canal','drain']}
        if _ox_version >= (2, 0):
            gdf_water = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_water_tags)
        elif _ox_version >= (1, 3):
            gdf_water = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_water_tags)
        else:
            gdf_water = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_water_tags)
        from shapely.geometry import Polygon as _WPoly, MultiPolygon as _WMPoly
        from shapely.geometry import LineString as _WLS, MultiLineString as _WMLS
        # Waterway half-widths from OSM width= tag (m); type-based realistic UK defaults
        _WATERWAY_HW = {'river': 15.0, 'tidal_channel': 30.0, 'canal': 5.5,
                        'stream': 2.0, 'drain': 1.0, 'ditch': 0.5}
        def _water_half_w(row):
            _w = str(row.get('width', '') or '').strip()
            if _w:
                try: return float(_w.split()[0]) / 2.0
                except: pass
            return _WATERWAY_HW.get(str(row.get('waterway', '') or '').lower(), 3.0)
        w_verts, w_faces, w_off = [], [], 0
        for _, row in gdf_water.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty: continue
            polys = []
            if isinstance(geom, _WPoly): polys = [geom]
            elif isinstance(geom, _WMPoly): polys = list(geom.geoms)
            elif isinstance(geom, (_WLS, _WMLS)):
                for line in (geom.geoms if isinstance(geom, _WMLS) else [geom]):
                    # buffer in UTM metres (5 m half-width) not degrees
                    _lcoords = []
                    for lon, lat in line.coords:
                        ex, ny = to_utm.transform(lon, lat)
                        _lcoords.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_lcoords) >= 2:
                        wp = sg.LineString(_lcoords).buffer(_water_half_w(row))
                        if not wp.is_empty: polys.append(wp)
                    continue   # already in UTM; skip second conversion below
            for poly in polys:
                if poly.is_empty: continue
                # flatten MultiPolygon into individual polygons
                from shapely.geometry import MultiPolygon as _MP2
                sub_polys = list(poly.geoms) if isinstance(poly, _MP2) else [poly]
                for poly in sub_polys:
                    if poly.exterior is None or poly.is_empty: continue
                # check if already UTM (from line buffer above) or still WGS84
                _first = poly.exterior.coords[0]
                _is_utm = abs(_first[0]) > 180 or abs(_first[1]) > 90
                if _is_utm:
                    coords_utm = [(x, y) for x, y in poly.exterior.coords[:-1]]
                else:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                n = len(coords_utm)
                if n < 3: continue
                if sg.Polygon(coords_utm).area < 1.0: continue   # m^2
                pts = np.array(coords_utm, np.float32)
                verts = np.hstack([pts, np.zeros((n,1), np.float32)])
                cx, cy = pts[:,0].mean(), pts[:,1].mean()
                base = w_off
                w_verts.append(verts)
                w_verts.append(np.array([[cx, cy, 0.0]], np.float32))
                for i in range(n):
                    w_faces.append([base+i, base+(i+1)%n, base+n])
                w_off += n + 1
        if w_verts:
            wv_all = np.concatenate(w_verts).astype(np.float32)
            wf_all = np.array(w_faces, np.int32)
            ply_name = 'water_itu_water.ply'
            _write_ply(wv_all, wf_all, os.path.join(MESH_DIR, ply_name))
            mat_plys['itu_water'] = [('meshes/' + ply_name, 'water')]
            print(f'  Water: {len(wv_all):,} verts, {len(wf_all):,} faces → {ply_name}')
        else:
            print('  Water: no geometry generated')
    except Exception as _e:
        print(f'  Water download failed: {_e}')
else:
    print('Water: skipped (INCLUDE_WATER=False)')

# ── Vegetation → itu_vegetation PLY ─────────────────────────────────────────
if INCLUDE_VEGETATION:
    print('\nDownloading OSM vegetation ...')
    try:
        _veg_tags = {'landuse': ['forest', 'orchard', 'recreation_ground', 'allotments'],
                     'natural': ['wood', 'scrub'],
                     'leisure': ['park', 'nature_reserve', 'garden']}
        if _ox_version >= (2, 0):
            gdf_veg = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_veg_tags)
        elif _ox_version >= (1, 3):
            gdf_veg = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_veg_tags)
        else:
            gdf_veg = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_veg_tags)
        from shapely.geometry import Polygon as _VPoly, MultiPolygon as _VMPoly

        # Merge greenspace.gpkg (richer boundaries) into gdf_veg
        _gs_path = globals().get('GS_GPKG', os.path.join(LIDAR_DIR, 'greenspace.gpkg'))
        if _gs_path and os.path.exists(str(_gs_path)):
            try:
                import geopandas as _gpd_gs
                import pandas as _pd_gs
                _gs_extra = _gpd_gs.read_file(str(_gs_path), layer='greenspace')
                _gs_extra = _gs_extra[_gs_extra.geometry.notna()].copy()
                # Spatial dedup: drop rows whose centroid is already inside an OSM polygon
                _veg_sidx = gdf_veg.sindex if len(gdf_veg) else None
                if _veg_sidx is not None:
                    _keep_gs = []
                    for _, _gr in _gs_extra.iterrows():
                        _hits = list(_veg_sidx.intersection(_gr.geometry.bounds))
                        _dup = any(gdf_veg.iloc[_h].geometry.contains(_gr.geometry.centroid)
                                   for _h in _hits)
                        _keep_gs.append(not _dup)
                    _gs_extra = _gs_extra[_keep_gs]
                if len(_gs_extra):
                    gdf_veg = _pd_gs.concat([gdf_veg, _gs_extra], ignore_index=True)
                    print(f'  + {len(_gs_extra)} greenspace polygons merged into vegetation')
            except Exception as _eg:
                print(f'  Greenspace merge skipped: {_eg}')

        vg_verts, vg_faces, vg_off = [], [], 0
        _veg_min  = float(globals().get('VEG_MIN_AREA_M2', 50.0))
        _veg_hmap = dict(globals().get('VEG_CLASS_HEIGHT_M', {}))
        _veg_tall = float(globals().get('VEG_TALL_MIN_M', 1.5))
        _veg_def  = float(globals().get('VEG_CANOPY_M', 12.0))

        # ── VOM sampler (vegetation heights per polygon) ─────────────────────────
        _vom_ds  = None
        _vom_arr = None
        _vom_tfm = None
        _vom_nd  = None
        _vom_to_raster = None
        _vom_path_c4 = globals().get('VOM_TIFF', os.path.join(LIDAR_DIR, 'vom.tif'))
        if _vom_path_c4 and os.path.exists(str(_vom_path_c4)):
            import rasterio as _rio_vom_c4
            from pyproj import Transformer as _TrVomC4
            _vom_ds  = _rio_vom_c4.open(str(_vom_path_c4))
            _vom_nd  = _vom_ds.nodata or 0.0
            _vom_arr = _vom_ds.read(1)
            _vom_crs_epsg = _vom_ds.crs.to_epsg() or 27700
            _vom_to_raster = _TrVomC4.from_crs('EPSG:4326', f'EPSG:{_vom_crs_epsg}', always_xy=True)
            print(f'VOM loaded: {_vom_arr.shape}  nodata={_vom_nd}')
        else:
            print('VOM not available — vegetation heights from nDSM only')

        # ── EPC spatial index (wall/glazing/roof material) ──────────────────
        _epc_sindex = None
        _epc_gdf_c4 = None
        _epc_path_c4 = globals().get('EPC_GPKG', os.path.join(LIDAR_DIR, 'epc_materials.gpkg'))
        if _epc_path_c4 and os.path.exists(str(_epc_path_c4)):
            try:
                import geopandas as _gpd_epc_c4
                _epc_gdf_c4 = _gpd_epc_c4.read_file(str(_epc_path_c4), layer='epc')
                _epc_sindex = _epc_gdf_c4.sindex
                print(f'EPC loaded: {len(_epc_gdf_c4):,} records')
            except Exception as _e_epc:
                print(f'EPC load failed: {_e_epc}')
        else:
            print('EPC not available -- using age/type heuristic')

        # ── NFI woodland spatial index ────────────────────────────────────
        _nfi_sindex = None
        _nfi_gdf_c4 = None
        _nfi_path_c4 = globals().get('NFI_GPKG', os.path.join(LIDAR_DIR, 'nfi_woodland.gpkg'))
        if _nfi_path_c4 and os.path.exists(str(_nfi_path_c4)):
            try:
                import geopandas as _gpd_nfi_c4
                _nfi_gdf_c4 = _gpd_nfi_c4.read_file(str(_nfi_path_c4), layer='nfi')
                _nfi_sindex = _nfi_gdf_c4.sindex
                print(f'NFI loaded: {len(_nfi_gdf_c4)} woodland polygons')
            except Exception as _e_nfi:
                print(f'NFI load failed: {_e_nfi}')
        else:
            print('NFI not available -- woodland heights from nDSM/VOM')

        def _epc_material(geom):
            """Return ITU material from nearest EPC record within 25m, or None."""
            if _epc_sindex is None or _epc_gdf_c4 is None:
                return None
            try:
                _cen = geom.centroid
                _buf = _cen.buffer(0.0003)  # ~25m in degrees
                _hits = list(_epc_sindex.intersection(_buf.bounds))
                if not _hits:
                    return None
                _near = _epc_gdf_c4.iloc[_hits]
                _dists = _near.geometry.distance(_cen)
                _best = _near.iloc[_dists.argmin()]
                if _dists.min() > 0.0003:
                    return None
                _walls = str(_best.get('walls', '') or '').lower()
                _glaz  = str(_best.get('glazing', '') or '').lower()
                # Map EPC walls description to ITU material
                if 'curtain' in _walls or ('glass' in _walls and 'glazed' in _walls):
                    return 'itu_glass'
                if 'metal' in _walls or 'steel' in _walls or 'cladding' in _walls:
                    return 'itu_metal'
                if 'concrete' in _walls or 'precast' in _walls or 'system built' in _walls:
                    return 'itu_concrete'
                if 'brick' in _walls or 'stone' in _walls or 'cavity' in _walls:
                    return 'itu_brick'
                if 'timber' in _walls or 'wood' in _walls:
                    return 'itu_wood'
            except Exception:
                pass
            return None

        def _vom_height(lon, lat):
            if _vom_arr is None or _vom_to_raster is None:
                return None
            try:
                _ex, _ny = _vom_to_raster.transform(lon, lat)
                _row, _col = _vom_ds.index(_ex, _ny)
                if 0 <= _row < _vom_arr.shape[0] and 0 <= _col < _vom_arr.shape[1]:
                    _v = float(_vom_arr[_row, _col])
                    if _v > 2.5 and _v != float(_vom_nd):
                        return _v
            except Exception:
                pass
            return None

        def _veg_height(row):
            _cen = row.geometry.centroid
            # 1. VOM at polygon centroid (vegetation-only LiDAR — no building contamination)
            try:
                _vh = _vom_height(_cen.x, _cen.y)
                if _vh is not None:
                    return float(np.clip(_vh, 0.0, 35.0))
            except Exception:
                pass
            # 1.5. NFI field-surveyed canopy height for large woodland polygons
            if _nfi_sindex is not None:
                try:
                    _nfi_hits = list(_nfi_sindex.intersection(_cen.buffer(0.0001).bounds))
                    for _ni in _nfi_hits:
                        _nfi_row = _nfi_gdf_c4.iloc[_ni]
                        if _nfi_row.geometry.contains(_cen):
                            _nfi_h = float(_nfi_row.get('canopy_h', 0) or 0)
                            if _nfi_h > 1.0:
                                return float(np.clip(_nfi_h, 0.0, 35.0))
                except Exception:
                    pass
            # 2. nDSM multi-point sampling -- 15 random points, median of [1.5-35m]
            #    statistically removes building spikes without predefined cap
            if _ndsm_arr is not None:
                try:
                    import random as _rv
                    from shapely.geometry import Point as _PtV
                    _bv = row.geometry.bounds
                    _hpts = []
                    _att = 0
                    while len(_hpts) < 15 and _att < 75:
                        _px = _rv.uniform(_bv[0], _bv[2])
                        _py = _rv.uniform(_bv[1], _bv[3])
                        _att += 1
                        try:
                            if row.geometry.contains(_PtV(_px, _py)):
                                _hv = _ndsm_height(_px, _py)
                                if _hv is not None and 1.5 <= _hv <= 35.0:
                                    _hpts.append(_hv)
                        except Exception:
                            pass
                    if len(_hpts) >= 2:
                        return float(np.median(_hpts))
                    elif len(_hpts) == 1:
                        return float(_hpts[0])
                    # centroid fallback for small/narrow polygons (street trees etc.)
                    _h0 = _ndsm_height(_cen.x, _cen.y)
                    if _h0 is not None and 1.5 <= _h0 <= 35.0:
                        return float(_h0)
                except Exception:
                    pass
            # 3. OSM class tag → VEG_CLASS_HEIGHT_M dict (natural, landuse, leisure)
            for _k in ('natural', 'landuse', 'leisure'):
                _t = str(row.get(_k, '') or '').lower()
                if _t in _veg_hmap:
                    return _veg_hmap[_t]
            # 3. VEG_CANOPY_M default
            return _veg_def
        _veg_skip = 0
        for _, row in gdf_veg.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty: continue
            polys = []
            if isinstance(geom, _VPoly): polys = [geom]
            elif isinstance(geom, _VMPoly): polys = list(geom.geoms)
            for poly in polys:
                if poly.is_empty: continue
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                n = len(coords_utm)
                if n < 3: continue
                if sg.Polygon(coords_utm).area < _veg_min: continue   # m^2
                # VEG_3D_GEOMETRY=True : hollow canopy shell (walls + roof cap, open bottom)
                #   height capped at VEG_HEIGHT_CAP_M — diffracts/scatters horizontal rays.
                # VEG_3D_GEOMETRY=False: flat patch at terrain height, no horizontal blocking
                #   use with P.833 post-hoc attenuation in DEM notebook.
                try:
                    _cen = poly.centroid
                    base_z = 0.0 if FLAT_TERRAIN else float(local_z(_cen.x, _cen.y))
                except Exception:
                    base_z = 0.0
                if not VEG_3D_GEOMETRY:
                    # flat fan triangulation at terrain height
                    _n = len(coords_utm)
                    _vp = np.array([[x, y, base_z] for x, y in coords_utm], np.float32)
                    for _fi in range(_n - 2):
                        vg_faces.append([vg_off, vg_off + _fi + 1, vg_off + _fi + 2])
                    vg_verts.append(_vp)
                    vg_off += _n
                    continue
                # ── canopy shell extrusion (VEG_3D_GEOMETRY=True) ──────────────────
                _veg_h = min(_veg_height(row), float(globals().get('VEG_HEIGHT_CAP_M', 20.0)))
                if _veg_h < _veg_tall:
                    _veg_skip += 1; continue
                _res = _extrude_building(coords_utm, base_z, _veg_h, roof_h=0.0, roof_shape='flat')
                if _res is None: continue
                _wv, _wf, _rv, _rf = _res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0:
                        continue
                    _base = vg_off
                    _arr = np.asarray(_vv, np.float32)
                    vg_verts.append(_arr)
                    for _f in _ff:
                        vg_faces.append([_base+int(_f[0]), _base+int(_f[1]), _base+int(_f[2])])
                    vg_off += len(_arr)
                # ── Internal vegetation grid for large parks ───────────────────────
                # Fills interior with a regular grid of canopy columns so rays crossing
                # mid-park are blocked (not just the perimeter shell).
                _grid_m   = float(globals().get('VEG_GRID_CELL_M', 30.0))
                _grid_min = float(globals().get('VEG_GRID_MIN_AREA', 5000.0))
                _poly_local = sg.Polygon(coords_utm)
                if _grid_m > 0 and _poly_local.area >= _grid_min:
                    _xmin, _ymin, _xmax, _ymax = _poly_local.bounds
                    _xs = np.arange(_xmin + _grid_m/2, _xmax, _grid_m)
                    _ys = np.arange(_ymin + _grid_m/2, _ymax, _grid_m)
                    _half = _grid_m / 2.0 * 0.85   # slightly smaller than cell to leave gaps
                    _n_grid = 0
                    for _gx in _xs:
                        for _gy in _ys:
                            _cell_pt = sg.Point(_gx, _gy)
                            if not _poly_local.contains(_cell_pt):
                                continue
                            # nDSM per-cell height lookup (WGS84 round-trip)
                            _cell_h = _veg_h
                            if _ndsm_arr is not None:
                                try:
                                    _cell_lon, _cell_lat = to_wgs84.transform(
                                        center_utm[0] + _gx, center_utm[1] + _gy)
                                    _nd = _ndsm_height(_cell_lon, _cell_lat)
                                    if _nd is not None:
                                        if _nd < _veg_tall:
                                            continue  # open lawn/path — no tree
                                        _cell_h = min(_nd, float(globals().get(
                                            'VEG_HEIGHT_CAP_M', 20.0)))
                                except Exception:
                                    pass
                            try:
                                _gz = 0.0 if FLAT_TERRAIN else float(local_z(
                                    center_utm[0] + _gx, center_utm[1] + _gy))
                            except Exception:
                                _gz = base_z
                            _cell_coords = [
                                (_gx - _half, _gy - _half),
                                (_gx + _half, _gy - _half),
                                (_gx + _half, _gy + _half),
                                (_gx - _half, _gy + _half),
                            ]
                            _cr = _extrude_building(_cell_coords, _gz, _cell_h,
                                                    roof_h=0.0, roof_shape='flat')
                            if _cr is None: continue
                            for _vv2, _ff2 in ((_cr[0], _cr[1]), (_cr[2], _cr[3])):
                                if _vv2 is None or len(_vv2) == 0: continue
                                _b2 = vg_off
                                _a2 = np.asarray(_vv2, np.float32)
                                vg_verts.append(_a2)
                                for _f2 in _ff2:
                                    vg_faces.append([_b2+int(_f2[0]), _b2+int(_f2[1]), _b2+int(_f2[2])])
                                vg_off += len(_a2)
                            _n_grid += 1
        if not VEG_3D_GEOMETRY:
            print(f'  Vegetation: {len(vg_faces):,} flat ground patches '
                  f'(VEG_3D_GEOMETRY=False — P.833 attenuation in DEM notebook)')
        elif VEG_3D_GEOMETRY:
            print(f'  Vegetation: canopy shell geometry — walls+cap, height capped at {globals().get("VEG_HEIGHT_CAP_M",10.0):.0f}m')
        elif _veg_skip > 0:
            print(f'  Vegetation: skipped {_veg_skip} ground-cover polygons (< {_veg_tall} m)')
        if vg_verts:
            vv_all = np.concatenate(vg_verts).astype(np.float32)
            vf_all = np.array(vg_faces, np.int32)
            ply_name = 'veg_itu_vegetation.ply'
            _write_ply(vv_all, vf_all, os.path.join(MESH_DIR, ply_name))
            mat_plys['itu_vegetation'] = [('meshes/' + ply_name, 'vegetation')]
            print(f'  Vegetation: {len(vv_all):,} verts, {len(vf_all):,} faces → {ply_name}')
            print(f'  Vegetation: skipped {_veg_skip} ground-cover polygons (< {_veg_tall} m)')
        else:
            print('  Vegetation: no geometry generated')
    except Exception as _e:
        print(f'  Vegetation download failed: {_e}')
else:
    print('Vegetation: skipped (INCLUDE_VEGETATION=False)')


# ── Individual trees → itu_vegetation disks ─────────────────────────────────
# OSM natural=tree points → flat disk at canopy height → itu_vegetation material
# Represents per-tree diffraction / scattering obstacle (ITU-R P.833-10 §3)
# Uses nDSM-sampled height when available; falls back to TREE_DEFAULT_HT_M
if globals().get('INCLUDE_TREES', True):
    print('\nDownloading OSM individual trees ...')
    try:
        _tree_tags = {'natural': 'tree'}
        if _ox_version >= (2, 0):
            gdf_trees = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_tree_tags)
        elif _ox_version >= (1, 3):
            gdf_trees = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_tree_tags)
        else:
            gdf_trees = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_tree_tags)
        _tree_def_h = float(globals().get('TREE_DEFAULT_HT_M',  8.0))
        _tree_r     = float(globals().get('TREE_DISK_RADIUS_M', 3.0))
        _tree_segs  = 8      # octagon disk — enough for RF, fast to build
        tv_verts, tv_faces, tv_off = [], [], 0
        _tree_count = 0
        for _, trow in gdf_trees.iterrows():
            geom = trow.geometry
            if geom is None or geom.is_empty: continue
            # Accept point geometry only (individual trees, not rows/areas)
            if geom.geom_type != 'Point': continue
            tx_utm, ty_utm = to_utm.transform(geom.x, geom.y)
            lx = tx_utm - center_utm[0]
            ly = ty_utm - center_utm[1]
            # Canopy height: OSM height= tag → nDSM sample → default
            _ht = None
            _h_tag = str(trow.get('height', '') or '')
            if _h_tag and _h_tag not in ('nan','None',''):
                try: _ht = float(_h_tag.replace('m','').strip())
                except ValueError: pass
            if _ht is None and _ndsm_arr is not None:
                try: _ht = _ndsm_height(geom.x, geom.y)  # nDSM = height above ground
                except Exception: pass
            if _ht is None or _ht < 1.5:
                _ht = _tree_def_h
            # Build octagon disk at canopy height (flat horizontal polygon)
            _angles = [2 * np.pi * k / _tree_segs for k in range(_tree_segs)]
            _disk   = np.array(
                [[lx + _tree_r * np.cos(a), ly + _tree_r * np.sin(a), _ht]
                 for a in _angles], dtype=np.float32)
            tv_verts.append(_disk)
            for fi in range(_tree_segs - 2):
                tv_faces.append([tv_off, tv_off + fi + 1, tv_off + fi + 2])
            tv_off += _tree_segs
            _tree_count += 1
        if tv_verts:
            # Merge with existing vegetation PLY if present, else create new
            _tv_v = np.concatenate(tv_verts).astype(np.float32)
            _tv_f = np.array(tv_faces, np.int32)
            _tree_ply = 'trees_itu_vegetation.ply'
            _write_ply(_tv_v, _tv_f, os.path.join(MESH_DIR, _tree_ply))
            if 'itu_vegetation' in mat_plys:
                mat_plys['itu_vegetation'].append(('meshes/' + _tree_ply, 'trees'))
            else:
                mat_plys['itu_vegetation'] = [('meshes/' + _tree_ply, 'trees')]
            print(f'  Trees: {_tree_count} individual trees → {_tree_ply} '
                  f'({len(_tv_v):,} verts, {len(_tv_f):,} faces)')
        else:
            print('  Trees: no individual tree points found in scene bbox')
    except Exception as _e:
        print(f'  Trees download failed: {_e}')
else:
    print('Trees: skipped (INCLUDE_TREES=False)')

# ── Bridges → itu_concrete PLY ───────────────────────────────────────────────
if globals().get('INCLUDE_BRIDGES', True):   # enabled: structural bridge deck polygons
    print('\nDownloading OSM bridges ...')
    try:
        # man_made=bridge returns actual bridge STRUCTURE polygons (deck footprints).
        # bridge=True returns road ways tagged bridge=yes (LineStrings) — no geometry.
        _br_tags = {'man_made': 'bridge'}
        if _ox_version >= (2, 0):
            gdf_br = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_br_tags)
        elif _ox_version >= (1, 3):
            gdf_br = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_br_tags)
        else:
            gdf_br = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH,
                                            east=SCENE_EAST, west=SCENE_WEST, tags=_br_tags)
        _br_polys = gdf_br[gdf_br.geometry.geom_type.isin(['Polygon','MultiPolygon'])]
        print(f'  {len(_br_polys)} bridge structure polygons')
        _br_v, _br_f, _br_off = [], [], 0
        _br_count = 0
        _DECK_THICKNESS = 1.5   # structural slab thickness (m) — depth of concrete cross-section
        def _br_ndsm_z(poly, base_z):
            """Sample nDSM at ~20 perimeter + centroid points, return p75 deck height or None."""
            if _ndsm_arr is None:
                return None
            try:
                bdry = poly.exterior
                _n_pts = 20
                _hs = []
                for _t in [i / _n_pts for i in range(_n_pts)]:
                    _pt = bdry.interpolate(_t, normalized=True)
                    _h = _ndsm_height(_pt.x, _pt.y)
                    if _h is not None and _h > 1.5:
                        _hs.append(_h)
                _hc = _ndsm_height(poly.centroid.x, poly.centroid.y)
                if _hc is not None and _hc > 1.5:
                    _hs.append(_hc)
                if _hs:
                    return base_z + float(np.percentile(_hs, 75))
            except Exception:
                pass
            return None
        for _, row in _br_polys.iterrows():
            try:
                geom = row.geometry
                polys = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for poly in polys:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    _cen = poly.centroid
                    base_z = local_z(_cen.x, _cen.y)
                    # Deck elevation: nDSM perimeter p75 > OSM ele= > layer-based last resort
                    slab_base = _br_ndsm_z(poly, base_z)
                    if slab_base is None:
                        _ele_br = str(row.get('ele', '') or '').strip().replace('m', '')
                        if _ele_br:
                            try:
                                slab_base = float(_ele_br) - float(globals().get('origin_elev_asl', 0.0))
                            except (ValueError, TypeError):
                                pass
                    if slab_base is None:
                        # last resort: layer tag × min clearance — skip if no layer info
                        try:
                            _ly = max(1, abs(int(float(str(row.get('layer', '') or '0').strip() or '0'))))
                            if _ly < 1: continue  # no data → skip, don't guess
                            slab_base = base_z + _ly * 4.5
                        except Exception:
                            continue
                    res = _extrude_building(coords_utm, slab_base, _DECK_THICKNESS,
                                           roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _br_f_s = [[fi + _br_off for fi in face] for face in _ff]
                        _br_v.append(_vv); _br_f.extend(_br_f_s); _br_off += len(_vv)
                    _br_count += 1
            except Exception: pass
        if _br_v:
            _brv = np.concatenate(_br_v).astype(np.float32)
            _brf = np.array(_br_f, np.int32)
            _write_ply(_brv, _brf, os.path.join(MESH_DIR, 'bld_itu_concrete_bridges.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/bld_itu_concrete_bridges.ply', 'buildings'))
            print(f'  Bridges: {_br_count} → bld_itu_concrete_bridges.ply ({len(_brv):,} verts, {len(_brf):,} faces)')
        else:
            print('  Bridges: none found (man_made=bridge) — check bbox or OSM coverage')
    except Exception as _e:
        print(f'  Bridges download failed: {_e}')
else:
    print('Bridges: skipped (INCLUDE_BRIDGES=False)')  # set INCLUDE_BRIDGES=True to enable

# ── Railway embankments → itu_concrete PLY ───────────────────────────────────
if globals().get('INCLUDE_EMBANKMENTS', True):   # enabled: railway embankment diffraction edges
    print('\nDownloading OSM railway embankments ...')
    try:
        _emb_tags = {'railway': ['rail','light_rail','tram','subway'],
                     'man_made': ['embankment']}
        if _ox_version >= (2, 0):
            gdf_emb = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_emb_tags)
        else:
            gdf_emb = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_emb_tags)
        if 'embankment' in gdf_emb.columns:
            gdf_emb = gdf_emb[gdf_emb['embankment'].fillna('').str.lower() == 'yes']
        elif 'man_made' in gdf_emb.columns:
            gdf_emb = gdf_emb[gdf_emb['man_made'].fillna('').str.lower() == 'embankment']
        else:
            gdf_emb = gdf_emb.iloc[0:0]  # empty — no embankment tags found
        _emb_lines = gdf_emb[gdf_emb.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_emb_lines)} railway/embankment lines')
        ev_all, ef_all, e_off = [], [], 0
        _EMB_H_DEF = 4.0   # fallback height when nDSM unavailable
        _EMB_H_MIN = 1.5   # ignore nDSM below this
        _EMB_H_MAX = 8.0   # cap
        _EMB_W = 8.0       # half-width (m)
        for _, row in _emb_lines.iterrows():
            try:
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    coords = list(line.coords)
                    _lcoords = []
                    for x, y in coords:
                        ex, ny = to_utm.transform(x, y)
                        _lcoords.append((ex, ny))
                    poly = sg.LineString(_lcoords).buffer(_EMB_W, cap_style=2)
                    _clon, _clat = to_wgs84.transform(poly.centroid.x, poly.centroid.y)
                    base_z = local_z(_clon, _clat)
                    _nd_samples = []
                    if _ndsm_arr is not None:
                        for _sx, _sy in _lcoords[::max(1,len(_lcoords)//5)]:
                            try:
                                _slon, _slat = to_wgs84.transform(_sx, _sy)
                                _nd = _ndsm_height(_slon, _slat)
                                if _nd is not None and _nd >= _EMB_H_MIN:
                                    _nd_samples.append(_nd)
                            except Exception: pass
                    _EMB_H = min(max(_nd_samples) if _nd_samples else _EMB_H_DEF, _EMB_H_MAX)
                    ext_coords_utm = [(x - center_utm[0], y - center_utm[1])
                                      for x, y in poly.exterior.coords[:-1]]
                    res = _extrude_building(ext_coords_utm, base_z, _EMB_H,
                                           roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _ef_s = [[fi + e_off for fi in face] for face in _ff]
                        ev_all.append(_vv); ef_all.extend(_ef_s); e_off += len(_vv)
            except Exception: pass
        if ev_all:
            ev = np.concatenate(ev_all).astype(np.float32)
            ef = np.array(ef_all, np.int32)
            _write_ply(ev, ef, os.path.join(MESH_DIR, 'bld_itu_medium_dry_ground_embankments.ply'))
            mat_plys.setdefault('itu_medium_dry_ground', []).append(
                (os.path.basename(MESH_DIR)+'/bld_itu_medium_dry_ground_embankments.ply', 'buildings'))
            print(f'  Embankments: {len(ev):,} verts, {len(ef):,} faces → bld_itu_medium_dry_ground_embankments.ply')
        else:
            print('  Embankments: no geometry found')
    except Exception as _e:
        print(f'  Embankments download failed: {_e}')
else:
    print('Embankments: skipped (INCLUDE_EMBANKMENTS=False)')

# ── Road embankments (highway + embankment=yes) → itu_concrete PLY ──────────
# M1, A610, A52 raised road sections — earth/concrete mounds blocking lateral scatter
if globals().get('INCLUDE_ROAD_EMBANKMENTS', True):
    print('\nDownloading OSM road embankments ...')
    try:
        _rdmb_tags = {'highway': ['motorway','trunk','primary','secondary']}
        if _ox_version >= (2, 0):
            gdf_rdmb = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_rdmb_tags)
        else:
            gdf_rdmb = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_rdmb_tags)
        if 'embankment' in gdf_rdmb.columns:
            _rdmb_mask = gdf_rdmb['embankment'].fillna('') == 'yes'
        else:
            _rdmb_mask = gdf_rdmb.index.map(lambda _: False)
        _rdmb_lines = gdf_rdmb[_rdmb_mask & gdf_rdmb.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_rdmb_lines)} road embankment ways')
        rdmb_v, rdmb_f, rdmb_off = [], [], 0
        _RDMB_H_DEF = 4.0   # fallback height when nDSM unavailable
        _RDMB_H_MIN = 1.5   # ignore nDSM below this
        _RDMB_H_MAX = 10.0  # cap
        _RDMB_W = 12.0      # half-width buffer — motorway wider than railway
        for _, row in _rdmb_lines.iterrows():
            try:
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    _lcoords = [(*(to_utm.transform(x, y)),) for x, y in list(line.coords)]
                    poly = sg.LineString(_lcoords).buffer(_RDMB_W, cap_style=2)
                    _rclon, _rclat = to_wgs84.transform(poly.centroid.x, poly.centroid.y)
                    base_z = local_z(_rclon, _rclat)
                    _rnd_s = []
                    if _ndsm_arr is not None:
                        for _rsx, _rsy in _lcoords[::max(1, len(_lcoords)//5)]:
                            try:
                                _rslon, _rslat = to_wgs84.transform(_rsx, _rsy)
                                _rnd = _ndsm_height(_rslon, _rslat)
                                if _rnd is not None and _rnd >= _RDMB_H_MIN:
                                    _rnd_s.append(_rnd)
                            except Exception: pass
                    _RDMB_H = min(max(_rnd_s) if _rnd_s else _RDMB_H_DEF, _RDMB_H_MAX)
                    ext_coords = [(x - center_utm[0], y - center_utm[1])
                                  for x, y in poly.exterior.coords[:-1]]
                    res = _extrude_building(ext_coords, base_z, _RDMB_H, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    for _vv, _ff in zip(res[0::2], res[1::2]):
                        if _vv is None or len(_vv) == 0: continue
                        rdmb_f.extend([[fi + rdmb_off for fi in face] for face in _ff])
                        rdmb_v.append(_vv); rdmb_off += len(_vv)
            except Exception: pass
        if rdmb_v:
            _rv = np.concatenate(rdmb_v).astype(np.float32)
            _rf = np.array(rdmb_f, np.int32)
            _write_ply(_rv, _rf, os.path.join(MESH_DIR, 'infra_itu_medium_dry_ground_road_embankments.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_medium_dry_ground_road_embankments.ply', 'infrastructure'))
            print(f'  Road embankments: {len(_rv):,} verts, {len(_rf):,} faces → infra_itu_medium_dry_ground_road_embankments.ply')
        else:
            print('  Road embankments: none found (highway + embankment=yes) — check OSM data for bbox')
    except Exception as _e:
        print(f'  Road embankments download failed: {_e}')
else:
    print('Road embankments: skipped (INCLUDE_ROAD_EMBANKMENTS=False)')

# ── EA Flood Defence Structures → itu_concrete / itu_metal PLY ──────────────
# Thames embankments, flood walls, levees — real physical blockers along river
# Data from CELL 2d-flood (flood_defences.geojson), EA ArcGIS REST, no auth
_flood_json = globals().get('FLOOD_DEF_JSON',
              os.path.join(LIDAR_DIR, 'flood_defences.geojson'))
if _flood_json and os.path.exists(str(_flood_json)):
    print('\nExtruding EA flood defence structures...')
    try:
        import geopandas as _gpd_fld
        from shapely.geometry import LineString as _LSFld, MultiLineString as _MLSFld
        _gdf_fld = _gpd_fld.read_file(str(_flood_json))
        print(f'  {len(_gdf_fld)} flood defence features')
        _fld_v_con, _fld_f_con, _fld_off_con = [], [], 0
        _fld_v_met, _fld_f_met, _fld_off_met = [], [], 0
        _FLD_W     = 3.0   # half-width buffer (m) for line features
        _FLD_H_DEF = 3.5   # fallback height when nDSM unavailable (Thames avg)
        _FLD_H_MIN = 1.0
        _FLD_H_MAX = 12.0
        for _, _frow in _gdf_fld.iterrows():
            try:
                _fgeom = _frow.geometry
                if _fgeom is None or _fgeom.is_empty:
                    continue
                # Determine material from defence type attribute
                _dtype = str(_frow.get('DefenceType', '')
                          or _frow.get('defence_type', '')
                          or _frow.get('type', '') or '').lower()
                _fmat = 'metal' if any(k in _dtype for k in
                        ['metal','steel','gate','sluice','barrier','movable']) else 'concrete'
                # Convert geometry to UTM polygon
                _fgeom_type = _fgeom.geom_type
                if _fgeom_type in ('LineString', 'MultiLineString'):
                    _flines = (list(_fgeom.geoms) if _fgeom_type == 'MultiLineString'
                               else [_fgeom])
                    _fpolys = []
                    for _fl in _flines:
                        _futm = [to_utm.transform(x, y) for x, y in _fl.coords]
                        _fpolys.append(_LSFld(_futm).buffer(_FLD_W, cap_style=2))
                elif _fgeom_type in ('Polygon', 'MultiPolygon'):
                    _sub = (list(_fgeom.geoms) if _fgeom_type == 'MultiPolygon'
                            else [_fgeom])
                    _fpolys = []
                    for _fp in _sub:
                        _futm_coords = [to_utm.transform(x, y)
                                        for x, y in _fp.exterior.coords]
                        from shapely.geometry import Polygon as _PolyFld
                        _fpolys.append(_PolyFld(_futm_coords))
                else:
                    continue
                for _fpoly in _fpolys:
                    # Get height from nDSM at feature centroid
                    _fclon, _fclat = to_wgs84.transform(
                        _fpoly.centroid.x, _fpoly.centroid.y)
                    _base_z_f = local_z(_fclon, _fclat)
                    # Sample nDSM along feature for height
                    _fh_vals = []
                    if _ndsm_arr is not None:
                        _fext = list(_fpoly.exterior.coords)
                        _fsample = _fext[::max(1, len(_fext)//8)]
                        for _fx, _fy in _fsample:
                            try:
                                _flon2, _flat2 = to_wgs84.transform(_fx, _fy)
                                _fh = _ndsm_height(_flon2, _flat2)
                                if _fh is not None and _fh >= _FLD_H_MIN:
                                    _fh_vals.append(_fh)
                            except Exception:
                                pass
                    _FH = float(np.clip(
                        np.median(_fh_vals) if len(_fh_vals) >= 2
                        else (_fh_vals[0] if _fh_vals else _FLD_H_DEF),
                        _FLD_H_MIN, _FLD_H_MAX))
                    _fext_coords = [(x - center_utm[0], y - center_utm[1])
                                    for x, y in _fpoly.exterior.coords[:-1]]
                    _fres = _extrude_building(_fext_coords, _base_z_f,
                                              _FH, roof_h=0.0, roof_shape='flat')
                    if _fres is None:
                        continue
                    if _fmat == 'metal':
                        _tgt_v, _tgt_f, _tgt_off = _fld_v_met, _fld_f_met, _fld_off_met
                    else:
                        _tgt_v, _tgt_f, _tgt_off = _fld_v_con, _fld_f_con, _fld_off_con
                    for _fvv, _fff in zip(_fres[0::2], _fres[1::2]):
                        if _fvv is None or len(_fvv) == 0:
                            continue
                        _tgt_f.extend([[fi + _tgt_off for fi in face] for face in _fff])
                        _tgt_v.append(_fvv)
                        _tgt_off += len(_fvv)
                    if _fmat == 'metal':
                        _fld_off_met = _tgt_off
                    else:
                        _fld_off_con = _tgt_off
            except Exception:
                pass
        # Write concrete flood defences PLY
        if _fld_v_con:
            _fv_con = np.vstack(_fld_v_con)
            _ff_con = np.array(_fld_f_con)
            _ply_fld_con = os.path.join(MESH_DIR, 'infra_itu_concrete_flood_defences.ply')
            _write_ply(_ply_fld_con, _fv_con, _ff_con)
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_concrete_flood_defences.ply',
                 'infrastructure'))
            print(f'  Flood defences (concrete): {len(_fv_con):,} verts → infra_itu_concrete_flood_defences.ply')
        # Write metal flood defences PLY
        if _fld_v_met:
            _fv_met = np.vstack(_fld_v_met)
            _ff_met = np.array(_fld_f_met)
            _ply_fld_met = os.path.join(MESH_DIR, 'infra_itu_metal_flood_defences.ply')
            _write_ply(_ply_fld_met, _fv_met, _ff_met)
            mat_plys.setdefault('itu_metal', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_metal_flood_defences.ply',
                 'infrastructure'))
            print(f'  Flood defences (metal): {len(_fv_met):,} verts → infra_itu_metal_flood_defences.ply')
        if not _fld_v_con and not _fld_v_met:
            print('  Flood defences: no extrudable geometry found')
    except Exception as _e_fld:
        print(f'  Flood defences error: {_e_fld}')
else:
    print('Flood defences: no data (run CELL 2d-flood first)')


# ── Road cuttings (highway + cutting=yes) → itu_concrete wall PLY ───────────
# A610, A52 depressed sections — vertical retaining walls on both sides block
# scatter paths entering/exiting the cut (ITU-R P.2040-2 concrete walls)
if globals().get('INCLUDE_ROAD_CUTTINGS', True):
    print('\nDownloading OSM road cuttings ...')
    try:
        _cut_tags = {'highway': ['motorway','trunk','primary','secondary']}
        if _ox_version >= (2, 0):
            gdf_cut = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_cut_tags)
        else:
            gdf_cut = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_cut_tags)
        if 'cutting' in gdf_cut.columns:
            _cut_mask = gdf_cut['cutting'].fillna('').isin(['yes','left','right','both'])
        else:
            _cut_mask = gdf_cut.index.map(lambda _: False)
        _cut_lines = gdf_cut[_cut_mask & gdf_cut.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_cut_lines)} road cutting ways')
        cut_v, cut_f, cut_off = [], [], 0
        def _cut_dims(row, line_wgs84):
            """Derive cut half-width from road width and cut height from DTM cross-section."""
            cut_w = _road_width(row) / 2.0 + 2.0   # road half-width + 2m shoulder
            # DTM perpendicular profile: measure terrain rise at cut wall face
            _max_h = 0.0
            _coords = list(line_wgs84.coords)
            for _i in range(len(_coords) - 1):
                _lx0, _ly0 = _coords[_i]; _lx1, _ly1 = _coords[_i + 1]
                _ex0, _ny0 = to_utm.transform(_lx0, _ly0)
                _ex1, _ny1 = to_utm.transform(_lx1, _ly1)
                _ddx, _ddy = _ex1 - _ex0, _ny1 - _ny0
                _seg_l = max((_ddx**2 + _ddy**2) ** 0.5, 1e-6)
                _px, _py = -_ddy / _seg_l, _ddx / _seg_l   # perpendicular unit vector
                _mx, _my = (_ex0 + _ex1) / 2, (_ny0 + _ny1) / 2
                _mlon, _mlat = to_wgs84.transform(_mx, _my)
                _road_z = local_z(_mlon, _mlat)
                for _side in (1, -1):
                    _ex_e = _mx + _side * (cut_w + 1.0) * _px
                    _ny_e = _my + _side * (cut_w + 1.0) * _py
                    _elon, _elat = to_wgs84.transform(_ex_e, _ny_e)
                    _edge_z = local_z(_elon, _elat)
                    _h = _edge_z - _road_z
                    if _h > _max_h:
                        _max_h = _h
            # Clamp to [1.0, 20.0]; fall back to 3m only if DTM entirely flat
            cut_h = float(np.clip(_max_h, 1.0, 20.0)) if _max_h > 0.3 else 3.0
            return cut_w, cut_h
        for _, row in _cut_lines.iterrows():
            try:
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    _lcoords = [(*(to_utm.transform(x, y)),) for x, y in list(line.coords)]
                    _cut_w, _cut_h = _cut_dims(row, line)
                    ring_poly = sg.LineString(_lcoords).buffer(_cut_w, cap_style=2)
                    ring_pts = [(x - center_utm[0], y - center_utm[1])
                                for x, y in ring_poly.exterior.coords[:-1]]
                    base_z = local_z(*to_wgs84.transform(ring_poly.centroid.x, ring_poly.centroid.y))
                    n = len(ring_pts)
                    for k in range(n):
                        x0, y0 = ring_pts[k]
                        x1, y1 = ring_pts[(k+1) % n]
                        verts = np.array([[x0, y0, base_z],
                                          [x1, y1, base_z],
                                          [x1, y1, base_z + _cut_h],
                                          [x0, y0, base_z + _cut_h]], dtype=np.float32)
                        cut_f.extend([[cut_off, cut_off+1, cut_off+2],
                                      [cut_off, cut_off+2, cut_off+3]])
                        cut_v.append(verts); cut_off += 4
            except Exception: pass
        if cut_v:
            _cv = np.concatenate(cut_v).astype(np.float32)
            _cf = np.array(cut_f, np.int32)
            _write_ply(_cv, _cf, os.path.join(MESH_DIR, 'infra_itu_medium_dry_ground_road_cuttings.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_medium_dry_ground_road_cuttings.ply', 'infrastructure'))
            print(f'  Road cuttings: {len(_cv):,} verts, {len(_cf):,} faces → infra_itu_medium_dry_ground_road_cuttings.ply')
        else:
            print('  Road cuttings: none found (highway + cutting=yes) — check OSM data for bbox')
    except Exception as _e:
        print(f'  Road cuttings download failed: {_e}')
else:
    print('Road cuttings: skipped (INCLUDE_ROAD_CUTTINGS=False)')

# ── Highway bridge decks (highway + bridge=yes) → itu_concrete PLY ──────────
# Motorway flyovers, trunk road bridges — deck slab as flat concrete panel
# Width derived from highway type; clearance 5 m above terrain
if globals().get('INCLUDE_HWY_BRIDGES', True):
    print('\nDownloading OSM highway bridges ...')
    try:
        _hbr_tags = {'highway': ['motorway','trunk','primary','secondary','tertiary']}
        if _ox_version >= (2, 0):
            gdf_hbr = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_hbr_tags)
        else:
            gdf_hbr = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_hbr_tags)
        if 'bridge' in gdf_hbr.columns:
            _hbr_mask = gdf_hbr['bridge'].fillna('').isin(['yes','viaduct','movable'])
        else:
            _hbr_mask = gdf_hbr.index.map(lambda _: False)
        _hbr_lines = gdf_hbr[_hbr_mask & gdf_hbr.geometry.geom_type.isin(['LineString','MultiLineString'])]
        print(f'  {len(_hbr_lines)} highway bridge ways')
        hbr_v, hbr_f, hbr_off = [], [], 0
        _DECK_THICKNESS = 1.5   # structural slab thickness (m)
        _hbr_count = 0
        for _, row in _hbr_lines.iterrows():
            try:
                half_w = _road_width(row) / 2.0   # uses width= tag → lanes×lane_w → type default
                geom = row.geometry
                lines = list(geom.geoms) if 'Multi' in geom.geom_type else [geom]
                for line in lines:
                    _lcoords = [(*(to_utm.transform(x, y)),) for x, y in list(line.coords)]
                    deck = sg.LineString(_lcoords).buffer(half_w, cap_style=2)
                    deck_coords = [(x - center_utm[0], y - center_utm[1])
                                   for x, y in deck.exterior.coords[:-1]]
                    _hbr_lon, _hbr_lat = to_wgs84.transform(deck.centroid.x, deck.centroid.y)
                    base_z = local_z(_hbr_lon, _hbr_lat)
                    # Deck elevation: nDSM multi-point p75 along line > OSM ele= > layer last resort
                    _hbr_slab = None
                    if _ndsm_arr is not None:
                        try:
                            _n_smp = max(5, int(line.length / 0.00005))
                            _hbr_hs = []
                            for _st in [i / _n_smp for i in range(_n_smp + 1)]:
                                _sp = line.interpolate(_st, normalized=True)
                                _sh = _ndsm_height(_sp.x, _sp.y)
                                if _sh is not None and _sh > 1.5:
                                    _hbr_hs.append(_sh)
                            if _hbr_hs:
                                _hbr_slab = base_z + float(np.percentile(_hbr_hs, 75))
                        except Exception:
                            pass
                    if _hbr_slab is None:
                        _ele_hbr = str(row.get('ele', '') or '').strip().replace('m', '')
                        if _ele_hbr:
                            try:
                                _hbr_slab = float(_ele_hbr) - float(globals().get('origin_elev_asl', 0.0))
                            except (ValueError, TypeError):
                                pass
                    if _hbr_slab is None:
                        try:
                            _hly = max(1, abs(int(float(str(row.get('layer', '') or '0').strip() or '0'))))
                            if _hly < 1: continue
                            _hbr_slab = base_z + _hly * 4.5
                        except Exception:
                            continue
                    res = _extrude_building(deck_coords, _hbr_slab,
                                           _DECK_THICKNESS, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    for _vv, _ff in zip(res[0::2], res[1::2]):
                        if _vv is None or len(_vv) == 0: continue
                        hbr_f.extend([[fi + hbr_off for fi in face] for face in _ff])
                        hbr_v.append(_vv); hbr_off += len(_vv)
                    _hbr_count += 1
            except Exception: pass
        if hbr_v:
            _hv = np.concatenate(hbr_v).astype(np.float32)
            _hf = np.array(hbr_f, np.int32)
            _write_ply(_hv, _hf, os.path.join(MESH_DIR, 'infra_itu_concrete_hwy_bridges.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                (os.path.basename(MESH_DIR)+'/infra_itu_concrete_hwy_bridges.ply', 'infrastructure'))
            print(f'  Hwy bridges: {_hbr_count} decks → infra_itu_concrete_hwy_bridges.ply ({len(_hv):,} verts, {len(_hf):,} faces)')
        else:
            print('  Highway bridges: none found (highway + bridge=yes) — check OSM data for bbox')
    except Exception as _e:
        print(f'  Highway bridges download failed: {_e}')
else:
    print('Highway bridges: skipped (INCLUDE_HWY_BRIDGES=False)')

# ── Railway tracks → itu_metal PLY → itu_metal PLY ───────────────────────────────────────────
# OSM railway=rail/light_rail/tram lines → flat ribbon mesh at terrain height
# Steel rails are strong specular reflectors at 915 MHz (εr≈1, σ=1e7 S/m)
# Effect: 2-4 dB signal enhancement along tram/rail corridors (Lienard 1997)
if globals().get('INCLUDE_RAILWAYS', True):
    print('\nDownloading OSM railway tracks ...')
    try:
        _rail_tags = {'railway': ['rail', 'light_rail', 'tram', 'subway', 'narrow_gauge']}
        if _ox_version >= (2, 0):
            gdf_rail = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_rail_tags)
        elif _ox_version >= (1, 3):
            gdf_rail = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_rail_tags)
        else:
            gdf_rail = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_rail_tags)
        _RAIL_W = 1.5    # track half-width (m) — standard gauge 1435 mm
        rv_all, rf_all, rv_off = [], [], 0
        for _, rrow in gdf_rail.iterrows():
            geom = rrow.geometry
            if geom is None or geom.is_empty: continue
            if globals().get('EXCLUDE_TUNNELS', True) and \
               str(rrow.get('tunnel', '')).lower() in ('yes', 'building_passage', 'subway'):
                continue
            lines_geom = []
            if geom.geom_type == 'LineString':
                lines_geom = [geom]
            elif geom.geom_type == 'MultiLineString':
                lines_geom = list(geom.geoms)
            for lg in lines_geom:
                coords = list(lg.coords)
                if len(coords) < 2: continue
                pts_utm = []
                for lon, lat in coords:
                    rx_, ry_ = to_utm.transform(lon, lat)
                    pts_utm.append((rx_ - center_utm[0], ry_ - center_utm[1]))
                # Build ribbon along track — one quad per segment
                for k in range(len(pts_utm) - 1):
                    x0, y0 = pts_utm[k]
                    x1, y1 = pts_utm[k + 1]
                    dx, dy = x1 - x0, y1 - y0
                    seg_len = (dx**2 + dy**2)**0.5
                    if seg_len < 0.1: continue
                    nx, ny = -dy / seg_len * _RAIL_W, dx / seg_len * _RAIL_W
                    try:
                        z0 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x0 + center_utm[0], y0 + center_utm[1])))
                        z1 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x1 + center_utm[0], y1 + center_utm[1])))
                    except Exception:
                        z0 = z1 = 0.0
                    quad = np.array([
                        [x0 - nx, y0 - ny, z0],
                        [x0 + nx, y0 + ny, z0],
                        [x1 + nx, y1 + ny, z1],
                        [x1 - nx, y1 - ny, z1],
                    ], dtype=np.float32)
                    rv_all.append(quad)
                    rf_all.append([rv_off, rv_off + 1, rv_off + 2])
                    rf_all.append([rv_off, rv_off + 2, rv_off + 3])
                    rv_off += 4
        if rv_all:
            rv = np.concatenate(rv_all).astype(np.float32)
            rf = np.array(rf_all, np.int32)
            _write_ply(rv, rf, os.path.join(MESH_DIR, 'rail_itu_metal.ply'))
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/' + 'rail_itu_metal.ply', 'railways'))
            print(f'  Railways: {len(rv):,} verts, {len(rf):,} faces → rail_itu_metal.ply')
        else:
            print('  Railways: no geometry found')
    except Exception as _e:
        print(f'  Railways download failed: {_e}')
else:
    print('Railways: skipped (INCLUDE_RAILWAYS=False)')

# ── Barriers (walls / fences / noise barriers) → PLY ─────────────────────────
# OSM barrier=wall/noise_barrier → itu_concrete  (brick walls, noise screens)
#     barrier=fence/railing      → itu_metal     (metal fence panels)
# Effect: 1-8 dB shadow behind wall depending on height and frequency
# Noise barriers along major roads are 3-5 m concrete → 5-8 dB NLOS attenuation
if globals().get('INCLUDE_BARRIERS', True):
    print('\nDownloading OSM barriers ...')
    try:
        _bar_tags = {'barrier': ['wall', 'fence', 'noise_barrier', 'retaining_wall',
                                  'guard_rail', 'railing', 'jersey_barrier']}
        if _ox_version >= (2, 0):
            gdf_bar = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_bar_tags)
        elif _ox_version >= (1, 3):
            gdf_bar = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_bar_tags)
        else:
            gdf_bar = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_bar_tags)
        # Default heights per barrier type (m)
        _BAR_HEIGHTS = {
            'wall': 2.0, 'noise_barrier': 4.0, 'retaining_wall': 2.5,
            'fence': 1.8, 'railing': 1.2, 'guard_rail': 0.8, 'jersey_barrier': 1.0,
        }
        # Material per barrier type
        _BAR_MAT = {
            'wall': 'itu_concrete', 'noise_barrier': 'itu_concrete',
            'retaining_wall': 'itu_concrete', 'jersey_barrier': 'itu_concrete',
            'fence': 'itu_metal', 'railing': 'itu_metal', 'guard_rail': 'itu_metal',
        }
        _bar_min_len = float(globals().get('BARRIER_MIN_LEN_M', 10.0))
        bar_geom = {}   # mat → (verts_list, faces_list, offset)
        _bar_count = 0
        for _, brow in gdf_bar.iterrows():
            geom = brow.geometry
            if geom is None or geom.is_empty: continue
            b_type = str(brow.get('barrier', 'wall') or 'wall').lower()
            b_mat  = _BAR_MAT.get(b_type, 'itu_concrete')
            b_h    = _BAR_HEIGHTS.get(b_type, 2.0)
            # Override height from OSM height= tag if present
            _ht = str(brow.get('height', '') or '')
            if _ht and _ht not in ('nan', 'None', ''):
                try: b_h = float(_ht.replace('m','').strip())
                except ValueError: pass
            lines_geom = []
            if geom.geom_type == 'LineString':     lines_geom = [geom]
            elif geom.geom_type == 'MultiLineString': lines_geom = list(geom.geoms)
            elif geom.geom_type in ('Polygon','MultiPolygon'):
                # Barrier polygons → use exterior ring as wall line
                polys = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
                for p in polys:
                    from shapely.geometry import LineString as _LS
                    lines_geom.append(_LS(p.exterior.coords))
            for lg in lines_geom:
                coords = list(lg.coords)
                if len(coords) < 2: continue
                pts_utm = []
                for lon, lat in coords:
                    bx_, by_ = to_utm.transform(lon, lat)
                    pts_utm.append((bx_ - center_utm[0], by_ - center_utm[1]))
                # Check total length — skip trivially short segments
                _seg_len_total = sum(
                    ((pts_utm[k+1][0]-pts_utm[k][0])**2 +
                     (pts_utm[k+1][1]-pts_utm[k][1])**2)**0.5
                    for k in range(len(pts_utm)-1))
                if _seg_len_total < _bar_min_len: continue
                if b_mat not in bar_geom:
                    bar_geom[b_mat] = ([], [], 0)
                bv_l, bf_l, b_off = bar_geom[b_mat]
                for k in range(len(pts_utm) - 1):
                    x0, y0 = pts_utm[k]
                    x1, y1 = pts_utm[k + 1]
                    try:
                        z0 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x0+center_utm[0], y0+center_utm[1])))
                        z1 = 0.0 if FLAT_TERRAIN else float(local_z(
                            *to_wgs84.transform(x1+center_utm[0], y1+center_utm[1])))
                    except Exception:
                        z0 = z1 = 0.0
                    # Vertical quad: bottom-left, bottom-right, top-right, top-left
                    quad = np.array([
                        [x0, y0, z0],
                        [x1, y1, z1],
                        [x1, y1, z1 + b_h],
                        [x0, y0, z0 + b_h],
                    ], dtype=np.float32)
                    bv_l.append(quad)
                    bf_l.append([b_off, b_off + 1, b_off + 2])
                    bf_l.append([b_off, b_off + 2, b_off + 3])
                    b_off += 4
                bar_geom[b_mat] = (bv_l, bf_l, b_off)
                _bar_count += 1
        if bar_geom:
            for b_mat, (bv_l, bf_l, _) in bar_geom.items():
                if not bv_l: continue
                bv = np.concatenate(bv_l).astype(np.float32)
                bf = np.array(bf_l, np.int32)
                _mat_tag = b_mat.replace('itu_', '')
                _bar_ply = f'barriers_{b_mat}.ply'
                _write_ply(bv, bf, os.path.join(MESH_DIR, _bar_ply))
                mat_plys.setdefault(b_mat, []).append(
                    ('meshes/' + _bar_ply, 'barriers'))
                print(f'  Barriers ({_mat_tag}): {len(bv):,} verts, {len(bf):,} faces → {_bar_ply}')
            print(f'  Barriers total: {_bar_count} segments processed')
        else:
            print('  Barriers: no geometry found')
    except Exception as _e:
        print(f'  Barriers download failed: {_e}')
else:
    print('Barriers: skipped (INCLUDE_BARRIERS=False)')


# ── Power pylons / transmission towers → itu_metal PLY ───────────────────────
# Lattice steel towers are dominant metallic diffractors at 915 MHz [DeE04].
# Approximated as solid box of equivalent projected cross-section.
if globals().get('INCLUDE_PYLONS', True):   # enabled electricity pylons — significant scatterers in open areas:
    print('\nDownloading OSM power pylons ...')
    try:
        _pyl_tags = {'power': ['tower', 'pole']}
        if _ox_version >= (2, 0):
            gdf_pyl = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_pyl_tags)
        elif _ox_version >= (1, 3):
            gdf_pyl = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_pyl_tags)
        else:
            gdf_pyl = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_pyl_tags)
        _pyl_v, _pyl_f, _pyl_off = [], [], 0
        _pyl_count = 0
        _PYL_H = {'tower': 30.0, 'pole': 8.0}
        _PYL_W = {'tower':  3.0, 'pole': 0.4}
        for _, prow in gdf_pyl.iterrows():
            geom = prow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            ptype = str(prow.get('power', 'tower') or 'tower').lower()
            _ht = _PYL_H.get(ptype, 25.0)
            _ht_tag = str(prow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = _PYL_W.get(ptype, 2.0) / 2.0
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _pyl_f_s = [[fi + _pyl_off for fi in face] for face in _ff]
                _pyl_v.append(_vv); _pyl_f.extend(_pyl_f_s); _pyl_off += len(_vv)
            _pyl_count += 1
        if _pyl_v:
            _pv = np.concatenate(_pyl_v).astype(np.float32)
            _pf = np.array(_pyl_f, np.int32)
            _ply_name = 'infra_itu_metal_pylons.ply'
            _write_ply(_pv, _pf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Pylons: {_pyl_count} → {_ply_name} ({len(_pv):,} verts, {len(_pf):,} faces)')
        else:
            print('  Pylons: none found in bbox')
    except Exception as _e:
        print(f'  Pylons: download failed — {_e}')
else:
    print('Pylons: skipped (INCLUDE_PYLONS=False)')

# ── Telecom masts / broadcast towers → itu_metal PLY ─────────────────────────
# Tall slender metal masts are strong vertical diffraction edges at 915 MHz.
if globals().get('INCLUDE_MASTS', True):   # enabled telecom/radio masts — point scatterers:
    print('\nDownloading OSM telecom masts ...')
    try:
        _mst_tags = {'man_made': ['mast', 'communications_tower', 'tower']}
        if _ox_version >= (2, 0):
            gdf_mst = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_mst_tags)
        elif _ox_version >= (1, 3):
            gdf_mst = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_mst_tags)
        else:
            gdf_mst = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_mst_tags)
        _mst_v, _mst_f, _mst_off = [], [], 0
        _mst_count = 0
        for _, mrow in gdf_mst.iterrows():
            geom = mrow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            _ht = 30.0
            _ht_tag = str(mrow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = 1.0  # 2m width column
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _mst_f_s = [[fi + _mst_off for fi in face] for face in _ff]
                _mst_v.append(_vv); _mst_f.extend(_mst_f_s); _mst_off += len(_vv)
            _mst_count += 1
        if _mst_v:
            _mv = np.concatenate(_mst_v).astype(np.float32)
            _mf = np.array(_mst_f, np.int32)
            _ply_name = 'infra_itu_metal_masts.ply'
            _write_ply(_mv, _mf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Masts: {_mst_count} → {_ply_name} ({len(_mv):,} verts, {len(_mf):,} faces)')
        else:
            print('  Masts: none found in bbox')
    except Exception as _e:
        print(f'  Masts: download failed — {_e}')
else:
    print('Masts: skipped (INCLUDE_MASTS=False)')

# ── Industrial chimneys → itu_concrete PLY ───────────────────────────────────
# Circular concrete stacks are significant vertical diffractors.
if globals().get('INCLUDE_CHIMNEYS', False):
    print('\nDownloading OSM chimneys ...')
    try:
        _chi_tags = {'man_made': 'chimney'}
        if _ox_version >= (2, 0):
            gdf_chi = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_chi_tags)
        elif _ox_version >= (1, 3):
            gdf_chi = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_chi_tags)
        else:
            gdf_chi = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_chi_tags)
        _chi_v, _chi_f, _chi_off = [], [], 0
        _chi_count = 0
        for _, crow in gdf_chi.iterrows():
            geom = crow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            _ht = 25.0
            _ht_tag = str(crow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = 1.5  # 3m diameter approximation
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _chi_f_s = [[fi + _chi_off for fi in face] for face in _ff]
                _chi_v.append(_vv); _chi_f.extend(_chi_f_s); _chi_off += len(_vv)
            _chi_count += 1
        if _chi_v:
            _cv = np.concatenate(_chi_v).astype(np.float32)
            _cf = np.array(_chi_f, np.int32)
            _ply_name = 'infra_itu_concrete_chimneys.ply'
            _write_ply(_cv, _cf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_concrete', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Chimneys: {_chi_count} → {_ply_name} ({len(_cv):,} verts, {len(_cf):,} faces)')
        else:
            print('  Chimneys: none found in bbox')
    except Exception as _e:
        print(f'  Chimneys: download failed — {_e}')
else:
    print('Chimneys: skipped (INCLUDE_CHIMNEYS=False)')

# ── Water towers → itu_metal PLY ─────────────────────────────────────────────
if globals().get('INCLUDE_WATER_TOWERS', True):   # enabled water towers — elevated concrete cylinders:
    print('\nDownloading OSM water towers ...')
    try:
        _wt_tags = {'man_made': 'water_tower'}
        if _ox_version >= (2, 0):
            gdf_wt = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_wt_tags)
        elif _ox_version >= (1, 3):
            gdf_wt = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_wt_tags)
        else:
            gdf_wt = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_wt_tags)
        _wt_v, _wt_f, _wt_off = [], [], 0
        _wt_count = 0
        for _, wrow in gdf_wt.iterrows():
            geom = wrow.geometry
            if geom is None or geom.is_empty: continue
            cx, cy = (geom.x, geom.y) if geom.geom_type == 'Point' else (geom.centroid.x, geom.centroid.y)
            px, py = to_utm.transform(cx, cy)
            lx = px - center_utm[0]; ly = py - center_utm[1]
            base_z = local_z(cx, cy)
            _ht = 15.0
            _ht_tag = str(wrow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            hw = 2.5  # 5m diameter approximation
            sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
            res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
            if res is None: continue
            _wv, _wf, _rv, _rf = res
            for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                _wt_f_s = [[fi + _wt_off for fi in face] for face in _ff]
                _wt_v.append(_vv); _wt_f.extend(_wt_f_s); _wt_off += len(_vv)
            _wt_count += 1
        if _wt_v:
            _wtv = np.concatenate(_wt_v).astype(np.float32)
            _wtf = np.array(_wt_f, np.int32)
            _ply_name = 'infra_itu_metal_watertowers.ply'
            _write_ply(_wtv, _wtf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Water towers: {_wt_count} → {_ply_name} ({len(_wtv):,} verts, {len(_wtf):,} faces)')
        else:
            print('  Water towers: none found in bbox')
    except Exception as _e:
        print(f'  Water towers: download failed — {_e}')
else:
    print('Water towers: skipped (INCLUDE_WATER_TOWERS=False)')

# ── Storage tanks → itu_metal PLY ────────────────────────────────────────────
# Curved cylindrical metal surfaces produce strong backscatter at sub-GHz [DeE04 §IV].
if globals().get('INCLUDE_STORAGE_TANKS', True):   # enabled industrial tanks/silos — large metal reflectors:
    print('\nDownloading OSM storage tanks ...')
    try:
        _stk_tags = {'man_made': ['storage_tank', 'silo']}
        if _ox_version >= (2, 0):
            gdf_stk = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_stk_tags)
        elif _ox_version >= (1, 3):
            gdf_stk = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_stk_tags)
        else:
            gdf_stk = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_stk_tags)
        _stk_v, _stk_f, _stk_off = [], [], 0
        _stk_count = 0
        for _, srow in gdf_stk.iterrows():
            geom = srow.geometry
            if geom is None or geom.is_empty: continue
            _ht = 8.0
            _ht_tag = str(srow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            if geom.geom_type in ('Polygon', 'MultiPolygon'):
                polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
                for poly in polys:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    base_z = local_z(poly.centroid.x, poly.centroid.y)
                    res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _stk_f_s = [[fi + _stk_off for fi in face] for face in _ff]
                        _stk_v.append(_vv); _stk_f.extend(_stk_f_s); _stk_off += len(_vv)
                    _stk_count += 1
            elif geom.geom_type == 'Point':
                px, py = to_utm.transform(geom.x, geom.y)
                lx = px - center_utm[0]; ly = py - center_utm[1]
                base_z = local_z(geom.x, geom.y)
                hw = 5.0
                sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
                res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _stk_f_s = [[fi + _stk_off for fi in face] for face in _ff]
                    _stk_v.append(_vv); _stk_f.extend(_stk_f_s); _stk_off += len(_vv)
                _stk_count += 1
        if _stk_v:
            _sv = np.concatenate(_stk_v).astype(np.float32)
            _sf = np.array(_stk_f, np.int32)
            _ply_name = 'infra_itu_metal_tanks.ply'
            _write_ply(_sv, _sf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Storage tanks: {_stk_count} → {_ply_name} ({len(_sv):,} verts, {len(_sf):,} faces)')
        else:
            print('  Storage tanks: none found in bbox')
    except Exception as _e:
        print(f'  Storage tanks: download failed — {_e}')
else:
    print('Storage tanks: skipped (INCLUDE_STORAGE_TANKS=False)')

# ── Stadiums → itu_metal PLY ──────────────────────────────────────────────────
# Large metal-roofed sports venues are significant reflectors [Xia24].
# Only include leisure=stadium features NOT already captured by building export.
if globals().get('INCLUDE_STADIUMS', True):   # enabled: large structures blocking propagation
    print('\nDownloading OSM stadiums ...')
    try:
        _std_tags = {'leisure': 'stadium'}
        if _ox_version >= (2, 0):
            gdf_std = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_std_tags)
        elif _ox_version >= (1, 3):
            gdf_std = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_std_tags)
        else:
            gdf_std = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_std_tags)
        _std_v, _std_f, _std_off = [], [], 0
        _std_count = 0
        for _, srow in gdf_std.iterrows():
            geom = srow.geometry
            if geom is None or geom.is_empty: continue
            if str(srow.get('building', '') or '').lower() in ('yes','stadium','sports_hall'):
                continue  # already in building export
            if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
            _ht = 15.0  # typical stand roof height
            _ht_tag = str(srow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
            for poly in polys:
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                base_z = local_z(poly.centroid.x, poly.centroid.y)
                res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _std_f_s = [[fi + _std_off for fi in face] for face in _ff]
                    _std_v.append(_vv); _std_f.extend(_std_f_s); _std_off += len(_vv)
                _std_count += 1
        if _std_v:
            _stv = np.concatenate(_std_v).astype(np.float32)
            _stf = np.array(_std_f, np.int32)
            _ply_name = 'infra_itu_metal_stadiums.ply'
            _write_ply(_stv, _stf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Stadiums: {_std_count} → {_ply_name} ({len(_stv):,} verts, {len(_stf):,} faces)')
        else:
            print('  Stadiums: none found in bbox (all may already be in building export)')
    except Exception as _e:
        print(f'  Stadiums: download failed — {_e}')
else:
    print('Stadiums: skipped (INCLUDE_STADIUMS=False)')

# ── Power substations → itu_metal PLY ────────────────────────────────────────
# Fenced transformer enclosures with metallic equipment inside.
if globals().get('INCLUDE_SUBSTATIONS', True):   # enabled electricity substations — metal structures:
    print('\nDownloading OSM power substations ...')
    try:
        _sub_tags = {'power': 'substation'}
        if _ox_version >= (2, 0):
            gdf_sub = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_sub_tags)
        elif _ox_version >= (1, 3):
            gdf_sub = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_sub_tags)
        else:
            gdf_sub = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_sub_tags)
        _sub_v, _sub_f, _sub_off = [], [], 0
        _sub_count = 0
        for _, subrow in gdf_sub.iterrows():
            geom = subrow.geometry
            if geom is None or geom.is_empty: continue
            if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
            _ht = 4.0  # substation fence/equipment height
            polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
            for poly in polys:
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                base_z = local_z(poly.centroid.x, poly.centroid.y)
                res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _sub_f_s = [[fi + _sub_off for fi in face] for face in _ff]
                    _sub_v.append(_vv); _sub_f.extend(_sub_f_s); _sub_off += len(_vv)
                _sub_count += 1
        if _sub_v:
            _sbv = np.concatenate(_sub_v).astype(np.float32)
            _sbf = np.array(_sub_f, np.int32)
            _ply_name = 'infra_itu_metal_substations.ply'
            _write_ply(_sbv, _sbf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_metal', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Substations: {_sub_count} → {_ply_name} ({len(_sbv):,} verts, {len(_sbf):,} faces)')
        else:
            print('  Substations: none found in bbox')
    except Exception as _e:
        print(f'  Substations: download failed — {_e}')
else:
    print('Substations: skipped (INCLUDE_SUBSTATIONS=False)')


# ── Multi-storey car parks → itu_concrete PLY ────────────────────────────────
# Large open-floor concrete structures with different EM signature from solid buildings.
# Primary cause of −13 dB bias in 300–700m band along Nottingham city centre [GS22].
if globals().get('INCLUDE_CAR_PARKS', True):   # enabled multi-storey car parks:
    print('\nDownloading OSM multi-storey car parks ...')
    try:
        _cp_tags = {'building': ['parking', 'car_park', 'carpark']}
        if _ox_version >= (2, 0):
            gdf_cp = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_cp_tags)
        elif _ox_version >= (1, 3):
            gdf_cp = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_cp_tags)
        else:
            gdf_cp = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_cp_tags)
        _cp_v, _cp_f, _cp_off = [], [], 0
        _cp_count = 0
        for _, cprow in gdf_cp.iterrows():
            geom = cprow.geometry
            if geom is None or geom.is_empty: continue
            if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
            _ht = 12.0  # typical 4-level MSCP ~12m
            _ht_tag = str(cprow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            _lvls = str(cprow.get('building:levels', '') or '')
            if _lvls not in ('nan', 'None', ''):
                try: _ht = max(_ht, float(_lvls) * 3.0)
                except ValueError: pass
            polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
            for poly in polys:
                coords_utm = []
                for lon, lat in poly.exterior.coords[:-1]:
                    ex, ny = to_utm.transform(lon, lat)
                    coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                base_z = local_z(poly.centroid.x, poly.centroid.y)
                res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _cp_f_s = [[fi + _cp_off for fi in face] for face in _ff]
                    _cp_v.append(_vv); _cp_f.extend(_cp_f_s); _cp_off += len(_vv)
                _cp_count += 1
        if _cp_v:
            _cpv = np.concatenate(_cp_v).astype(np.float32)
            _cpf = np.array(_cp_f, np.int32)
            _ply_name = 'infra_itu_concrete_carparks.ply'
            _write_ply(_cpv, _cpf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_concrete', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Car parks: {_cp_count} → {_ply_name} ({len(_cpv):,} verts, {len(_cpf):,} faces)')
        else:
            print('  Car parks: none found in bbox')
    except Exception as _e:
        print(f'  Car parks: download failed — {_e}')
else:
    print('Car parks: skipped (INCLUDE_CAR_PARKS=False)')

# ── Cooling towers → itu_concrete PLY ────────────────────────────────────────
# Large cylindrical concrete structures — dominant long-range reflectors at sub-GHz.
# Ratcliffe-on-Soar power station is within 10 km south of Nottingham city centre.
if globals().get('INCLUDE_COOLING_TOWERS', True):   # enabled industrial cooling towers:
    print('\nDownloading OSM cooling towers ...')
    try:
        _ct_tags = {'man_made': 'cooling_tower'}
        if _ox_version >= (2, 0):
            gdf_ct = ox.features_from_bbox(bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_ct_tags)
        elif _ox_version >= (1, 3):
            gdf_ct = ox.features_from_bbox(bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_ct_tags)
        else:
            gdf_ct = ox.features_from_bbox(north=SCENE_NORTH, south=SCENE_SOUTH, east=SCENE_EAST, west=SCENE_WEST, tags=_ct_tags)
        _ct_v, _ct_f, _ct_off = [], [], 0
        _ct_count = 0
        for _, ctrow in gdf_ct.iterrows():
            geom = ctrow.geometry
            if geom is None or geom.is_empty: continue
            _ht = 60.0  # Ratcliffe towers ~114m; generic default 60m
            _ht_tag = str(ctrow.get('height', '') or '')
            if _ht_tag not in ('nan', 'None', ''):
                try: _ht = float(_ht_tag.replace('m', '').strip())
                except ValueError: pass
            if geom.geom_type in ('Polygon', 'MultiPolygon'):
                polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
                for poly in polys:
                    coords_utm = []
                    for lon, lat in poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    base_z = local_z(poly.centroid.x, poly.centroid.y)
                    res = _extrude_building(coords_utm, base_z, _ht, roof_h=0.0, roof_shape='flat')
                    if res is None: continue
                    _wv, _wf, _rv, _rf = res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                        _ct_f_s = [[fi + _ct_off for fi in face] for face in _ff]
                        _ct_v.append(_vv); _ct_f.extend(_ct_f_s); _ct_off += len(_vv)
                    _ct_count += 1
            elif geom.geom_type == 'Point':
                px, py = to_utm.transform(geom.x, geom.y)
                lx = px - center_utm[0]; ly = py - center_utm[1]
                base_z = local_z(geom.x, geom.y)
                hw = 10.0  # 20m diameter approx
                sq = [(lx-hw, ly-hw), (lx+hw, ly-hw), (lx+hw, ly+hw), (lx-hw, ly+hw)]
                res = _extrude_building(sq, base_z, _ht, roof_h=0.0, roof_shape='flat')
                if res is None: continue
                _wv, _wf, _rv, _rf = res
                for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                    if _vv is None or len(_vv) == 0 or _ff is None or len(_ff) == 0: continue
                    _ct_f_s = [[fi + _ct_off for fi in face] for face in _ff]
                    _ct_v.append(_vv); _ct_f.extend(_ct_f_s); _ct_off += len(_vv)
                _ct_count += 1
        if _ct_v:
            _ctv = np.concatenate(_ct_v).astype(np.float32)
            _ctf = np.array(_ct_f, np.int32)
            _ply_name = 'infra_itu_concrete_coolingtowers.ply'
            _write_ply(_ctv, _ctf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault('itu_concrete', []).append(('meshes/' + _ply_name, 'infrastructure'))
            print(f'  Cooling towers: {_ct_count} → {_ply_name} ({len(_ctv):,} verts, {len(_ctf):,} faces)')
        else:
            print('  Cooling towers: none found in bbox')
    except Exception as _e:
        print(f'  Cooling towers: download failed — {_e}')
else:
    print('Cooling towers: skipped (INCLUDE_COOLING_TOWERS=False)')

# ── Fuel station canopies → itu_metal PLY ────────────────────────────────────
# amenity=fuel → flat metal canopy roof at +5m clearance, 0.3m thick slab
# ITU-R P.2040-2: itu_metal  er=1.0  σ=1e7  s=0.05  xpd=0.01  wt=0.005m
# Effect: strong specular reflector (metal roof) — visible at both 915 MHz & 3.6 GHz
if globals().get('INCLUDE_FUEL_CANOPIES', True):
    print('\nDownloading OSM fuel station canopies ...')
    try:
        _fuel_tags = {'amenity': 'fuel'}
        if _ox_version >= (2, 0):
            gdf_fuel = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_fuel_tags)
        elif _ox_version >= (1, 3):
            gdf_fuel = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_fuel_tags)
        else:
            gdf_fuel = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_fuel_tags)
        _FUEL_CLEARANCE  = 5.0   # m — typical forecourt canopy clearance
        _FUEL_THICKNESS  = 0.3   # m — metal roof slab thickness
        _FUEL_BUF        = 12.0  # m — point feature buffer radius
        _fuel_v, _fuel_f, _fuel_off, _fuel_cnt = [], [], 0, 0
        for _, row in gdf_fuel.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                if geom.geom_type == 'Point':
                    _px, _py = to_utm.transform(geom.x, geom.y)
                    import shapely.geometry as _sg2
                    geom = _sg2.Point(_px, _py).buffer(_FUEL_BUF)
                    _poly_list = [geom]
                elif geom.geom_type == 'Polygon':
                    _poly_list = [geom]
                    geom = None
                    for lon, lat in _poly_list[0].exterior.coords[:-1]:
                        pass
                    _pts = []
                    for lon, lat in _poly_list[0].exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _pts.append((ex, ny))
                    import shapely.geometry as _sg2
                    _poly_list = [_sg2.Polygon(_pts)]
                elif geom.geom_type == 'MultiPolygon':
                    _all_pts = []
                    for _sp in geom.geoms:
                        _pts = []
                        for lon, lat in _sp.exterior.coords[:-1]:
                            ex, ny = to_utm.transform(lon, lat)
                            _pts.append((ex, ny))
                        import shapely.geometry as _sg2
                        _all_pts.append(_sg2.Polygon(_pts))
                    _poly_list = _all_pts
                else:
                    continue
                for _poly in _poly_list:
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _coords_local = [(_p[0] - center_utm[0], _p[1] - center_utm[1])
                                     for _p in _poly.exterior.coords[:-1]]
                    if len(_coords_local) < 3: continue
                    _res = _extrude_building(_coords_local, _bz + _FUEL_CLEARANCE,
                                             _FUEL_THICKNESS, roof_h=0.0, roof_shape='flat')
                    if _res is None: continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0: continue
                        _fuel_f.extend([[fi + _fuel_off for fi in face] for face in _ff])
                        _fuel_v.append(_vv); _fuel_off += len(_vv)
                    _fuel_cnt += 1
            except Exception: pass
        if _fuel_v:
            _fv = np.concatenate(_fuel_v).astype(np.float32)
            _ff2 = np.array(_fuel_f, np.int32)
            _write_ply(_fv, _ff2, os.path.join(MESH_DIR, 'infra_itu_metal_fuel_canopies.ply'))
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_fuel_canopies.ply', 'fuel_canopies'))
            print(f'  Fuel canopies: {_fuel_cnt} → infra_itu_metal_fuel_canopies.ply '
                  f'({len(_fv):,} verts, {len(_ff2):,} faces)')
        else:
            print('  Fuel canopies: none found or no polygon geometry')
    except Exception as _e:
        print(f'  Fuel canopies download failed: {_e}')
else:
    print('Fuel canopies: skipped (INCLUDE_FUEL_CANOPIES=False)')

# ── Bus stations → itu_metal PLY ─────────────────────────────────────────────
# amenity=bus_station → extruded metal structure 6m high
# ITU-R P.2040-2: itu_metal  er=1.0  σ=1e7  s=0.05  xpd=0.01  wt=0.005m
# Effect: large covered roof in city centres — specular + diffraction edges
if globals().get('INCLUDE_BUS_STATIONS', True):
    print('\nDownloading OSM bus stations ...')
    try:
        _bus_tags = {'amenity': 'bus_station'}
        if _ox_version >= (2, 0):
            gdf_bus = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_bus_tags)
        elif _ox_version >= (1, 3):
            gdf_bus = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_bus_tags)
        else:
            gdf_bus = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_bus_tags)
        _BUS_H   = 6.0   # m — typical bus station roof height
        _BUS_BUF = 15.0  # m — point feature buffer radius
        _bus_v, _bus_f, _bus_off, _bus_cnt = [], [], 0, 0
        for _, row in gdf_bus.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                _ht = _BUS_H
                _ht_tag = str(row.get('height', '') or '')
                if _ht_tag not in ('nan', 'None', ''):
                    try: _ht = float(_ht_tag.replace('m', '').strip())
                    except ValueError: pass
                if geom.geom_type == 'Point':
                    _px, _py = to_utm.transform(geom.x, geom.y)
                    import shapely.geometry as _sg2
                    geom = _sg2.Point(_px, _py).buffer(_BUS_BUF)
                    _poly_list = [geom]
                elif geom.geom_type in ('Polygon', 'MultiPolygon'):
                    _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                    _utm_polys = []
                    for _sp in _poly_list:
                        _pts = []
                        for lon, lat in _sp.exterior.coords[:-1]:
                            ex, ny = to_utm.transform(lon, lat)
                            _pts.append((ex, ny))
                        import shapely.geometry as _sg2
                        _utm_polys.append(_sg2.Polygon(_pts))
                    _poly_list = _utm_polys
                else:
                    continue
                for _poly in _poly_list:
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _coords_local = [(_p[0] - center_utm[0], _p[1] - center_utm[1])
                                     for _p in _poly.exterior.coords[:-1]]
                    if len(_coords_local) < 3: continue
                    _res = _extrude_building(_coords_local, _bz, _ht,
                                             roof_h=0.0, roof_shape='flat')
                    if _res is None: continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0: continue
                        _bus_f.extend([[fi + _bus_off for fi in face] for face in _ff])
                        _bus_v.append(_vv); _bus_off += len(_vv)
                    _bus_cnt += 1
            except Exception: pass
        if _bus_v:
            _bv = np.concatenate(_bus_v).astype(np.float32)
            _bf = np.array(_bus_f, np.int32)
            _write_ply(_bv, _bf, os.path.join(MESH_DIR, 'infra_itu_metal_bus_stations.ply'))
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_bus_stations.ply', 'bus_stations'))
            print(f'  Bus stations: {_bus_cnt} → infra_itu_metal_bus_stations.ply '
                  f'({len(_bv):,} verts, {len(_bf):,} faces)')
        else:
            print('  Bus stations: none found or no polygon geometry')
    except Exception as _e:
        print(f'  Bus stations download failed: {_e}')
else:
    print('Bus stations: skipped (INCLUDE_BUS_STATIONS=False)')

# ── Surface car parks → itu_asphalt flat patch ───────────────────────────────
# amenity=parking (surface/open-air) → flat asphalt ground patch
# ITU-R P.2040-2: itu_asphalt  er=2.56  σ=0.005  s=0.30  xpd=0.15  wt=0.05m
# Effect: improves ground reflection accuracy — critical at 3.6 GHz (shorter Fresnel zone)
# Excludes multi-storey (parking=multi-storey) — already in INCLUDE_CAR_PARKS
if globals().get('INCLUDE_SURFACE_PARKS', True):
    print('\nDownloading OSM surface car parks ...')
    try:
        _spk_tags = {'amenity': 'parking'}
        if _ox_version >= (2, 0):
            gdf_spk = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_spk_tags)
        elif _ox_version >= (1, 3):
            gdf_spk = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_spk_tags)
        else:
            gdf_spk = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_spk_tags)
        _MULTI_STOREY = {'multi-storey', 'multi_storey', 'multistorey',
                          'parking_garage', 'underground', 'rooftop'}
        _spk_v, _spk_f, _spk_off, _spk_cnt = [], [], 0, 0
        for _, row in gdf_spk.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                _pk_type = str(row.get('parking', '') or '').lower()
                if _pk_type in _MULTI_STOREY: continue
                _bldg = str(row.get('building', '') or '').lower()
                if _bldg in ('parking', 'car_park', 'carpark', 'yes'): continue
                if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
                _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for _poly in _poly_list:
                    _coords_utm = []
                    for lon, lat in _poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_coords_utm) < 3: continue
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _n = len(_coords_utm)
                    _verts = np.array([[x, y, _bz] for x, y in _coords_utm], np.float32)
                    _cx2 = _verts[:, 0].mean(); _cy2 = _verts[:, 1].mean()
                    _cen_v = np.array([[_cx2, _cy2, _bz]], np.float32)
                    _spk_v.append(_verts); _spk_v.append(_cen_v)
                    for _i in range(_n):
                        _spk_f.append([_spk_off + _i,
                                        _spk_off + (_i + 1) % _n,
                                        _spk_off + _n])
                    _spk_off += _n + 1
                    _spk_cnt += 1
            except Exception: pass
        if _spk_v:
            _sv = np.concatenate(_spk_v).astype(np.float32)
            _sf = np.array(_spk_f, np.int32)
            _write_ply(_sv, _sf, os.path.join(MESH_DIR, 'surface_itu_asphalt_carparks.ply'))
            mat_plys.setdefault('itu_asphalt', []).append(
                ('meshes/surface_itu_asphalt_carparks.ply', 'surface_parks'))
            print(f'  Surface car parks: {_spk_cnt} → surface_itu_asphalt_carparks.ply '
                  f'({len(_sv):,} verts, {len(_sf):,} faces)')
        else:
            print('  Surface car parks: none found or no polygon geometry')
    except Exception as _e:
        print(f'  Surface car parks download failed: {_e}')
else:
    print('Surface car parks: skipped (INCLUDE_SURFACE_PARKS=False)')

# ── Grass / Parks / Sports Pitches → itu_wet_ground PLY ─────────────────────
# OSM: landuse=grass/meadow/park/recreation_ground + leisure=pitch/park/garden
# Material: itu_wet_ground (εr=30, σ=0.02, S=0.10) — ITU-R P.2040-2 Table 3
# Geometry: flat polygon at terrain z — significant ground reflector in open areas
if globals().get('INCLUDE_GREENSPACES', True):
    print('\nDownloading OSM greenspaces (grass/parks/pitches) ...')
    try:
        _gs_tags = {
            'landuse': ['grass', 'meadow', 'recreation_ground', 'village_green', 'allotments'],
            'leisure': ['pitch', 'park', 'garden', 'playing_fields', 'recreation_ground'],
            'natural': ['grassland', 'scrub', 'heath'],
        }
        if _ox_version >= (2, 0):
            gdf_gs = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_gs_tags)
        elif _ox_version >= (1, 3):
            gdf_gs = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_gs_tags)
        else:
            gdf_gs = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_gs_tags)
        _GS_MIN_AREA = float(globals().get('GREENSPACE_MIN_AREA_M2', 200.0))
        _gs_v, _gs_f, _gs_off, _gs_cnt, _gs_skip = [], [], 0, 0, 0
        for _, row in gdf_gs.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
                _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for _poly in _poly_list:
                    ex0, ny0 = to_utm.transform(_poly.exterior.coords[0][0], _poly.exterior.coords[0][1])
                    _area = _poly.area
                    if _area < _GS_MIN_AREA:
                        _gs_skip += 1; continue
                    _coords_utm = []
                    for lon, lat in _poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_coords_utm) < 3: continue
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _n = len(_coords_utm)
                    _verts = np.array([[x, y, _bz] for x, y in _coords_utm], np.float32)
                    _cx2 = _verts[:, 0].mean(); _cy2 = _verts[:, 1].mean()
                    _cen_v = np.array([[_cx2, _cy2, _bz]], np.float32)
                    _gs_v.append(_verts); _gs_v.append(_cen_v)
                    for _i in range(_n):
                        _gs_f.append([_gs_off + _i,
                                      _gs_off + (_i + 1) % _n,
                                      _gs_off + _n])
                    _gs_off += _n + 1
                    _gs_cnt += 1
            except Exception: pass
        if _gs_v:
            _gv = np.concatenate(_gs_v).astype(np.float32)
            _gf = np.array(_gs_f, np.int32)
            _write_ply(_gv, _gf, os.path.join(MESH_DIR, 'surface_itu_vegetation_greenspaces.ply'))
            mat_plys.setdefault('itu_wet_ground', []).append(
                ('meshes/surface_itu_vegetation_greenspaces.ply', 'greenspaces'))
            print(f'  Greenspaces: {_gs_cnt} polygons → surface_itu_vegetation_greenspaces.ply '
                  f'({len(_gv):,} verts, {len(_gf):,} faces)  skipped={_gs_skip} (< {_GS_MIN_AREA} m²)')
        else:
            print('  Greenspaces: none found')
    except Exception as _e:
        print(f'  Greenspaces download failed: {_e}')
else:
    print('Greenspaces: skipped (INCLUDE_GREENSPACES=False)')

# ── Footways / Pedestrian Areas → itu_concrete PLY ──────────────────────────
# OSM: highway=footway/pedestrian/living_street/steps + place=square
# Material: itu_concrete (εr=5.31, σ=0.092) — paved urban surfaces
# Geometry: flat polygon at terrain z — ground reflectors in city centres
# Portable: works globally (OSM coverage in all cities)
if globals().get('INCLUDE_FOOTWAYS', True):
    print('\nDownloading OSM footways / pedestrian areas ...')
    try:
        _fw_tags = {
            'highway': ['pedestrian', 'living_street', 'footway'],
            'place'  : ['square'],
            'area'   : ['yes'],
        }
        if _ox_version >= (2, 0):
            gdf_fw = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_fw_tags)
        elif _ox_version >= (1, 3):
            gdf_fw = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_fw_tags)
        else:
            gdf_fw = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_fw_tags)
        _FW_MIN_AREA = float(globals().get('FOOTWAY_MIN_AREA_M2', 50.0))
        _fw_v, _fw_f, _fw_off, _fw_cnt, _fw_skip = [], [], 0, 0, 0
        for _, row in gdf_fw.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
                _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for _poly in _poly_list:
                    if _poly.area < _FW_MIN_AREA:
                        _fw_skip += 1; continue
                    _coords_utm = []
                    for lon, lat in _poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_coords_utm) < 3: continue
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _n = len(_coords_utm)
                    _verts = np.array([[x, y, _bz] for x, y in _coords_utm], np.float32)
                    _cx2 = _verts[:, 0].mean(); _cy2 = _verts[:, 1].mean()
                    _cen_v = np.array([[_cx2, _cy2, _bz]], np.float32)
                    _fw_v.append(_verts); _fw_v.append(_cen_v)
                    for _i in range(_n):
                        _fw_f.append([_fw_off + _i,
                                      _fw_off + (_i + 1) % _n,
                                      _fw_off + _n])
                    _fw_off += _n + 1
                    _fw_cnt += 1
            except Exception: pass
        if _fw_v:
            _fv = np.concatenate(_fw_v).astype(np.float32)
            _ff = np.array(_fw_f, np.int32)
            _write_ply(_fv, _ff, os.path.join(MESH_DIR, 'surface_itu_concrete_footways.ply'))
            mat_plys.setdefault('itu_concrete', []).append(
                ('meshes/surface_itu_concrete_footways.ply', 'surfaces'))
            print(f'  Footways: {_fw_cnt} polygons → surface_itu_concrete_footways.ply '
                  f'({len(_fv):,} verts, {len(_ff):,} faces)  skipped={_fw_skip} (< {_FW_MIN_AREA} m²)')
        else:
            print('  Footways: none found (no polygon geometry — LineString footways excluded)')
    except Exception as _e:
        print(f'  Footways download failed: {_e}')
else:
    print('Footways: skipped (INCLUDE_FOOTWAYS=False)')



# ── Greenhouses → itu_glass PLY ──────────────────────────────────────────────
# building=greenhouse/glasshouse → extruded glass walls + roof
# ITU-R P.2040-2: itu_glass  er=6.27(f^0)  σ=0.0043·f^1.1925  s=0.08  xpd=0.02  wt=0.012m
# Effect: glass transmits more RF than brick — prevents over-attenuation in horticultural areas
if globals().get('INCLUDE_GREENHOUSES', True):
    print('\nDownloading OSM greenhouses ...')
    try:
        _gh_tags = {'building': ['greenhouse', 'glasshouse', 'conservatory']}
        if _ox_version >= (2, 0):
            gdf_gh = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_gh_tags)
        elif _ox_version >= (1, 3):
            gdf_gh = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_gh_tags)
        else:
            gdf_gh = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST,   west=SCENE_WEST, tags=_gh_tags)
        _GH_H_DEFAULT = 4.0   # m — typical greenhouse height
        _gh_v, _gh_f, _gh_off, _gh_cnt = [], [], 0, 0
        for _, row in gdf_gh.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                if geom.geom_type not in ('Polygon', 'MultiPolygon'): continue
                _ht = _GH_H_DEFAULT
                for _htag in ('height', 'building:height'):
                    _hv = str(row.get(_htag, '') or '')
                    if _hv not in ('nan', 'None', ''):
                        try: _ht = float(_hv.replace('m', '').strip()); break
                        except ValueError: pass
                _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for _poly in _poly_list:
                    _coords_utm = []
                    for lon, lat in _poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_coords_utm) < 3: continue
                    _cen_lon, _cen_lat = to_wgs84.transform(
                        _poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _res = _extrude_building(_coords_utm, _bz, _ht,
                                             roof_h=0.0, roof_shape='flat')
                    if _res is None: continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0: continue
                        _gh_f.extend([[fi + _gh_off for fi in face] for face in _ff])
                        _gh_v.append(_vv); _gh_off += len(_vv)
                    _gh_cnt += 1
            except Exception: pass
        if _gh_v:
            _gv = np.concatenate(_gh_v).astype(np.float32)
            _gf = np.array(_gh_f, np.int32)
            _write_ply(_gv, _gf, os.path.join(MESH_DIR, 'bld_itu_glass_greenhouses.ply'))
            mat_plys.setdefault('itu_glass', []).append(
                ('meshes/bld_itu_glass_greenhouses.ply', 'greenhouses'))
            print(f'  Greenhouses: {_gh_cnt} → bld_itu_glass_greenhouses.ply '
                  f'({len(_gv):,} verts, {len(_gf):,} faces)')
        else:
            print('  Greenhouses: none found in bbox')
    except Exception as _e:
        print(f'  Greenhouses download failed: {_e}')
else:
    print('Greenhouses: skipped (INCLUDE_GREENHOUSES=False)')



# ====================================================================
# ====================================================================
# V3 BLOCK -- ROOFTOP EQUIPMENT  (itu_metal, ITU-R P.2040-2 Table 3)
# HVAC / plant-room boxes on commercial/industrial rooftops.
# Hard diffraction edges critical above 2.8 GHz.
# er=1.0, sigma=1e7 S/m  (ITU-R P.2040-2 metal Table 3)
# Guard: entire block in try/except -- scene build never stops.
# ====================================================================
if globals().get('INCLUDE_ROOFTOP_EQUIPMENT', False):
    print('\nBuilding rooftop equipment meshes (V3, itu_metal) ...')
    try:
        import math as _mth_rte
        if 'gdf_bld' not in dir() or gdf_bld is None or len(gdf_bld) == 0:
            raise RuntimeError('gdf_bld not available -- run building download first')
        _rte_bld_types = set(globals().get('ROOFTOP_BLDG_TYPES',
            {'commercial','retail','industrial','office','supermarket',
             'warehouse','factory','hospital','university','school'}))
        _rte_h    = max(0.5, float(globals().get('ROOFTOP_EQUIP_HEIGHT_M', 2.5)))
        _rte_frac = min(0.9, max(0.05, float(globals().get('ROOFTOP_EQUIP_FRACTION', 0.25))))
        _rte_verts, _rte_faces, _rte_off, _rte_cnt, _rte_skip = [], [], 0, 0, 0
        for _, row in gdf_bld.iterrows():
            try:
                _btag = str(row.get('building', '') or '').lower()
                if _btag not in _rte_bld_types:
                    continue
                geom = row.geometry
                if geom is None or geom.is_empty:
                    continue
                _poly_list = (list(geom.geoms) if geom.geom_type == 'MultiPolygon'
                              else [geom])
                for _poly in _poly_list:
                    if _poly.geom_type != 'Polygon' or _poly.is_empty:
                        continue
                    _scale    = _mth_rte.sqrt(_rte_frac)
                    _buf_dist = -(_poly.length * (1 - _scale) / (4 * _mth_rte.pi))
                    try:
                        _small = _poly.buffer(_buf_dist)
                    except Exception:
                        _small = None
                    if _small is None or _small.is_empty:
                        continue
                    _bh = float(globals().get('DEFAULT_HEIGHT_M', 8.0))
                    for _ht in ('height', 'building:height'):
                        _hv = str(row.get(_ht, '') or '')
                        if _hv not in ('nan', 'None', '', 'none'):
                            try:
                                _bh = float(_hv.replace('m','').replace(' ','').strip())
                                break
                            except (ValueError, AttributeError):
                                pass
                    _bh = max(1.0, _bh)
                    try:
                        _cen_lon = float(_poly.centroid.x)
                        _cen_lat = float(_poly.centroid.y)
                        _base_z  = (0.0 if FLAT_TERRAIN
                                    else float(local_z(_cen_lon, _cen_lat)))
                    except Exception:
                        _base_z = 0.0
                    _coords_box = []
                    try:
                        _ext = (_small if _small.geom_type == 'Polygon'
                                else list(_small.geoms)[0])
                        for lon, lat in _ext.exterior.coords[:-1]:
                            ex, ny = to_utm.transform(lon, lat)
                            _coords_box.append((ex - center_utm[0], ny - center_utm[1]))
                    except Exception:
                        continue
                    if len(_coords_box) < 3:
                        continue
                    _res = _extrude_building(_coords_box, _base_z + _bh,
                                             _rte_h, roof_h=0.0, roof_shape='flat')
                    if _res is None:
                        continue
                    _wv, _wf, _rv, _rf = _res
                    for _vv, _ff in ((_wv, _wf), (_rv, _rf)):
                        if _vv is None or len(_vv) == 0:
                            continue
                        _rte_faces.extend([[fi + _rte_off for fi in face] for face in _ff])
                        _rte_verts.append(np.asarray(_vv, np.float32))
                        _rte_off += len(_vv)
                    _rte_cnt += 1
            except Exception:
                _rte_skip += 1
                continue
        if _rte_verts:
            _rv_arr  = np.concatenate(_rte_verts).astype(np.float32)
            _rf_arr  = np.array(_rte_faces, np.int32)
            _rte_ply = os.path.join(MESH_DIR, 'infra_itu_metal_rooftop_equip.ply')
            _write_ply(_rv_arr, _rf_arr, _rte_ply)
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_rooftop_equip.ply', 'rooftop_equipment'))
            print(f'  Rooftop equipment: {_rte_cnt} boxes, {_rte_skip} skipped '
                  f'-> infra_itu_metal_rooftop_equip.ply '
                  f'({len(_rv_arr):,} verts, {len(_rf_arr):,} faces)')
        else:
            print(f'  Rooftop equipment: none generated (0 matching or all skipped={_rte_skip})')
    except Exception as _e_rte:
        print(f'  [WARN] Rooftop equipment failed: {_e_rte} -- continuing')
else:
    print('Rooftop equipment: skipped (INCLUDE_ROOFTOP_EQUIPMENT=False)')


# ====================================================================
# V4 HELPERS -- shared box + cylinder mesh generators
# ====================================================================
def _box_mesh(cx, cy, base_z, length, width, height, angle_rad):
    import math as _mb
    _hl, _hw2 = length / 2, width / 2
    _c, _s    = _mb.cos(angle_rad), _mb.sin(angle_rad)
    _corners  = [(-_hl,-_hw2), (_hl,-_hw2), (_hl,_hw2), (-_hl,_hw2)]
    _pts = [(cx + _c*lx - _s*ly, cy + _s*lx + _c*ly) for lx, ly in _corners]
    _vt  = np.array([
        [_pts[0][0], _pts[0][1], base_z],         [_pts[1][0], _pts[1][1], base_z],
        [_pts[2][0], _pts[2][1], base_z],         [_pts[3][0], _pts[3][1], base_z],
        [_pts[0][0], _pts[0][1], base_z+height],  [_pts[1][0], _pts[1][1], base_z+height],
        [_pts[2][0], _pts[2][1], base_z+height],  [_pts[3][0], _pts[3][1], base_z+height],
    ], np.float32)
    _fc = np.array([[0,1,2],[0,2,3],[4,6,5],[4,7,6],
                    [0,1,5],[0,5,4],[2,3,7],[2,7,6],
                    [1,2,6],[1,6,5],[3,0,4],[3,4,7]], np.int32)
    return _vt, _fc

def _cylinder_mesh(cx, cy, base_z, radius, height, n_segs):
    import math as _mc
    n_segs = max(3, int(n_segs))
    radius = max(1e-4, float(radius))
    height = max(1e-4, float(height))
    _angs   = [2 * _mc.pi * i / n_segs for i in range(n_segs)]
    _ring_b = np.array([[cx+radius*_mc.cos(a), cy+radius*_mc.sin(a), base_z]
                         for a in _angs], np.float32)
    _ring_t = np.array([[cx+radius*_mc.cos(a), cy+radius*_mc.sin(a), base_z+height]
                         for a in _angs], np.float32)
    _cap_b  = np.array([[cx, cy, base_z]],           np.float32)
    _cap_t  = np.array([[cx, cy, base_z + height]],  np.float32)
    _verts  = np.concatenate([_ring_b, _ring_t, _cap_b, _cap_t])
    _faces  = []
    for _i in range(n_segs):
        _j = (_i + 1) % n_segs
        _faces += [[_i, _j, n_segs+_j], [_i, n_segs+_j, n_segs+_i]]
        _faces += [[2*n_segs,   _j,         _i         ]]
        _faces += [[2*n_segs+1, n_segs+_i,  n_segs+_j  ]]
    return _verts, np.array(_faces, np.int32)


# ====================================================================
# V4 BLOCK A -- PARKED VEHICLES  (itu_metal, ITU-R P.2040-2 Table 3)
# Box 4.5x1.8x1.4m along OSM parking lanes/areas.
# Full Fresnel-zone blocker at 28 GHz. Guard: OSM fail = skip silently.
# ====================================================================
if globals().get('INCLUDE_PARKED_VEHICLES', False):
    print('\nBuilding parked vehicle meshes (V4, itu_metal) ...')
    try:
        import math as _mth_pv
        _veh_l  = max(1.0, float(globals().get('VEHICLE_LENGTH_M',  4.5)))
        _veh_w  = max(0.5, float(globals().get('VEHICLE_WIDTH_M',   1.8)))
        _veh_h  = max(0.3, float(globals().get('VEHICLE_HEIGHT_M',  1.4)))
        _veh_sp = max(_veh_l + 0.1, float(globals().get('VEHICLE_SPACING_M', 6.0)))
        _veh_of = max(0.0, float(globals().get('VEHICLE_OFFSET_M',  2.5)))
        gdf_pk  = None
        try:
            _pk_tags = {'amenity': 'parking',
                        'parking': ['lane','street_side','on_street']}
            gdf_pk = (ox.features_from_bbox(
                          bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
                          tags=_pk_tags)
                      if _ox_version >= (2, 0) else
                      ox.features_from_bbox(
                          bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
                          tags=_pk_tags))
            print(f'  Parking features: {len(gdf_pk)}')
        except Exception as _epk:
            print(f'  [WARN] Parking OSM download failed: {_epk} -- no vehicles placed')
        _pveh_verts, _pveh_faces, _pveh_off, _pveh_cnt, _pveh_skip = [], [], 0, 0, 0
        if gdf_pk is not None and len(gdf_pk) > 0:
            for _, row in gdf_pk.iterrows():
                try:
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _lines_utm = []
                    if geom.geom_type == 'LineString':
                        _pts = []
                        for lon, lat in geom.coords:
                            ex, ny = to_utm.transform(lon, lat)
                            _pts.append((ex - center_utm[0], ny - center_utm[1]))
                        if len(_pts) >= 2:
                            _lines_utm.append(_pts)
                    elif geom.geom_type in ('Polygon', 'MultiPolygon'):
                        _plist = (list(geom.geoms) if geom.geom_type == 'MultiPolygon'
                                  else [geom])
                        for _pp in _plist:
                            if _pp.is_empty:
                                continue
                            _pts = []
                            for lon, lat in _pp.exterior.coords[:-1]:
                                ex, ny = to_utm.transform(lon, lat)
                                _pts.append((ex - center_utm[0], ny - center_utm[1]))
                            if len(_pts) >= 2:
                                _lines_utm.append(_pts)
                    for _pts_l in _lines_utm:
                        for _si in range(len(_pts_l) - 1):
                            _x0, _y0 = _pts_l[_si]
                            _x1, _y1 = _pts_l[_si + 1]
                            _seg_len  = _mth_pv.hypot(_x1-_x0, _y1-_y0)
                            if _seg_len < _veh_l:
                                continue
                            _dx, _dy = (_x1-_x0)/_seg_len, (_y1-_y0)/_seg_len
                            _nx, _ny = -_dy, _dx
                            _angle   = _mth_pv.atan2(_dy, _dx)
                            _pos = 0.0
                            while _pos + _veh_l <= _seg_len:
                                _mid = _pos + _veh_l / 2
                                _vcx = _x0 + _dx*_mid + _nx*_veh_of
                                _vcy = _y0 + _dy*_mid + _ny*_veh_of
                                try:
                                    _lon_v, _lat_v = to_wgs84.transform(
                                        _vcx + center_utm[0], _vcy + center_utm[1])
                                    _bz_v = (0.0 if FLAT_TERRAIN
                                             else float(local_z(_lon_v, _lat_v)))
                                except Exception:
                                    _bz_v = 0.0
                                _vt, _fc = _box_mesh(_vcx, _vcy, _bz_v,
                                                      _veh_l, _veh_w, _veh_h, _angle)
                                _pveh_faces.extend([[fi+_pveh_off for fi in f] for f in _fc])
                                _pveh_verts.append(_vt)
                                _pveh_off += 8
                                _pveh_cnt += 1
                                _pos += _veh_sp
                except Exception:
                    _pveh_skip += 1
                    continue
        if _pveh_verts:
            _pv_arr = np.concatenate(_pveh_verts).astype(np.float32)
            _pf_arr = np.array(_pveh_faces, np.int32)
            _pv_ply = os.path.join(MESH_DIR, 'infra_itu_metal_parked_vehicles.ply')
            _write_ply(_pv_arr, _pf_arr, _pv_ply)
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_parked_vehicles.ply', 'parked_vehicles'))
            print(f'  Parked vehicles: {_pveh_cnt} boxes, {_pveh_skip} skipped '
                  f'-> infra_itu_metal_parked_vehicles.ply '
                  f'({len(_pv_arr):,} verts, {len(_pf_arr):,} faces)')
        else:
            print(f'  Parked vehicles: none placed (no geometry or all skipped={_pveh_skip})')
    except Exception as _e_pv:
        print(f'  [WARN] Parked vehicles failed: {_e_pv} -- continuing')
else:
    print('Parked vehicles: skipped (INCLUDE_PARKED_VEHICLES=False)')


# ====================================================================
# V4 BLOCK B -- STREET LAMPS  (itu_metal, ITU-R P.2040-2 Table 3)
# Hexagonal cylinders r=0.05m h=6m every 35m on road edges.
# Point scatterers above 10 GHz. Guard: road download fail = skip.
# ====================================================================
if globals().get('INCLUDE_STREET_LAMPS', False):
    print('\nBuilding street lamp meshes (V4, itu_metal) ...')
    try:
        import math as _mth_lmp
        _lmp_h   = max(1.0,  float(globals().get('LAMP_HEIGHT_M',      6.0)))
        _lmp_r   = max(0.01, float(globals().get('LAMP_RADIUS_M',      0.05)))
        _lmp_sp  = max(5.0,  float(globals().get('LAMP_SPACING_M',     35.0)))
        _lmp_of  = max(0.0,  float(globals().get('LAMP_ROAD_OFFSET_M', 2.5)))
        _lmp_rts = set(globals().get('LAMP_ROAD_TYPES',
                        {'primary','secondary','tertiary','residential','unclassified'}))
        _N_POLE_SEGS = 6
        _edges_road  = None
        try:
            _G_roads = (ox.graph_from_bbox(
                            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH))
                        if _ox_version >= (2, 0) else
                        ox.graph_from_bbox(
                            north=SCENE_NORTH, south=SCENE_SOUTH,
                            east=SCENE_EAST,   west=SCENE_WEST,
                            network_type='drive'))
            _edges_road = ox.graph_to_gdfs(_G_roads, nodes=False, edges=True)
            print(f'  Road edges: {len(_edges_road)}')
        except Exception as _elmp:
            print(f'  [WARN] Road network download failed: {_elmp} -- no lamps placed')
        _lmp_verts, _lmp_faces, _lmp_off, _lmp_cnt, _lmp_skip = [], [], 0, 0, 0
        if _edges_road is not None and len(_edges_road) > 0:
            for _, row in _edges_road.iterrows():
                try:
                    _hw = row.get('highway', '')
                    if isinstance(_hw, list):
                        _hw = _hw[0] if _hw else ''
                    if str(_hw) not in _lmp_rts:
                        continue
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _pts_utm = []
                    for lon, lat in geom.coords:
                        ex, ny = to_utm.transform(lon, lat)
                        _pts_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_pts_utm) < 2:
                        continue
                    for _si in range(len(_pts_utm) - 1):
                        _x0, _y0 = _pts_utm[_si]
                        _x1, _y1 = _pts_utm[_si + 1]
                        _seg_len  = _mth_lmp.hypot(_x1-_x0, _y1-_y0)
                        if _seg_len < 1.0:
                            continue
                        _dx, _dy = (_x1-_x0)/_seg_len, (_y1-_y0)/_seg_len
                        _nx, _ny = -_dy, _dx
                        _n_lamps = max(1, int(_seg_len / _lmp_sp))
                        for _li in range(_n_lamps):
                            _t  = (_li + 0.5) * _seg_len / _n_lamps
                            _px = _x0 + _dx*_t + _nx*_lmp_of
                            _py = _y0 + _dy*_t + _ny*_lmp_of
                            try:
                                _lon_l, _lat_l = to_wgs84.transform(
                                    _px + center_utm[0], _py + center_utm[1])
                                _bz_l = (0.0 if FLAT_TERRAIN
                                         else float(local_z(_lon_l, _lat_l)))
                            except Exception:
                                _bz_l = 0.0
                            _vt, _fc = _cylinder_mesh(
                                _px, _py, _bz_l, _lmp_r, _lmp_h, _N_POLE_SEGS)
                            _lmp_faces.extend([[fi+_lmp_off for fi in f] for f in _fc])
                            _lmp_verts.append(_vt)
                            _lmp_off += len(_vt)
                            _lmp_cnt += 1
                except Exception:
                    _lmp_skip += 1
                    continue
        if _lmp_verts:
            _lv_arr = np.concatenate(_lmp_verts).astype(np.float32)
            _lf_arr = np.array(_lmp_faces, np.int32)
            _lp_ply = os.path.join(MESH_DIR, 'infra_itu_metal_street_lamps.ply')
            _write_ply(_lv_arr, _lf_arr, _lp_ply)
            mat_plys.setdefault('itu_metal', []).append(
                ('meshes/infra_itu_metal_street_lamps.ply', 'street_lamps'))
            print(f'  Street lamps: {_lmp_cnt} poles, {_lmp_skip} skipped '
                  f'-> infra_itu_metal_street_lamps.ply '
                  f'({len(_lv_arr):,} verts, {len(_lf_arr):,} faces)')
        else:
            print(f'  Street lamps: none placed (no matching edges or all skipped={_lmp_skip})')
    except Exception as _e_lmp:
        print(f'  [WARN] Street lamps failed: {_e_lmp} -- continuing')
else:
    print('Street lamps: skipped (INCLUDE_STREET_LAMPS=False)')


# ====================================================================
# V5 BLOCK A -- HUMAN BODY PHANTOMS  (itu_concrete, ITU-R P.1238 / P.2040-2)
# Cylinder r=0.2m h=1.7m at each RX. 15-20 dB blockage at 60 GHz.
# Material: itu_concrete (er=5.31, ITU-R P.2040-2 Table 3).
# Guard: missing/bad CSV rows silently skipped.
# ====================================================================
if globals().get('INCLUDE_BODY_PHANTOMS', False):
    print('\nBuilding human body phantom meshes (V5, itu_concrete) ...')
    try:
        _ph_h    = max(0.5,  float(globals().get('PHANTOM_HEIGHT_M', 1.7)))
        _ph_r    = max(0.05, float(globals().get('PHANTOM_RADIUS_M', 0.2)))
        _rx_agl  = float(globals().get('RX_AGL_M', 1.5))
        _N_PH_SEGS = 8
        _RX_CSV  = globals().get('MEASUREMENT_CSV', globals().get('RX_CSV', None))
        if _RX_CSV is None:
            print('  [SKIP] MEASUREMENT_CSV not set in Cell 0')
        elif not os.path.exists(str(_RX_CSV)):
            print(f'  [SKIP] File not found: {_RX_CSV}')
        else:
            import pandas as _pd_ph
            try:
                _df_rx = _pd_ph.read_csv(str(_RX_CSV))
            except Exception as _ecsv:
                _df_rx = None
                print(f'  [SKIP] Cannot read CSV: {_ecsv}')
            if _df_rx is not None and len(_df_rx) > 0:
                _lon_col = next((c for c in _df_rx.columns
                                  if 'lon' in c.lower() or 'lng' in c.lower()), None)
                _lat_col = next((c for c in _df_rx.columns
                                  if 'lat' in c.lower() and 'lon' not in c.lower()), None)
                if _lat_col is None:
                    _lat_col = next((c for c in _df_rx.columns
                                     if 'lat' in c.lower()), None)
                if _lon_col is None or _lat_col is None:
                    print(f'  [SKIP] No lon/lat columns in CSV. '
                          f'Available: {list(_df_rx.columns)}')
                else:
                    _ph_verts, _ph_faces, _ph_off = [], [], 0
                    _ph_cnt, _ph_skip = 0, 0
                    for _, row in _df_rx.iterrows():
                        try:
                            _lon_p = float(row[_lon_col])
                            _lat_p = float(row[_lat_col])
                            if not (-180 <= _lon_p <= 180 and -90 <= _lat_p <= 90):
                                _ph_skip += 1; continue
                            _ex, _ny = to_utm.transform(_lon_p, _lat_p)
                            _pxp  = _ex - center_utm[0]
                            _pyp  = _ny - center_utm[1]
                            _bz_p = (0.0 if FLAT_TERRAIN
                                     else float(local_z(_lon_p, _lat_p)))
                            _bot  = _bz_p + _rx_agl - _ph_h
                            _vt, _fc = _cylinder_mesh(
                                _pxp, _pyp, _bot, _ph_r, _ph_h, _N_PH_SEGS)
                            _ph_faces.extend([[fi+_ph_off for fi in f] for f in _fc])
                            _ph_verts.append(_vt)
                            _ph_off += len(_vt)
                            _ph_cnt += 1
                        except Exception:
                            _ph_skip += 1
                            continue
                    if _ph_verts:
                        _phv_arr = np.concatenate(_ph_verts).astype(np.float32)
                        _phf_arr = np.array(_ph_faces, np.int32)
                        _pp_ply  = os.path.join(
                            MESH_DIR, 'infra_itu_concrete_body_phantoms.ply')
                        _write_ply(_phv_arr, _phf_arr, _pp_ply)
                        mat_plys.setdefault('itu_concrete', []).append(
                            ('meshes/infra_itu_concrete_body_phantoms.ply',
                             'body_phantoms'))
                        print(f'  Body phantoms: {_ph_cnt} written, {_ph_skip} skipped '
                              f'-> infra_itu_concrete_body_phantoms.ply '
                              f'({len(_phv_arr):,} verts, {len(_phf_arr):,} faces)')
                    else:
                        print(f'  Body phantoms: none placed (all skipped={_ph_skip})')
    except Exception as _e_ph:
        print(f'  [WARN] Body phantoms failed: {_e_ph} -- continuing')
else:
    print('Body phantoms: skipped (INCLUDE_BODY_PHANTOMS=False)')


# ====================================================================
# V5 BLOCK B -- BUS SHELTERS / KIOSKS  (itu_metal + itu_glass, ITU-R P.2040-2)
# Metal roof (er=1, sigma=1e7) + 3 glass sides (er=6.27, sigma~0).
# Glass transparent at 915 MHz; hard blocker at 28-60 GHz.
# Guard: OSM fail + per-feature errors silently skip.
# ====================================================================
if globals().get('INCLUDE_SHELTERS', True):   # enabled bus shelters, phone boxes:
    print('\nBuilding bus shelter / kiosk meshes (V5, itu_metal + itu_glass) ...')
    try:
        _sh_h = max(1.0, float(globals().get('SHELTER_HEIGHT_M', 2.5)))
        _sh_d = max(0.5, float(globals().get('SHELTER_DEPTH_M',  1.5)))
        _sh_w = max(0.5, float(globals().get('SHELTER_WIDTH_M',  3.0)))
        gdf_sh = None
        try:
            _sh_tags = {'amenity': ['shelter','telephone','vending_machine'],
                        'man_made': ['telephone_box']}
            gdf_sh = (ox.features_from_bbox(
                          bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
                          tags=_sh_tags)
                      if _ox_version >= (2, 0) else
                      ox.features_from_bbox(
                          bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
                          tags=_sh_tags))
            print(f'  Shelter features: {len(gdf_sh)}')
        except Exception as _esh:
            print(f'  [WARN] Shelter OSM download failed: {_esh} -- no shelters placed')
        _sh_metal_v, _sh_metal_f, _sh_m_off = [], [], 0
        _sh_glass_v, _sh_glass_f, _sh_g_off = [], [], 0
        _sh_cnt, _sh_skip = 0, 0
        if gdf_sh is not None and len(gdf_sh) > 0:
            for _, row in gdf_sh.iterrows():
                try:
                    geom = row.geometry
                    if geom is None or geom.is_empty:
                        continue
                    _cx_wgs = float(geom.centroid.x)
                    _cy_wgs = float(geom.centroid.y)
                    _ex, _ny = to_utm.transform(_cx_wgs, _cy_wgs)
                    _sx   = _ex - center_utm[0]
                    _sy   = _ny - center_utm[1]
                    _bz_s = (0.0 if FLAT_TERRAIN
                              else float(local_z(_cx_wgs, _cy_wgs)))
                    _hw_s, _hd_s = _sh_w / 2, _sh_d / 2
                    _roof_v = np.array([
                        [_sx-_hw_s, _sy-_hd_s, _bz_s+_sh_h],
                        [_sx+_hw_s, _sy-_hd_s, _bz_s+_sh_h],
                        [_sx+_hw_s, _sy+_hd_s, _bz_s+_sh_h],
                        [_sx-_hw_s, _sy+_hd_s, _bz_s+_sh_h],
                    ], np.float32)
                    _roof_f = np.array([[0,1,2],[0,2,3]], np.int32)
                    _sh_metal_f.extend([[fi+_sh_m_off for fi in f] for f in _roof_f])
                    _sh_metal_v.append(_roof_v); _sh_m_off += 4
                    for _p0, _p1 in [
                        ([_sx-_hw_s, _sy-_hd_s], [_sx+_hw_s, _sy-_hd_s]),
                        ([_sx-_hw_s, _sy-_hd_s], [_sx-_hw_s, _sy+_hd_s]),
                        ([_sx+_hw_s, _sy-_hd_s], [_sx+_hw_s, _sy+_hd_s]),
                    ]:
                        _panel_v = np.array([
                            [_p0[0], _p0[1], _bz_s],
                            [_p1[0], _p1[1], _bz_s],
                            [_p1[0], _p1[1], _bz_s+_sh_h],
                            [_p0[0], _p0[1], _bz_s+_sh_h],
                        ], np.float32)
                        _panel_f = np.array([[0,1,2],[0,2,3]], np.int32)
                        _sh_glass_f.extend([[fi+_sh_g_off for fi in f] for f in _panel_f])
                        _sh_glass_v.append(_panel_v); _sh_g_off += 4
                    _sh_cnt += 1
                except Exception:
                    _sh_skip += 1
                    continue
        for _mat_key, _vlist, _flist, _fname, _label in [
            ('itu_metal', _sh_metal_v, _sh_metal_f,
             'infra_itu_metal_shelter_roofs.ply', 'shelter_roofs'),
            ('itu_glass', _sh_glass_v, _sh_glass_f,
             'infra_itu_glass_shelter_walls.ply', 'shelter_walls'),
        ]:
            if _vlist:
                _sv_arr = np.concatenate(_vlist).astype(np.float32)
                _sf_arr = np.array(_flist, np.int32)
                _sp_ply = os.path.join(MESH_DIR, _fname)
                _write_ply(_sv_arr, _sf_arr, _sp_ply)
                mat_plys.setdefault(_mat_key, []).append((f'meshes/{_fname}', _label))
                print(f'  Shelters ({_mat_key}): {_sh_cnt} written, '
                      f'{_sh_skip} skipped -> {_fname} '
                      f'({len(_sv_arr):,} verts, {len(_sf_arr):,} faces)')
        if not _sh_metal_v:
            print(f'  Shelters: none placed (no OSM features; skipped={_sh_skip})')
    except Exception as _e_sh:
        print(f'  [WARN] Shelters failed: {_e_sh} -- continuing')
else:
    print('Shelters: skipped (INCLUDE_SHELTERS=False)')

# ── Landuse / Open ground → material mapping ──────────────────────────────
# Covers retail/industrial land, cemeteries, farmland, reservoirs
# not caught by building= or natural= tags above.
if globals().get('INCLUDE_LANDUSE', True):
    print('\nDownloading OSM landuse polygons ...')
    try:
        _lu_tags = {
            'landuse': ['retail','industrial','commercial','residential',
                        'cemetery','farmland','orchard','vineyard',
                        'reservoir','salt_pond'],
            'natural': ['reservoir','sand','beach','mud','shingle'],
        }
        if _ox_version >= (2, 0):
            gdf_lu = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_lu_tags)
        elif _ox_version >= (1, 3):
            gdf_lu = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_lu_tags)
        else:
            gdf_lu = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_lu_tags)
        _LU_MAT = {
            'retail': 'itu_concrete', 'commercial': 'itu_concrete',
            'industrial': 'itu_metal', 'residential': 'itu_brick',
            'cemetery': 'itu_wet_ground', 'farmland': 'itu_wet_ground',
            'orchard': 'itu_vegetation', 'vineyard': 'itu_vegetation',
            'reservoir': 'itu_water', 'salt_pond': 'itu_water',
            'sand': 'itu_wet_ground', 'beach': 'itu_wet_ground',
            'mud': 'itu_wet_ground', 'shingle': 'itu_wet_ground',
        }
        _LU_MIN_AREA = float(globals().get('LANDUSE_MIN_AREA_M2', 500.0))
        _lu_by_mat = {}  # mat_name → (verts, faces, offset)
        _lu_cnt, _lu_skip = 0, 0
        for _, row in gdf_lu.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                if geom.geom_type not in ('Polygon','MultiPolygon'): continue
                _lu_type = str(row.get('landuse', row.get('natural', '')) or '').lower()
                _mat = _LU_MAT.get(_lu_type, 'itu_wet_ground')
                _poly_list = list(geom.geoms) if geom.geom_type == 'MultiPolygon' else [geom]
                for _poly in _poly_list:
                    # NOTE: area must be measured in projected (UTM) coords --
                    # _poly.area on the raw WGS84 geometry is in degrees^2
                    # (~1e-5 for a real 500 m^2 plot), so the area filter
                    # below would always trigger and skip every polygon.
                    _coords_utm = []
                    for lon, lat in _poly.exterior.coords[:-1]:
                        ex, ny = to_utm.transform(lon, lat)
                        _coords_utm.append((ex - center_utm[0], ny - center_utm[1]))
                    if len(_coords_utm) < 3: continue
                    _poly_utm_area = sg.Polygon(_coords_utm).area
                    if _poly_utm_area < _LU_MIN_AREA:
                        _lu_skip += 1; continue
                    _cen_lon, _cen_lat = to_wgs84.transform(_poly.centroid.x, _poly.centroid.y)
                    _bz = 0.0 if FLAT_TERRAIN else float(local_z(_cen_lon, _cen_lat))
                    _n = len(_coords_utm)
                    _verts = np.array([[x, y, _bz] for x, y in _coords_utm], np.float32)
                    _cx2 = _verts[:,0].mean(); _cy2 = _verts[:,1].mean()
                    _cen_v = np.array([[_cx2, _cy2, _bz]], np.float32)
                    if _mat not in _lu_by_mat:
                        _lu_by_mat[_mat] = {'v': [], 'f': [], 'off': 0}
                    _d = _lu_by_mat[_mat]
                    for _i in range(_n):
                        _d['f'].append([_d['off']+_i, _d['off']+(_i+1)%_n, _d['off']+_n])
                    _d['v'].append(_verts); _d['v'].append(_cen_v)
                    _d['off'] += _n + 1
                    _lu_cnt += 1
            except Exception: pass
        for _mat, _d in _lu_by_mat.items():
            if not _d['v']: continue
            _lv = np.concatenate(_d['v']).astype(np.float32)
            _lf = np.array(_d['f'], np.int32)
            _ply_name = f'surface_{_mat}_landuse.ply'
            _write_ply(_lv, _lf, os.path.join(MESH_DIR, _ply_name))
            mat_plys.setdefault(_mat, []).append((f'meshes/{_ply_name}', 'landuse'))
        print(f'  Landuse: {_lu_cnt} polygons → {len(_lu_by_mat)} materials  skipped={_lu_skip}')
        for _m, _d in _lu_by_mat.items():
            print(f'    {_m}: {len(np.concatenate(_d["v"])):,} verts')
    except Exception as _e:
        print(f'  Landuse download failed: {_e}')
else:
    print('Landuse: skipped (INCLUDE_LANDUSE=False)')

# ── Tunnels → flag only (no geometry — mark shadow zones) ─────────────────
# Tunnels reduce path gain significantly; flag as metadata for Cell 15 MLP.
# No 3D mesh needed — P.2040 does not model tunnel interiors.
if globals().get('INCLUDE_TUNNELS', True):
    print('\nFlagging OSM tunnels ...')
    try:
        _tun_tags = {'tunnel': ['yes','building_passage','culvert']}
        if _ox_version >= (2, 0):
            gdf_tun = ox.features_from_bbox(
                bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH), tags=_tun_tags)
        elif _ox_version >= (1, 3):
            gdf_tun = ox.features_from_bbox(
                bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST), tags=_tun_tags)
        else:
            gdf_tun = ox.features_from_bbox(
                north=SCENE_NORTH, south=SCENE_SOUTH,
                east=SCENE_EAST, west=SCENE_WEST, tags=_tun_tags)
        _tun_coords = []
        for _, row in gdf_tun.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty: continue
                _cen = geom.centroid
                ex, ny = to_utm.transform(_cen.x, _cen.y)
                _tun_coords.append((ex - center_utm[0], ny - center_utm[1]))
            except Exception: pass
        import json as _jmod
        _tun_file = os.path.join(OUTPUT_DIR if 'OUTPUT_DIR' in dir() else MESH_DIR, 'tunnel_locations.json')
        with open(_tun_file, 'w') as _tf:
            _jmod.dump({'tunnels': _tun_coords}, _tf)
        print(f'  Tunnels: {len(_tun_coords)} flagged → tunnel_locations.json')
    except Exception as _e:
        print(f'  Tunnels download failed: {_e}')
else:
    print('Tunnels: skipped (INCLUDE_TUNNELS=False)')

# ── Save vegetation footprints for P.833 post-processing ─────────────────────
if INCLUDE_VEGETATION and 'gdf_veg' in dir() and len(gdf_veg) > 0:
    import geopandas as _gpd
    _veg_utm = gdf_veg.to_crs(epsg=UTM_EPSG)
    _veg_polys = _veg_utm[_veg_utm.geometry.geom_type.isin(['Polygon','MultiPolygon'])].copy()
    _veg_out = os.path.join(SCENE_DIR, 'vegetation_footprints.geojson')
    _keep_cols = ['geometry'] + [c for c in ['natural', 'landuse'] if c in _veg_polys.columns]
    _veg_polys[_keep_cols].to_file(_veg_out, driver='GeoJSON')
    print(f'  Vegetation footprints saved: {len(_veg_polys)} polygons → {_veg_out}')


## CELL 4A — Overture Maps Buildings (Alternative Source)

Tries [Overture Maps Foundation](https://overturemaps.org) buildings theme
(merged OSM + Microsoft/Google ML footprints + commercial sources) for the
same `SCENE_WEST/EAST/SOUTH/NORTH` bbox as CELL 4.

- Free, no API key, GeoParquet via the `overturemaps` Python package
- Output is remapped to the **same columns** CELL 4's `_bld_mat`/`_bld_height`/
  `_roof_mat`/`_roof_height` expect (`height`, `building`, `building:levels`,
  `building:material`, `roof:material`, `osmid`) — so it's a drop-in
  replacement for `gdf_bld`
- Falls back to the existing OSM `gdf_bld` from CELL 4 if Overture is
  unavailable (no package / no network / empty bbox)
- Does **not** overwrite `gdf_bld` automatically — inspect `gdf_bld_overture`
  first, then set `gdf_bld = gdf_bld_overture` to use it in CELL 4's
  extrusion loop


In [ ]:
# ============================================================
# CELL 4A — OVERTURE MAPS BUILDINGS (ALTERNATIVE SOURCE)
# ============================================================
# Same bbox as CELL 4 (SCENE_WEST/EAST/SOUTH/NORTH). Produces gdf_bld_overture
# with the same columns CELL 4 expects, so it's a drop-in for gdf_bld.
import time as _t4a
import pandas as pd

_OVERTURE_OK = False
gdf_bld_overture = None

try:
    from overturemaps import core as _ovt_core
    import geopandas as _gpd4a

    print('Downloading Overture Maps buildings ...')
    _t0 = _t4a.time()
    # bbox = (xmin, ymin, xmax, ymax) = (west, south, east, north)
    _gdf_ovt = _ovt_core.geodataframe(
        'building',
        bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH)
    )
    print(f'  {len(_gdf_ovt)} raw Overture building features  ({_t4a.time()-_t0:.1f}s)')

    if len(_gdf_ovt) > 0:
        # ── Remap Overture schema -> OSM-compatible columns used by CELL 4 ──
        # Overture 'class'/'subtype' values -> OSM building= tag equivalents
        _OVT_CLASS_MAP = {
            'residential': 'residential', 'house': 'house',
            'apartments': 'apartments',    'commercial': 'commercial',
            'retail': 'retail',            'industrial': 'industrial',
            'warehouse': 'warehouse',      'office': 'office',
            'civic': 'civic',              'education': 'school',
            'medical': 'hospital',         'religious': 'church',
            'agricultural': 'farm',        'garage': 'garage',
            'transportation': 'train_station',
            'entertainment': 'sports_centre',
            'service': 'commercial',
        }

        def _ovt_building_tag(row):
            for _k in ('subtype', 'class'):
                _v = row.get(_k)
                if _v and str(_v).lower() in _OVT_CLASS_MAP:
                    return _OVT_CLASS_MAP[str(_v).lower()]
            return 'yes'   # generic OSM default

        gdf_bld_overture = _gpd4a.GeoDataFrame({
            'geometry': _gdf_ovt.geometry,
            'osmid': _gdf_ovt.get('id', _gdf_ovt.index.astype(str)),
            'height': _gdf_ovt.get('height', pd.Series([None]*len(_gdf_ovt))),
            'building:levels': _gdf_ovt.get('num_floors', pd.Series([None]*len(_gdf_ovt))),
            'building:material': _gdf_ovt.get('facade_material', pd.Series([None]*len(_gdf_ovt))),
            'roof:material': _gdf_ovt.get('roof_material', pd.Series([None]*len(_gdf_ovt))),
            'roof:shape': _gdf_ovt.get('roof_shape', pd.Series([None]*len(_gdf_ovt))),
            'building': _gdf_ovt.apply(_ovt_building_tag, axis=1),
        }, geometry='geometry', crs='EPSG:4326')

        # Fill NaN/None height & levels as empty strings — _bld_height handles
        # missing values via try/except float() on '0'
        for _col in ('height', 'building:levels', 'building:material', 'roof:material'):
            gdf_bld_overture[_col] = gdf_bld_overture[_col].apply(
                lambda v: '' if v is None or (isinstance(v, float) and np.isnan(v)) else v)

        # Keep only polygonal geometries (Overture can include points for tiny structures)
        gdf_bld_overture = gdf_bld_overture[
            gdf_bld_overture.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
        ].copy()

        _OVERTURE_OK = len(gdf_bld_overture) > 0
        print(f'  Remapped: {len(gdf_bld_overture)} polygon buildings')
        _n_height = (gdf_bld_overture['height'] != '').sum()
        print(f'  With height attribute   : {_n_height} '
              f'({100*_n_height/max(len(gdf_bld_overture),1):.0f}%)')

        # Cache alongside CELL 4's OSM cache
        import pickle as _pkl4a
        _OVT_CACHE = os.path.join(BASE_DIR, 'buildings_gdf_overture_cache.pkl')
        try:
            with open(_OVT_CACHE, 'wb') as _pf:
                _pkl4a.dump(gdf_bld_overture, _pf)
            print(f'  buildings_gdf_overture_cache.pkl saved')
        except Exception as _pe:
            print(f'  [WARN] cache save failed: {_pe}')

except ImportError:
    print('overturemaps package not installed — run: pip install overturemaps')
except Exception as _e:
    print(f'Overture download failed: {_e}')

# ── Comparison vs OSM (CELL 4) ───────────────────────────────────────────────
if 'gdf_bld' in dir():
    print(f'\n{"Source":<20} {"Buildings":>10} {"With height":>12}')
    print('-' * 44)
    _osm_h = (gdf_bld['height'].astype(str).str.replace('m','').str.strip()
              .apply(lambda v: v not in ('', '0', 'nan', 'None'))).sum() \
             if 'height' in gdf_bld.columns else 0
    print(f'{"OSM (CELL 4)":<20} {len(gdf_bld):>10} {_osm_h:>12}')
    if _OVERTURE_OK:
        print(f'{"Overture (CELL 4A)":<20} {len(gdf_bld_overture):>10} {_n_height:>12}')

if _OVERTURE_OK:
    print('\nTo use Overture buildings in CELL 4\'s extrusion loop, run:')
    print('  gdf_bld = gdf_bld_overture')
    print('then re-run the extrusion portion of CELL 4 (after the gdf_bld download).')
else:
    print('\nOverture unavailable — continuing with OSM gdf_bld from CELL 4.')


## CELL 5 — Write scene.xml (Sionna 0.19)

Assembles all PLY meshes into a **Sionna 0.19 / Mitsuba 2.1.0** compatible scene XML.

**Prerequisites — must have run first:**
- Cell 2 (config) — sets `BASE_DIR`, `SCENE_DIR`, `MESH_DIR`
- Cell 14 (terrain) — sets `origin_elev_asl` and writes `origin_elev2.json`
- Cell 4 (buildings) — writes PLY files to `MESH_DIR`

**Output:** `scene_v4_full/scene.xml` — load this in `sionna019_differentiable_rt_fixed.ipynb`

Non-ITU materials (asphalt, vegetation, water) are embedded as custom `RadioMaterial`
entries with ITU-R P.2040-2 EM parameters computed at the configured frequency.


In [ ]:
# ============================================================
# CELL 5 — WRITE SCENE.XML  (Mitsuba 2.1.0 / Sionna 0.19)
# ============================================================
# ITU-R P.2040-2 material definitions as used in Sionna 0.19.
# Each shape references a PLY file and a material by name.

# Sionna 0.19 valid ITU-R P.2040-2 built-in names + custom radio-materials
# Valid built-ins: concrete, brick, plywood, glass, ceiling_board, chipboard,
#                  floorboard, metal, very_dry_ground, medium_dry_ground, wet_ground
# NOT valid in 0.19: itu_wood → itu_plywood  |  itu_asphalt/vegetation/water → ground built-ins
# ── ITU-R P.2040-2 (2023) Table 3 — frequency-portable material parameters ──
# Formula: er(f) = a*f^b   sigma(f) = c*f^d   f in GHz
# s=scattering coefficient  xpd=cross-pol discrimination
# Static columns (b=0 or d=0) are frequency-independent.
# ── Guard: load origin_elev_asl from origin_elev2.json if Cell 14 was not run ──
if 'origin_elev_asl' not in dir():
    import json as _j2
    _elev_json = os.path.join(SCENE_DIR, 'origin_elev2.json')
    if os.path.exists(_elev_json):
        origin_elev_asl = _j2.load(open(_elev_json))['origin_elev_asl_m']
        print(f'[Cell 5] origin_elev_asl loaded from origin_elev2.json: {origin_elev_asl:.2f} m ASL')
    else:
        origin_elev_asl = 0.0
        print('[Cell 5] WARNING: origin_elev_asl not defined and origin_elev2.json not found — using 0.0 m')

# ── Reconstruct mat_plys from disk if Cell 4 was not run this session ────────
if 'mat_plys' not in dir() or not mat_plys:
    mat_plys = {}
    _pfx_role5 = {'bld_': 'buildings', 'road_': 'roads',
                  'water_': 'water',   'veg_': 'vegetation',
                  'trees_': 'vegetation', 'rail_': 'railways',
                  'barriers_': 'barriers', 'infra_': 'infrastructure',
                  'surface_': 'surfaces'}
    _KNOWN_ITU5 = sorted([
        'itu_brick', 'itu_concrete', 'itu_glass', 'itu_wood', 'itu_metal',
        'itu_wet_ground', 'itu_medium_dry_ground', 'itu_very_dry_ground',
        'itu_asphalt', 'itu_vegetation', 'itu_water',
    ], key=len, reverse=True)
    def _ply_to_mat5(fname, pfx):
        raw = fname[len(pfx):-4]
        for km in _KNOWN_ITU5:
            if raw == km or raw.startswith(km + '_'):
                return km
        return raw
    for _fn5 in os.listdir(MESH_DIR):
        if not _fn5.endswith('.ply') or _fn5 == 'terrain.ply':
            continue
        for _pfx5, _role5 in _pfx_role5.items():
            if _fn5.startswith(_pfx5):
                _mat5 = _ply_to_mat5(_fn5, _pfx5)
                mat_plys.setdefault(_mat5, []).append(
                    (os.path.basename(MESH_DIR) + '/' + _fn5, _role5))
                break
    print(f'[Cell 5] mat_plys reconstructed from disk: {list(mat_plys.keys())}')

_ITU_P2040_PARAMS = {
    #                    a       b       c        d      s     xpd
    'itu_concrete' : (5.31,  0.000,  0.0326, 0.8095, 0.30, 0.10),
    'itu_brick'    : (3.91,  0.000,  0.0238, 0.0000, 0.25, 0.10),  # S=0.25 matches diff RT Cell 4A
    'itu_glass'    : (6.27,  0.000,  0.0043, 1.1925, 0.10, 0.05),
    'itu_plywood'  : (1.99,  0.000,  0.0047, 1.0718, 0.20, 0.10),  # P.2040 wood -> plywood in 0.19
    'itu_metal'    : (1.00,  0.000,  1.0e7,  0.0000, 0.05, 0.05),  # perfect conductor
    'itu_wet_ground'        : (30.0, -0.400, 0.1500, 1.3000, 0.35, 0.05),  # S=0.35 matches diff RT Cell 4A
    'itu_water'      : (80.0,  0.000, 0.0100, 0.0000, 0.02, 0.05),  # ITU-R P.527 fresh water — S=0.02 specular water surface
    'itu_medium_dry_ground'  : (15.0, -0.100, 0.0350, 1.6300, 0.10, 0.05),  # ITU-R P.2040-2 Table 3
    'itu_very_dry_ground'    : ( 3.0,  0.000, 0.00015,2.5200, 0.10, 0.05),  # ITU-R P.2040-2 Table 3
    # ── Correct ITU-R P.2040-2 values — not in Sionna 0.19 built-ins ────────
    'itu_vegetation'         : ( 1.50, 0.000, 0.0020, 0.5000, 0.40, 0.50),  # P.2040-2 Table 3 vegetation — S=0.40 matches sim notebooks
    'itu_asphalt'            : ( 2.56, 0.000, 0.0050, 0.0000, 0.30, 0.15),  # P.2040-2 Table 3 asphalt
}
_f_ghz_c5 = float(globals().get('FREQUENCY_HZ', 915.95e6)) / 1e9
ITU_MATERIALS = {}
for _mn, (_a, _b, _c, _d, _s, _xpd) in _ITU_P2040_PARAMS.items():
    _er  = round(_a * (_f_ghz_c5 ** _b), 6)
    _sig = round(_c * (_f_ghz_c5 ** _d), 8)
    ITU_MATERIALS[_mn] = (_er, _sig, _s, _xpd)
print(f'ITU_MATERIALS computed at {_f_ghz_c5*1e3:.2f} MHz (ITU-R P.2040-2):')
for _mn, (_er, _sig, _s, _xpd) in ITU_MATERIALS.items():
    print(f'  {_mn:<22}  er={_er:.4f}  sigma={_sig:.6f}  S={_s:.2f}')
# Valid Sionna 0.19 ITU built-in type names — any itu_ prefixed material NOT in
# this set will be written with a custom_ prefix to avoid ItuMaterial constructor
_VALID_ITU_019 = {
    'itu_concrete', 'itu_brick', 'itu_plywood', 'itu_glass',
    'itu_ceiling_board', 'itu_chipboard', 'itu_floorboard',
    'itu_metal', 'itu_very_dry_ground', 'itu_medium_dry_ground', 'itu_wet_ground',
}

def _xml_id(name):
    """All materials use their original id — diffuse BSDF accepts any id.""";
    return name

# Material name remapping: mat_plys keys → ITU_MATERIALS keys
_MAT_REMAP_019 = {
    'itu_wood'       : 'itu_plywood',          # itu_wood not in 0.19 registry → itu_plywood
    'itu_water'      : 'itu_medium_dry_ground', # 'water' not a valid ITU type in Sionna 0.19 — sim notebook CELL 4A overrides eps/sigma by name afterward
    'itu_vegetation' : 'itu_ceiling_board',     # 'vegetation' not valid — ceiling_board er~1.5 is closest built-in; refined in CELL 4A
    'itu_asphalt'    : 'itu_very_dry_ground',   # 'asphalt' not valid — very_dry_ground er~3.0 is closest built-in; refined in CELL 4A
}
TERRAIN_MATERIAL = 'itu_wet_ground'

# Collect all materials actually used
used_mats = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

lines = []
lines.append('<?xml version="1.0" encoding="utf-8"?>')
lines.append('<scene version="2.1.0">')
lines.append('')
lines.append('  <!-- ── ITU-R P.2040-2 Materials ────────────────────── -->')

# Apply mat_plys key remapping before writing
_used_mats_remapped = set()
for m in used_mats:
    _used_mats_remapped.add(_MAT_REMAP_019.get(m, m))
_used_mats_remapped.add(TERRAIN_MATERIAL)

# Iterate over the final remapped ids directly (not ITU_MATERIALS.keys()) --
# remap targets like itu_ceiling_board have no entry in ITU_MATERIALS (they
# aren't computed in _ITU_P2040_PARAMS), so writing only ITU_MATERIALS keys
# left them undeclared while shapes still referenced them ("unresolved
# reference" from Mitsuba's parser). The bsdf body itself only needs a
# placeholder diffuse reflectance regardless of material name.
# Distinct visual colours per material (rendering only, no EM effect) --
# previously hardcoded to a single grey for every material.
_ITU_COLOURS_019 = {
    'itu_concrete'          : '0.539 0.539 0.539',
    'itu_brick'              : '1.000 0.498 0.055',
    'itu_glass'              : '0.596 0.875 0.541',
    'itu_plywood'            : '0.514 0.376 0.220',
    'itu_metal'              : '0.220 0.220 0.254',
    'itu_wet_ground'         : '0.910 0.569 0.055',
    'itu_very_dry_ground'    : '0.498 0.498 0.498',
    'itu_medium_dry_ground'  : '0.780 0.780 0.780',
    'itu_ceiling_board'      : '0.180 0.450 0.180',  # proxy for vegetation
    'itu_chipboard'          : '0.514 0.376 0.220',
    'itu_floorboard'         : '0.514 0.376 0.220',
}
for mat_name in sorted(_used_mats_remapped):
    _id = _xml_id(mat_name)
    _rgb = _ITU_COLOURS_019.get(mat_name, '0.5 0.5 0.5')
    lines.append(f'  <bsdf type="diffuse" id="{_id}">')
    lines.append(f'    <rgb name="reflectance" value="{_rgb}"/>')
    lines.append(f'  </bsdf>')
    lines.append('')

lines.append('  <!-- ── Terrain ─────────────────────────────────────── -->')
lines.append('  <shape type="ply" id="mesh-ground">')
lines.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines.append(f'    <ref id="{_xml_id(TERRAIN_MATERIAL)}" name="bsdf"/>')
lines.append('    <boolean name="face_normals" value="true"/>')
lines.append('  </shape>')
lines.append('')

lines.append('  <!-- ── Buildings ───────────────────────────────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        _ref_mat = _xml_id(_MAT_REMAP_019.get(mat_name, mat_name))
        _mesh_id = 'mesh-' + ply_path.split('/')[-1].replace('.ply', '')
        lines.append(f'  <shape type="ply" id="{_mesh_id}">')
        lines.append(f'    <string name="filename" value="{ply_path}"/>')
        lines.append(f'    <ref id="{_ref_mat}" name="bsdf"/>')
        lines.append(f'    <boolean name="face_normals" value="true"/>')
        lines.append(f'  </shape>')

lines.append('')
lines.append('</scene>')

scene_xml = os.path.join(SCENE_DIR, 'scene.xml')
# ── Permanent safety net: dedupe declared bsdf ids + auto-fallback for any
# referenced-but-undeclared material id. Protects against future material
# names (new OSM tags, new ITU types) ever again causing a "duplicate ID"
# or "unresolved reference" crash -- instead of failing, it prints a clear
# warning and patches the XML with a generic diffuse bsdf.
import re as _re_guard

def _guard_bsdf_refs(_lines, _label):
    _seen_ids = set()
    _out = []
    _i = 0
    while _i < len(_lines):
        _line = _lines[_i]
        _m = _re_guard.match(r'\\s*<bsdf[^>]+id="([^"]+)"', _line)
        if _m:
            _mid = _m.group(1)
            _block = [_line]
            _j = _i
            if not _line.rstrip().endswith('/>'):
                while _lines[_j].strip() != '</bsdf>':
                    _j += 1
                    _block.append(_lines[_j])
            if _mid in _seen_ids:
                print(f'[{_label}] WARNING: duplicate bsdf id "{_mid}" -- dropping repeat declaration')
                _i = _j + 1
                if _i < len(_lines) and _lines[_i].strip() == '':
                    _i += 1
                continue
            _seen_ids.add(_mid)
            _out.extend(_block)
            _i = _j + 1
            if _i < len(_lines) and _lines[_i].strip() == '':
                _out.append(_lines[_i])
                _i += 1
            continue
        _out.append(_line)
        _i += 1
    _lines = _out

    _joined = '\\n'.join(_lines)
    _declared = set(_re_guard.findall(r'<bsdf[^>]+id="([^"]+)"', _joined))
    _referenced = set(_re_guard.findall(r'<ref id="([^"]+)" name="bsdf"', _joined))
    _missing = _referenced - _declared
    if _missing:
        print(f'[{_label}] WARNING: {len(_missing)} referenced material id(s) had no declared '
              f'bsdf -- adding generic diffuse fallback so the scene still loads: {sorted(_missing)}')
        _insert_at = next(_k for _k, _l in enumerate(_lines) if '<!-- ── Terrain' in _l)
        _fallback = []
        for _mid in sorted(_missing):
            _fallback += [f'  <bsdf type="diffuse" id="{_mid}">',
                          f'    <rgb name="reflectance" value="0.5 0.5 0.5"/>',
                          f'  </bsdf>', '']
        _lines[_insert_at:_insert_at] = _fallback
    return _lines

lines = _guard_bsdf_refs(lines, "CELL 5")

with open(scene_xml, 'w') as f:
    f.write('\n'.join(lines))

print(f'Wrote: {scene_xml}')
print(f'  Materials : {len(used_mats)}')
total_shapes = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes-1} building parts)')

# Save scene metadata for main notebook
# Guard: n_ok only defined when Cell 4 (buildings) ran in this session
_n_bld = globals().get('n_ok', sum(len(v) for v in mat_plys.values()))
meta = {
    'scene_center_lon'  : center_lon,
    'scene_center_lat'  : center_lat,
    'origin_elev_asl_m' : origin_elev_asl,
    'utm_epsg'          : UTM_EPSG,
    'bbox'              : {'west': SCENE_WEST, 'east': SCENE_EAST,
                           'south': SCENE_SOUTH, 'north': SCENE_NORTH},
    'n_buildings'       : _n_bld,
    'terrain_grid_n'    : TERRAIN_GRID_N,
    'tile_zoom'         : TILE_ZOOM,
}
params_json = os.path.join(BASE_DIR, 'scene_parameters.json')
with open(params_json, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  Metadata  : {params_json}')


In [ ]:
# ============================================================
# CELL 6 — VERIFY SCENE
# ============================================================
# Quick sanity checks before handing the scene to the main notebook.

import glob as glob_mod

ply_files = sorted(glob_mod.glob(os.path.join(MESH_DIR, '*.ply')))
total_kb = sum(os.path.getsize(p) for p in ply_files) / 1024

print('=' * 60)
print('SCENE VERIFICATION')
print('=' * 60)
print(f'PLY files   : {len(ply_files)}')
print(f'Total size  : {total_kb/1024:.1f} MB')
print()

print(f'scene.xml   : {os.path.getsize(scene_xml)/1024:.0f} KB')

# Load with Mitsuba to verify (requires Sionna env)
try:
    import mitsuba as mi
    mi.set_variant('scalar_rgb')
    scene_mi = mi.load_file(scene_xml)
    bbox = scene_mi.bbox()
    print()
    print(f'Mitsuba load: OK')
    print(f'  BBox X    : [{float(bbox.min[0]):.1f}, {float(bbox.max[0]):.1f}] m')
    print(f'  BBox Y    : [{float(bbox.min[1]):.1f}, {float(bbox.max[1]):.1f}] m')
    print(f'  BBox Z    : [{float(bbox.min[2]):.1f}, {float(bbox.max[2]):.1f}] m')
except Exception as e:
    print(f'Mitsuba load: {e}')

print()
print('Scene metadata (scene_parameters.json):')
with open(params_json) as f:
    print(json.dumps(json.load(f), indent=2))

print()
print('DONE — scene ready for sionna019_main_simulation.ipynb')
print(f'Set BASE_DIR = "{BASE_DIR}" in Cell 0c of the main notebook.')

## CELL 7 — 2D Scene Map: OSM Buildings + Receiver Locations + RSSI Heatmap


In [ ]:
# ============================================================
# CELL 7 — 2D MAP: OSM BUILDINGS + TX/RX DOTS
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import re, warnings

# ── Config — edit here ────────────────────────────────────────────────────
OFCOM_CSV   = os.path.join(BASE_DIR, 'london915.csv')

# Number of RX points to plot (first N rows in CSV order, same as simulation).
# Set to None to plot all rows in the file.
NUM_RX_PLOT = 1200      # ← adjust to match NUM_RX in simulation notebook

SAVE_FIG_2D = True
FIG_DPI_2D  = 150

# ── Auto-detect header row + parse CSV ───────────────────────────────────
_tx_lat = _tx_lon = None
_rx_rows = []

if not os.path.exists(OFCOM_CSV):
    print(f'CSV not found: {OFCOM_CSV}')
else:
    # Read TX site coordinates from metadata lines
    with open(OFCOM_CSV, 'r', encoding='utf-8', errors='replace') as _f:
        _all_lines = _f.readlines()

    for _line in _all_lines[:40]:
        if 'latitude'  in _line.lower() and _tx_lat is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lat = float(_m.group(1))
        if 'longitude' in _line.lower() and _tx_lon is None:
            _m = re.search(r'([-+]?\d+\.\d+)', _line)
            if _m: _tx_lon = float(_m.group(1))

    # Find header row: first line containing both 'Latitude' and 'Longitude'
    _hdr_idx = next(
        (i for i, l in enumerate(_all_lines) if 'Latitude' in l and 'Longitude' in l),
        None
    )

    if _hdr_idx is None:
        print('WARNING: could not find header row in CSV')
    else:
        _df = pd.read_csv(OFCOM_CSV, skiprows=_hdr_idx, low_memory=False)

        # Flexible column matching
        def _fcol(df, *kws):
            for c in df.columns:
                if all(k.lower() in c.strip().lower() for k in kws):
                    return c
            return None

        _lat_col  = _fcol(_df, 'latitude')
        _lon_col  = _fcol(_df, 'longitude')
        _rssi_col = _fcol(_df, 'measurement') or _fcol(_df, 'dbm')

        if _lat_col and _lon_col and _rssi_col:
            for _c in [_lat_col, _lon_col, _rssi_col]:
                _df[_c] = pd.to_numeric(_df[_c], errors='coerce')
            _sel = _df[[_lat_col, _lon_col, _rssi_col]].dropna().reset_index(drop=True)

            # Take first NUM_RX_PLOT rows (same order as simulation)
            if NUM_RX_PLOT is not None:
                _sel = _sel.head(NUM_RX_PLOT)

            _rx_rows = list(zip(_sel[_lon_col], _sel[_lat_col], _sel[_rssi_col]))
            print(f'Loaded {len(_rx_rows)} RX points  '
                  f'(RSSI {float(_sel[_rssi_col].min()):.1f} – {float(_sel[_rssi_col].max()):.1f} dBm)')
        else:
            print(f'WARNING: could not find lat/lon/rssi columns. '
                  f'Available: {list(_df.columns)}')

# ── Plot ──────────────────────────────────────────────────────────────────
fig2d, ax2d = plt.subplots(figsize=(11, 10), dpi=FIG_DPI_2D)

# Building footprints
_gdf_plot = gdf_bld.copy()
if hasattr(_gdf_plot, 'crs') and _gdf_plot.crs and str(_gdf_plot.crs) != 'EPSG:4326':
    _gdf_plot = _gdf_plot.to_crs('EPSG:4326')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _gdf_plot.plot(ax=ax2d, facecolor='#d8d0c4', edgecolor='#999999',
                   linewidth=0.15, alpha=0.85, zorder=2)

ax2d.set_xlim(SCENE_WEST, SCENE_EAST)
ax2d.set_ylim(SCENE_SOUTH, SCENE_NORTH)
ax2d.set_aspect('equal')
ax2d.set_facecolor('#eef2f5')
ax2d.set_xlabel('Longitude', fontsize=10)
ax2d.set_ylabel('Latitude',  fontsize=10)
ax2d.tick_params(labelsize=8)
ax2d.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, zorder=1)

# Scene bbox border
from matplotlib.patches import Rectangle as _Rect
ax2d.add_patch(_Rect((SCENE_WEST, SCENE_SOUTH),
                      SCENE_EAST - SCENE_WEST, SCENE_NORTH - SCENE_SOUTH,
                      linewidth=1.5, edgecolor='black', facecolor='none',
                      linestyle='--', zorder=6))

# RX dots coloured by RSSI
if _rx_rows:
    _lons_p = np.array([p[0] for p in _rx_rows])
    _lats_p = np.array([p[1] for p in _rx_rows])
    _rssi_p = np.array([p[2] for p in _rx_rows])
    _vmin_p = float(np.percentile(_rssi_p, 2))
    _vmax_p = float(np.percentile(_rssi_p, 98))
    _sc2d = ax2d.scatter(_lons_p, _lats_p, c=_rssi_p, s=2.5,
                         cmap='RdYlGn', vmin=_vmin_p, vmax=_vmax_p,
                         alpha=0.65, linewidths=0, zorder=5, label='RX (RSSI)')
    _cb2d = fig2d.colorbar(_sc2d, ax=ax2d, fraction=0.025, pad=0.01, shrink=0.7)
    _cb2d.set_label('RSSI (dBm)', fontsize=9)
    _cb2d.ax.tick_params(labelsize=8)

# TX star
if _tx_lon and _tx_lat:
    ax2d.plot(_tx_lon, _tx_lat, marker='*', markersize=16, color='red',
              markeredgecolor='darkred', markeredgewidth=0.8,
              zorder=10, label='TX (transmitter)')
    ax2d.annotate('TX', (_tx_lon, _tx_lat),
                  textcoords='offset points', xytext=(8, 5),
                  fontsize=9, color='darkred', fontweight='bold', zorder=11)

_n_bld = len(_gdf_plot) if '_gdf_plot' in dir() else 0
ax2d.set_title(
    f'Nottingham 915 MHz — OSM Scene Map\n'
    f'{_n_bld:,} buildings  |  {len(_rx_rows):,} RX (first {NUM_RX_PLOT})  |  '
    f'{(SCENE_EAST-SCENE_WEST)*111.32*np.cos(np.radians((SCENE_SOUTH+SCENE_NORTH)/2)):.1f} × '
    f'{(SCENE_NORTH-SCENE_SOUTH)*111.32:.1f} km',
    fontsize=11)

ax2d.legend(loc='upper right', fontsize=9, markerscale=3,
            framealpha=0.85, edgecolor='grey')

plt.tight_layout()

if SAVE_FIG_2D:
    _fig2d_path = os.path.join(BASE_DIR, 'scene_map_2d.png')
    fig2d.savefig(_fig2d_path, dpi=FIG_DPI_2D, bbox_inches='tight')
    print(f'Saved: {_fig2d_path}')

plt.show()
print('2D map done.')

## CELL 8 — Interactive 3D RSSI Heatmap (Plotly)


## CELL 9 — Customised Antenna Patterns (Ofcom 915 MHz)

Defines TX and RX antenna pattern functions matching the Ofcom drive-test equipment,
then exports all link-budget parameters to `scene_parameters.json` for automatic
pick-up by the simulation notebook.


In [ ]:
# ============================================================
# CELL 9 — CUSTOMISED ANTENNA PATTERNS  (Ofcom 915 MHz)
# ============================================================
# Defines TX / RX antenna pattern callables matching the Ofcom
# drive-test equipment used in the nottingham915.csv dataset.
#
# TX : collinear omni mast — donut shaped, peak = TX_ANTENNA_GAIN_DBI
# RX : vehicle rooftop — effectively isotropic; equipment losses
#      (cable, splitter, BPF) folded into RX_EXTRA_GAIN_DB
#
# Pattern functions are TensorFlow callables accepted by Sionna's
# PlanarArray.  A safe fallback to 'hw_dipole' / 'iso' is provided
# if the Sionna version does not accept callables.
#
# After defining patterns, all link-budget parameters are written to
# scene_parameters.json so the simulation notebook reads them
# automatically.
# ============================================================

import numpy as np
import json as _json

# ── Half-wave dipole directivity (linear) ─────────────────────────────────
_D_HW = 1.6409   # D of ideal half-wave dipole (= 2.15 dBi in linear)

# ── TX : donut omni scaled to TX_ANTENNA_GAIN_DBI ─────────────────────────
def _tx_pattern_915(theta, phi):
    """
    Ofcom 915 MHz TX — collinear omni mast.
    Pattern: donut (half-wave dipole shape) scaled so peak = TX_ANTENNA_GAIN_DBI.
    F_theta = scale * cos(pi/2 * cos(theta)) / sin(theta)
    F_phi   = 0
    """
    import tensorflow as tf
    _scale = np.float32((10 ** (TX_ANTENNA_GAIN_DBI / 10) / _D_HW) ** 0.5)
    cos_t  = tf.cos(theta)
    sin_t  = tf.sin(theta)
    safe_s = tf.where(tf.abs(sin_t) < 1e-6,
                      tf.ones_like(sin_t) * 1e-6, sin_t)
    f_theta = tf.cast(_scale * tf.cos(np.float32(np.pi / 2) * cos_t) / safe_s,
                      tf.complex64)
    f_phi   = tf.zeros_like(f_theta)
    return f_theta, f_phi

# ── RX : isotropic — equipment losses in RX_EXTRA_GAIN_DB ─────────────────
def _rx_pattern_915(theta, phi):
    """
    Ofcom 915 MHz RX — vehicle rooftop omni.
    Treated as isotropic (0 dBi); drive-test NLOS paths arrive from varied
    elevations so a directional pattern would introduce azimuth bias.
    Cable / splitter / filter losses are captured in RX_EXTRA_GAIN_DB.
    """
    import tensorflow as tf
    _iso = tf.cast(tf.ones_like(theta) / np.float32(np.sqrt(4.0 * np.pi)),
                   tf.complex64)
    f_phi = tf.zeros_like(_iso)
    return _iso, f_phi

# ── Helper: build PlanarArray with fallback ───────────────────────────────
def _make_antenna_array(pattern_fn, fallback='hw_dipole'):
    """
    Try to create a 1×1 PlanarArray with a callable pattern.
    Falls back to a named built-in if Sionna rejects the callable.
    Returns (array, mode_string).
    """
    try:
        from sionna.rt import PlanarArray
        arr = PlanarArray(num_rows=1, num_cols=1,
                          vertical_spacing=0.5, horizontal_spacing=0.5,
                          pattern=pattern_fn, polarization='V')
        return arr, 'callable'
    except Exception as _e:
        try:
            from sionna.rt import PlanarArray
            arr = PlanarArray(num_rows=1, num_cols=1,
                              vertical_spacing=0.5, horizontal_spacing=0.5,
                              pattern=fallback, polarization='V')
            return arr, f'fallback={fallback} ({_e})'
        except Exception as _e2:
            return None, f'failed ({_e2})'

# ── Verify pattern shape at sample angles ─────────────────────────────────
print('Antenna pattern check (915 MHz):')
print(f'  TX pattern : donut omni  peak = {TX_ANTENNA_GAIN_DBI:+.1f} dBi')
print(f'  RX pattern : isotropic   system gain = {RX_EXTRA_GAIN_DB:.1f} dB')
print(f'  Frequency  : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'  TX AGL     : {TX_AGL_M:.1f} m   RX AGL : {RX_AGL_M:.1f} m')
print(f'  TX EIRP    : {TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI:.1f} dBm  '
      f'(conducted={TX_CONDUCTED_DBM:.1f} + antenna={TX_ANTENNA_GAIN_DBI:.1f})')
print()

# Evaluate TX pattern at horizon (theta=pi/2) and zenith (theta=0)
try:
    import tensorflow as tf
    _th_h = tf.constant([np.pi/2], dtype=tf.float32)
    _ph_h = tf.constant([0.0],     dtype=tf.float32)
    _fth, _ = _tx_pattern_915(_th_h, _ph_h)
    _g_horizon = 10 * np.log10(float(tf.reduce_sum(tf.abs(_fth)**2).numpy()) * _D_HW)
    print(f'  TX gain at horizon (theta=90°): {_g_horizon:.2f} dBi  '
          f'(expected {TX_ANTENNA_GAIN_DBI:.2f} dBi)')

    _th_z = tf.constant([0.01], dtype=tf.float32)   # avoid exact 0
    _fth_z, _ = _tx_pattern_915(_th_z, _ph_h)
    _g_zenith = 10 * np.log10(max(float(tf.reduce_sum(tf.abs(_fth_z)**2).numpy()), 1e-12) * _D_HW)
    print(f'  TX gain at zenith  (theta≈0°) : {_g_zenith:.2f} dBi  (expected deep null)')
except Exception as _e:
    print(f'  Pattern evaluation skipped (TensorFlow not available in this env): {_e}')

# ── Optional: build arrays if sionna.rt is importable ─────────────────────
print()
try:
    _tx_arr, _tx_mode = _make_antenna_array(_tx_pattern_915, fallback='hw_dipole')
    _rx_arr, _rx_mode = _make_antenna_array(_rx_pattern_915, fallback='iso')
    print(f'  PlanarArray TX : [{_tx_mode}]')
    print(f'  PlanarArray RX : [{_rx_mode}]')
    print('  (Arrays ready — assign to scene.tx_array / scene.rx_array in sim notebook)')
    # Store for downstream cells
    _tx_array_915 = _tx_arr
    _rx_array_915 = _rx_arr
except Exception as _e:
    print(f'  sionna.rt not available in scene builder env — arrays not built: {_e}')
    print('  Pattern functions _tx_pattern_915 / _rx_pattern_915 are defined and')
    print('  can be copied to / imported by the simulation notebook.')

# ── Export antenna + link budget params to scene_parameters.json ──────────
_params_path = os.path.join(BASE_DIR, 'scene_parameters.json')
try:
    with open(_params_path) as _f:
        _meta = _json.load(_f)
except FileNotFoundError:
    _meta = {}

_meta.update({
    'frequency_hz'         : FREQUENCY_HZ,
    'antenna_pattern'      : ANTENNA_PATTERN,
    'tx_agl_m'             : TX_AGL_M,
    'rx_agl_m'             : RX_AGL_M,
    'tx_conducted_dbm'     : TX_CONDUCTED_DBM,
    'tx_antenna_gain_dbi'  : TX_ANTENNA_GAIN_DBI,
    'tx_eirp_dbm'          : TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI,
    'rx_extra_gain_db'     : RX_EXTRA_GAIN_DB,
    'site_correction_db'   : SITE_CORRECTION_DB,
})

with open(_params_path, 'w') as _f:
    _json.dump(_meta, _f, indent=2)

print()
print(f'scene_parameters.json updated: {_params_path}')
print(json.dumps({k: v for k, v in _meta.items()
                  if k in ('frequency_hz','antenna_pattern','tx_eirp_dbm',
                           'tx_agl_m','rx_agl_m','rx_extra_gain_db')}, indent=2))

---

# Section B — Blender Scene Conversion

These cells are **standalone** and can be run independently of Section A. They convert Blender-exported XML or OSM-built scenes into the target Sionna format.

Set the input/output paths at the top of each cell before running.


## CELL B1 — Blender XML → Sionna 0.19

Reads a Blender-exported or Sionna 2.0 scene XML and writes a Sionna 0.19 compatible version:
- Strips `mat-` prefix from ITU material IDs
- Replaces `diffuse`/`twosided` BSDFs with `conductor` + ITU-R P.2040-2 `eta`/`k`
- Maps colour-named materials to the nearest ITU equivalent

**Inputs to set:** `BLENDER_XML_IN`, `SIONNA19_XML_OUT`


In [ ]:
# ============================================================
# CELL B1 — WRITE SCENE_WITH_FULL_019.XML  (Sionna 0.19)
# ============================================================
# Same logic as Cell 5 (direct PLY discovery from MESH_DIR).
# Writes scene_with_full_019.xml — includes all current meshes.
# Plain diffuse BSDF — same format as scene.xml (confirmed working).
# ============================================================

# ── Guard: load origin_elev_asl if Cell 14 was not run ───────────────────────
if 'origin_elev_asl' not in dir():
    import json as _j2
    _elev_json = os.path.join(SCENE_DIR, 'origin_elev2.json')
    if os.path.exists(_elev_json):
        origin_elev_asl = _j2.load(open(_elev_json))['origin_elev_asl_m']
        print(f'[Cell B1] origin_elev_asl loaded: {origin_elev_asl:.2f} m ASL')
    else:
        origin_elev_asl = 0.0
        print('[Cell B1] WARNING: origin_elev_asl not defined — using 0.0 m')

# ── Reconstruct mat_plys from MESH_DIR ───────────────────────────────────────
_pfx_role_b1 = {'bld_': 'buildings',  'road_': 'roads',
                'water_': 'water',    'veg_': 'vegetation',
                'trees_': 'vegetation', 'rail_': 'railways',
                'barriers_': 'barriers', 'infra_': 'infrastructure',
                'surface_': 'surfaces'}
_KNOWN_ITU_B1 = sorted([
    'itu_brick', 'itu_concrete', 'itu_glass', 'itu_wood', 'itu_metal',
    'itu_wet_ground', 'itu_medium_dry_ground', 'itu_very_dry_ground',
    'itu_asphalt', 'itu_vegetation', 'itu_water',
], key=len, reverse=True)

def _ply_to_mat_b1(fname, pfx):
    raw = fname[len(pfx):-4]
    for km in _KNOWN_ITU_B1:
        if raw == km or raw.startswith(km + '_'):
            return km
    return raw

def _ply_has_vertices(path):
    try:
        with open(path, 'rb') as _f:
            for _line in _f:
                _s = _line.decode('ascii', 'ignore').strip()
                if _s.startswith('element vertex'):
                    return int(_s.split()[-1]) > 0
                if _s == 'end_header':
                    break
    except Exception:
        pass
    return False

mat_plys_b1 = {}
_empty_plys = []
for _fn in sorted(os.listdir(MESH_DIR)):
    if not _fn.endswith('.ply') or _fn == 'terrain.ply':
        continue
    _fp = os.path.join(MESH_DIR, _fn)
    if not _ply_has_vertices(_fp):
        _empty_plys.append(_fn)
        continue
    for _pfx, _role in _pfx_role_b1.items():
        if _fn.startswith(_pfx):
            _mat = _ply_to_mat_b1(_fn, _pfx)
            mat_plys_b1.setdefault(_mat, []).append(('meshes/' + _fn, _role))
            break

if _empty_plys:
    print(f'Skipped {len(_empty_plys)} empty PLY files: {sorted(_empty_plys)}')
print(f'PLY materials found: {sorted(mat_plys_b1.keys())}')

# Material remapping (itu_wood → itu_plywood for Sionna 0.19)
_MAT_REMAP_B1 = {
    'itu_wood'       : 'itu_plywood',
    'itu_water'      : 'itu_medium_dry_ground',  # 'water' not a valid ITU type in Sionna 0.19
    'itu_vegetation' : 'itu_ceiling_board',      # 'vegetation' not valid — closest built-in
    'itu_asphalt'    : 'itu_very_dry_ground',    # 'asphalt' not valid — closest built-in
}
TERRAIN_MAT_B1 = 'itu_wet_ground'

# Collect all used materials (remapped)
_used_b1 = set()
for m in mat_plys_b1:
    _used_b1.add(_MAT_REMAP_B1.get(m, m))
_used_b1.add(TERRAIN_MAT_B1)

# ── Build XML ─────────────────────────────────────────────────────────────────
lines_b1 = []
lines_b1.append('<?xml version="1.0" encoding="utf-8"?>')
lines_b1.append('<scene version="2.1.0">')
lines_b1.append('')
lines_b1.append('  <!-- ── ITU-R P.2040-2 Materials (Sionna 0.19) ────────── -->')

_ITU_COLOURS_B1 = {
    'itu_concrete'          : '0.539 0.539 0.539',
    'itu_brick'              : '1.000 0.498 0.055',
    'itu_glass'              : '0.596 0.875 0.541',
    'itu_plywood'            : '0.514 0.376 0.220',
    'itu_metal'              : '0.220 0.220 0.254',
    'itu_wet_ground'         : '0.910 0.569 0.055',
    'itu_very_dry_ground'    : '0.498 0.498 0.498',
    'itu_medium_dry_ground'  : '0.780 0.780 0.780',
    'itu_ceiling_board'      : '0.180 0.450 0.180',
}
for mat_name in sorted(_used_b1):
    _rgb_b1 = _ITU_COLOURS_B1.get(mat_name, '0.5 0.5 0.5')
    lines_b1.append(f'  <bsdf type="diffuse" id="{mat_name}">')
    lines_b1.append(f'    <rgb name="reflectance" value="{_rgb_b1}"/>')
    lines_b1.append(f'  </bsdf>')
    lines_b1.append('')

lines_b1.append('  <!-- ── Terrain ─────────────────────────────────────── -->')
lines_b1.append('  <shape type="ply" id="mesh-ground">')
lines_b1.append('    <string name="filename" value="meshes/terrain.ply"/>')
lines_b1.append(f'    <ref id="{TERRAIN_MAT_B1}" name="bsdf"/>')
lines_b1.append('    <boolean name="face_normals" value="true"/>')
lines_b1.append('  </shape>')
lines_b1.append('')

lines_b1.append('  <!-- ── Shapes ─────────────────────────────────────────── -->')
for mat_name in sorted(mat_plys_b1.keys()):
    ref_mat = _MAT_REMAP_B1.get(mat_name, mat_name)
    for ply_path, role in sorted(mat_plys_b1[mat_name]):
        mesh_id = 'mesh-' + ply_path.split('/')[-1].replace('.ply', '')
        lines_b1.append(f'  <shape type="ply" id="{mesh_id}">')
        lines_b1.append(f'    <string name="filename" value="{ply_path}"/>')
        lines_b1.append(f'    <ref id="{ref_mat}" name="bsdf"/>')
        lines_b1.append(f'    <boolean name="face_normals" value="true"/>')
        lines_b1.append(f'  </shape>')

lines_b1.append('')
lines_b1.append('</scene>')

out_xml = os.path.join(SCENE_DIR, 'scene_with_full_019.xml')
# ── Permanent safety net: dedupe declared bsdf ids + auto-fallback for any
# referenced-but-undeclared material id. Protects against future material
# names (new OSM tags, new ITU types) ever again causing a "duplicate ID"
# or "unresolved reference" crash -- instead of failing, it prints a clear
# warning and patches the XML with a generic diffuse bsdf.
import re as _re_guard

def _guard_bsdf_refs(_lines, _label):
    _seen_ids = set()
    _out = []
    _i = 0
    while _i < len(_lines):
        _line = _lines[_i]
        _m = _re_guard.match(r'\\s*<bsdf[^>]+id="([^"]+)"', _line)
        if _m:
            _mid = _m.group(1)
            _block = [_line]
            _j = _i
            if not _line.rstrip().endswith('/>'):
                while _lines[_j].strip() != '</bsdf>':
                    _j += 1
                    _block.append(_lines[_j])
            if _mid in _seen_ids:
                print(f'[{_label}] WARNING: duplicate bsdf id "{_mid}" -- dropping repeat declaration')
                _i = _j + 1
                if _i < len(_lines) and _lines[_i].strip() == '':
                    _i += 1
                continue
            _seen_ids.add(_mid)
            _out.extend(_block)
            _i = _j + 1
            if _i < len(_lines) and _lines[_i].strip() == '':
                _out.append(_lines[_i])
                _i += 1
            continue
        _out.append(_line)
        _i += 1
    _lines = _out

    _joined = '\\n'.join(_lines)
    _declared = set(_re_guard.findall(r'<bsdf[^>]+id="([^"]+)"', _joined))
    _referenced = set(_re_guard.findall(r'<ref id="([^"]+)" name="bsdf"', _joined))
    _missing = _referenced - _declared
    if _missing:
        print(f'[{_label}] WARNING: {len(_missing)} referenced material id(s) had no declared '
              f'bsdf -- adding generic diffuse fallback so the scene still loads: {sorted(_missing)}')
        _insert_at = next(_k for _k, _l in enumerate(_lines) if '<!-- ── Terrain' in _l)
        _fallback = []
        for _mid in sorted(_missing):
            _fallback += [f'  <bsdf type="diffuse" id="{_mid}">',
                          f'    <rgb name="reflectance" value="0.5 0.5 0.5"/>',
                          f'  </bsdf>', '']
        _lines[_insert_at:_insert_at] = _fallback
    return _lines

lines_b1 = _guard_bsdf_refs(lines_b1, "CELL B1")

with open(out_xml, 'w') as f:
    f.write('\n'.join(lines_b1))

total_shapes = 1 + sum(len(v) for v in mat_plys_b1.values())
print(f'Wrote: {out_xml}')
print(f'  Materials : {len(_used_b1)}')
print(f'  Shapes    : {total_shapes}  (1 terrain + {total_shapes-1} mesh parts)')


## CELL B2 — Blender XML → Sionna 2.0

Copies the original Blender XML and makes the minimum changes needed for Sionna 2.0:
- Maps colour/custom material IDs to nearest `mat-itu_*` equivalent
- Adds missing `mat-itu_*` BSDF definitions (twosided/diffuse)
- Does **not** modify the original Blender file

**Inputs to set:** `BLENDER_XML_IN`, `SIONNA2_XML_OUT`


In [ ]:
# ============================================================
# CELL MERGE — MERGE PLYs BY MATERIAL
# Run this BEFORE Cell B2 if you have 100K+ individual PLY files
# ============================================================
import os, glob, logging
from collections import defaultdict
from pathlib import Path

try:
    import trimesh
except ImportError:
    os.system('pip install trimesh -q')
    import trimesh

# Suppress all trimesh/logging output
logging.getLogger('trimesh').setLevel(logging.ERROR)

MESHES_DIR = MESH_DIR  # auto-derived from SCENARIO_NAME in CELL 0

def get_material(filename):
    name = Path(filename).stem
    return name[4:] if name.startswith('bld_') else name

ply_files = glob.glob(os.path.join(MESHES_DIR, '*.ply'))
print(f'Found {len(ply_files):,} PLY files')

materials = defaultdict(list)
for ply_path in ply_files:
    materials[get_material(ply_path)].append(ply_path)

print(f'Materials: {len(materials)}')
for mat in sorted(materials.keys()):
    print(f'  {mat}: {len(materials[mat]):,} files')
print()

for material, ply_list in sorted(materials.items()):
    if len(ply_list) == 1:
        print(f'✓ {material}: 1 file (skip)')
        continue

    print(f'• {material}: merging {len(ply_list):,} files...', end=' ', flush=True)

    meshes = []
    for ply_path in ply_list:
        try:
            meshes.append(trimesh.load(ply_path, process=False))
        except Exception:
            continue

    if not meshes:
        print('FAILED'); continue

    combined = trimesh.util.concatenate(meshes)
    output_path = os.path.join(MESHES_DIR, f'bld_{material}.ply')
    combined.export(output_path)

    for p in ply_list:
        os.remove(p)

    size_mb = os.path.getsize(output_path) / 1024**2
    print(f'✓ {combined.vertices.shape[0]:,} verts, {size_mb:.1f} MB')

merged = glob.glob(os.path.join(MESHES_DIR, '*.ply'))
print(f'\n✓ Done — {len(merged)} merged files:')
for f in sorted(merged):
    print(f'  {Path(f).name}  {os.path.getsize(f)/1024**2:.1f} MB')


In [ ]:
# ============================================================
# CELL B2 — BUILD scene_sionna2.xml FROM meshes_merged/
# ============================================================
# Reads existing meshes_merged/*.ply (Blender-merged files +
# terrain.ply from CELL 3). Writes a clean Sionna 2.0 XML.
# Does NOT parse untitled.xml or touch meshes/.
# ============================================================
import os
from lxml import etree as _lxe

_merged_dir = globals().get('MERGED_DIR', os.path.join(SCENE_DIR, 'meshes_merged'))
_out_xml    = os.path.join(SCENE_DIR, 'scene_sionna2.xml')

assert os.path.isdir(_merged_dir), f'meshes_merged/ not found: {_merged_dir}'

_ply_files = sorted(f for f in os.listdir(_merged_dir) if f.endswith('.ply'))
print(f'meshes_merged/: {len(_ply_files)} PLY files')

# ── ITU visual colours ───────────────────────────────────────────────────────
_COLOURS = {
    'mat-itu_concrete'        : '0.539 0.539 0.539',
    'mat-itu_brick'           : '1.000 0.498 0.055',
    'mat-itu_glass'           : '0.596 0.875 0.541',
    'mat-itu_wood'            : '0.043 0.580 0.184',
    'mat-itu_metal'           : '0.220 0.220 0.254',
    'mat-itu_wet_ground'      : '0.910 0.569 0.055',
    'mat-itu_very_dry_ground' : '0.498 0.498 0.498',
    'mat-itu_medium_dry_ground': '0.780 0.780 0.780',
}

# ── Blender material name → mat-itu_* ───────────────────────────────────────
_MAT_MAP = {
    'mat-itu_brick'            : 'mat-itu_brick',
    'mat-itu_metal'            : 'mat-itu_metal',
    'mat-itu_concrete'         : 'mat-itu_concrete',
    'mat-itu_glass'            : 'mat-itu_glass',
    'mat-itu_wood'             : 'mat-itu_wood',
    'mat-itu_wet_ground'       : 'mat-itu_wet_ground',
    'mat-itu_very_dry_ground'  : 'mat-itu_very_dry_ground',
    'mat-itu_medium_dry_ground': 'mat-itu_medium_dry_ground',
    'terrain'                  : f'mat-{globals().get("TERRAIN_MATERIAL","itu_wet_ground")}',
}

def _resolve(stem):
    if stem in _MAT_MAP:
        return _MAT_MAP[stem]
    # strip common prefixes and try
    core = stem.replace('mat-itu_','').replace('mat-','')
    _valid = {'concrete','brick','glass','wood','metal',
              'wet_ground','very_dry_ground','medium_dry_ground'}
    if core in _valid:
        return f'mat-itu_{core}'
    return 'mat-itu_concrete'   # fallback

# ── Build XML ────────────────────────────────────────────────────────────────
_root = _lxe.Element('scene', version='2.1.0')

# Integrator
_integ = _lxe.SubElement(_root, 'integrator', type='path', id='elm__0', name='elm__0')
_lxe.SubElement(_integ, 'integer', name='max_depth', value='12')

# Collect needed materials
_used_mats = set()
for _f in _ply_files:
    _used_mats.add(_resolve(_f.replace('.ply','')))

# Write twosided BSDFs
for _mid in sorted(_used_mats):
    _colour = _COLOURS.get(_mid, '0.5 0.5 0.5')
    _b = _lxe.SubElement(_root, 'bsdf', type='twosided', id=_mid, name=_mid)
    _i = _lxe.SubElement(_b, 'bsdf', type='diffuse')
    _lxe.SubElement(_i, 'rgb', value=_colour, name='reflectance')

# Write shapes
for _f in _ply_files:
    _stem  = _f.replace('.ply','')
    _bsdid = _resolve(_stem)
    _sh = _lxe.SubElement(_root, 'shape', type='ply', id=f'mesh-{_stem}')
    _lxe.SubElement(_sh, 'string', name='filename', value=f'meshes_merged/{_f}')
    _lxe.SubElement(_sh, 'ref', id=_bsdid, name='bsdf')
    _lxe.SubElement(_sh, 'boolean', name='face_normals', value='true')

_tree = _lxe.ElementTree(_root)
_tree.write(_out_xml, pretty_print=True, xml_declaration=True, encoding='utf-8')

print(f'\nWrote: {_out_xml}')
print(f'  Shapes   : {len(_ply_files)}')
print(f'  Materials: {sorted(_used_mats)}')
has_terrain = any('terrain' in f for f in _ply_files)
print(f'  Terrain  : {"yes" if has_terrain else "NO — run CELL 3 first"}')


## CELL B3 — OSM Scene → Sionna 2.0 Export

Generates a Sionna 2.0 compatible scene XML from the OSM-built scene (produced by Section A). Uses `mat-` prefixed material IDs, `twosided/diffuse` BSDFs for rendering, and `face_normals=true` on all shapes.

**Requires:** Section A cells 0–5 to have been run first (SCENE_DIR, MESH_DIR must be set).


In [ ]:
# ============================================================
# CELL B3 — WRITE SCENE_SIONNA2.XML  (Sionna 2.0 / Mitsuba 2.1.0)
# ============================================================
# Generates a Sionna 2.0 compatible scene XML alongside the
# existing Sionna 0.19 scene.xml.
# Key differences vs 0.19:
#   - Material IDs prefixed with "mat-"  (mat-itu_brick etc.)
#   - BSDF type = twosided/diffuse + rgb  (visual colour for rendering)
#   - Shapes have id + face_normals=true
#   - Integrator, emitter, sensor blocks included
# ============================================================

# ── Reconstruct mat_plys and config if running standalone ────────────────────
import os
if 'TERRAIN_MATERIAL' not in dir():
    TERRAIN_MATERIAL = 'itu_wet_ground'      # sub-1GHz handled by monkeypatch in sim notebook
# Always reconstruct mat_plys from disk to ensure correct subdirectory paths
mat_plys = {}
# Build skip-set from INCLUDE_ flags so old PLY files don't sneak in when features are disabled
_SKIP_PLY = set()
if not globals().get('INCLUDE_EMBANKMENTS',      True): _SKIP_PLY.add('bld_itu_medium_dry_ground_embankments.ply')
if not globals().get('INCLUDE_ROAD_EMBANKMENTS', True): _SKIP_PLY.add('infra_itu_medium_dry_ground_road_embankments.ply')
if not globals().get('INCLUDE_ROAD_CUTTINGS',    True): _SKIP_PLY.add('infra_itu_medium_dry_ground_road_cuttings.ply')
if not globals().get('INCLUDE_SURFACE_PARKS',    True): _SKIP_PLY.add('surface_itu_asphalt_carparks.ply')
if not globals().get('INCLUDE_TREES',            True): _SKIP_PLY.add('trees_itu_vegetation.ply')
if not globals().get('INCLUDE_RAILWAYS',         True): _SKIP_PLY.add('rail_itu_metal.ply')
if not globals().get('INCLUDE_BARRIERS',         True):
    _SKIP_PLY.update({'barriers_itu_metal.ply', 'barriers_itu_concrete.ply'})
if not globals().get('INCLUDE_SUBSTATIONS',      True): _SKIP_PLY.add('infra_itu_metal_substations.ply')
if _SKIP_PLY: print(f'  [B3] Skipping {len(_SKIP_PLY)} disabled PLY(s): {sorted(_SKIP_PLY)}')
_prefix_role = {'bld_': 'buildings',  'road_': 'roads',
                'water_': 'water',    'veg_': 'vegetation',
                'trees_': 'vegetation','rail_': 'railways',
                'barriers_': 'barriers','infra_': 'infrastructure',
                'surface_': 'surfaces'} # car parks, greenspaces, landuse
# Known ITU material names — sorted longest-first so 'itu_wet_ground' matches before 'itu_wet'
_KNOWN_ITU = sorted([
    'itu_brick', 'itu_concrete', 'itu_glass', 'itu_wood', 'itu_metal',
    'itu_wet_ground', 'itu_medium_dry_ground', 'itu_very_dry_ground',
    'itu_asphalt', 'itu_vegetation', 'itu_water',
], key=len, reverse=True)

def _ply_to_mat(fname, pfx):
    raw = fname[len(pfx):-4]          # e.g. 'itu_metal_pylons'
    for km in _KNOWN_ITU:
        if raw == km or raw.startswith(km + '_'):
            return km                  # e.g. 'itu_metal'
    return raw                         # fallback: use raw (e.g. 'itu_brick' already clean)

for _fname in os.listdir(MESH_DIR):
    if not _fname.endswith('.ply') or _fname == 'terrain.ply' or _fname in _SKIP_PLY:
        continue
    for _pfx, _role in _prefix_role.items():
        if _fname.startswith(_pfx):
            _mat = _ply_to_mat(_fname, _pfx)
            mat_plys.setdefault(_mat, []).append((os.path.basename(MESH_DIR) + '/' + _fname, _role))
            break
print(f'mat_plys from disk: {list(mat_plys.keys())}')
if 'center_lat' not in dir():
    import json as _json
    _p = os.path.join(BASE_DIR, 'scene_parameters.json')
    if os.path.exists(_p):
        _meta = _json.load(open(_p))
        center_lat = _meta['scene_center_lat']
        center_lon = _meta['scene_center_lon']
        UTM_EPSG   = _meta.get('utm_epsg', 27700)
    else:
        center_lat = center_lon = 0.0; UTM_EPSG = 27700

# ── ITU visual colours (for Mitsuba rendering — not EM properties) ──────────
ITU_COLOURS = {
    'itu_concrete'          : '0.539 0.539 0.539',
    'itu_brick'             : '1.000 0.498 0.055',
    'itu_glass'             : '0.596 0.875 0.541',
    'itu_wood'              : '0.514 0.376 0.220',  # warm wood brown
    'itu_plywood'           : '0.514 0.376 0.220',
    'itu_metal'             : '0.220 0.220 0.254',
    'itu_wet_ground'        : '0.910 0.569 0.055',  # wet soil brown
    'itu_very_dry_ground'   : '0.498 0.498 0.498',  # dry pale grey
    'itu_medium_dry_ground' : '0.780 0.780 0.780',  # medium grey
    'asphalt'               : '0.200 0.200 0.200',  # dark road grey (mapped from itu_asphalt)
    'vegetation'            : '0.180 0.450 0.180',  # green
    'water'                 : '0.180 0.330 0.640',  # blue
}

used_mats2 = set(mat_plys.keys()) | {TERRAIN_MATERIAL}

# Sionna 2.0 ITU material types do NOT include asphalt/vegetation/water.
# Give those non-ITU ids (no "itu_" prefix) so Sionna loads them as plain
# RadioMaterials (default eps=1, sigma=0) instead of failing ITU validation.
# The sim notebook (CELL 4A/4B) then sets their eps/sigma/scattering by name.
_NON_ITU_MAP = {
    'itu_asphalt'   : 'itu_very_dry_ground',    # Sionna v2 ITU registry: 'asphalt' not valid; very_dry_ground (εr=3.0) ≈ asphalt εr=2.56
    'itu_vegetation': 'itu_ceiling_board',      # Sionna v2: ceiling_board εr=1.50 matches vegetation εr=1.50 — not used elsewhere
    'itu_water'     : 'itu_medium_dry_ground',  # Sionna v2: medium_dry_ground (εr=15) — distinct from terrain (itu_wet_ground); corrected in Python after load
}
def _s2_matid(m):
    return _NON_ITU_MAP.get(m, m)

# Explicit radio-material properties for the non-ITU surfaces (Sionna has no
# ITU type for these). Written as <bsdf type="radio-material"> so Sionna loads
# them as real radio materials. The sim notebook CELL 4A/4B refine these.
# ── ITU-R P.2040-2 (2023) Table 3 — frequency-portable non-ITU materials ──
# These materials are not in Sionna 2.0 built-in registry so written as
# explicit radio-material BSDFs. Values computed from P.2040-2 formula.
_ITU_P2040_NON_ITU = {
    #                    a      b       c       d      s    rgb
    'itu_asphalt'   : (2.56, 0.000, 0.0050, 0.0000, 0.30, '0.200 0.200 0.200'),
    'itu_vegetation': (1.50, 0.000, 0.0020, 0.5000, 0.40, '0.180 0.450 0.180'),  # ITU-R P.833 — S=0.40 matches diff RT notebook
    'itu_water'     : (80.0, 0.000, 0.0100, 0.0000, 0.02, '0.180 0.330 0.640'),  # ITU-R P.527
}
_f_ghz_b3 = float(globals().get('FREQUENCY_HZ', 915.95e6)) / 1e9
_NON_ITU_RM = {}
for _mn, (_a, _b, _c, _d, _s, _rgb) in _ITU_P2040_NON_ITU.items():
    _NON_ITU_RM[_mn] = dict(
        er    = round(_a * (_f_ghz_b3 ** _b), 6),
        sigma = round(_c * (_f_ghz_b3 ** _d), 8),
        scat  = _s,
        rgb   = _rgb,
    )
print(f'Non-ITU materials computed at {_f_ghz_b3*1e3:.2f} MHz (ITU-R P.2040-2):')
for _mn, _v in _NON_ITU_RM.items():
    print(f'  {_mn:<18}  er={_v["er"]:.4f}  sigma={_v["sigma"]:.6f}  S={_v["scat"]:.2f}')

lines2 = []
lines2.append('<?xml version="1.0" ?>')
lines2.append('<scene version="2.1.0">')
lines2.append('')

# ── Metadata defaults ────────────────────────────────────────────────────────
lines2.append('  <!-- ── Scene metadata ──────────────────────────────────── -->')
lines2.append(f'  <default name="scenegen_version"     value="1.0.0"/>')
lines2.append(f'  <default name="scenegen_min_lat"     value="{SCENE_SOUTH}"/>')
lines2.append(f'  <default name="scenegen_max_lat"     value="{SCENE_NORTH}"/>')
lines2.append(f'  <default name="scenegen_min_lon"     value="{SCENE_WEST}"/>')
lines2.append(f'  <default name="scenegen_max_lon"     value="{SCENE_EAST}"/>')
lines2.append(f'  <default name="scenegen_center_lat"  value="{center_lat}"/>')
lines2.append(f'  <default name="scenegen_center_lon"  value="{center_lon}"/>')
lines2.append(f'  <default name="scenegen_UTM_zone"    value="EPSG:{UTM_EPSG}"/>')
lines2.append(f'  <default name="scenegen_ground_material"  value="mat-{TERRAIN_MATERIAL}"/>')
lines2.append('')

# ── Integrator ───────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Integrator ──────────────────────────────────────── -->')
lines2.append('  <integrator type="path">')
lines2.append('    <integer name="max_depth" value="12"/>')
lines2.append('  </integrator>')
lines2.append('')

# ── Materials — twosided diffuse (visual) ────────────────────────────────────
lines2.append('  <!-- ── ITU-R Materials (visual BSDF for rendering) ─────── -->')
# Dedupe by the FINAL remapped id, not the original material name -- multiple
# original names (e.g. itu_water, itu_medium_dry_ground) can map to the same
# proxy id and would otherwise emit a duplicate <bsdf> id, which Mitsuba's
# parser rejects ("duplicate ID").
_final_ids2 = sorted({_s2_matid(m) for m in used_mats2})
for _fid in _final_ids2:
    # radio-material: Sionna v2 looks up EM props from ITU registry by material ID.
    # Proxy IDs in _NON_ITU_MAP are all valid Sionna v2 ITU types — no error on load.
    # twosided/diffuse would make surfaces radio-transparent (0 paths solved).
    lines2.append(f'  <bsdf type="radio-material" id="mat-{_fid}"/>')
    lines2.append('')

# ── Emitter ──────────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Environment ─────────────────────────────────────── -->')
lines2.append('  <emitter type="constant" id="World">')
lines2.append('    <rgb value="1.0 1.0 1.0" name="radiance"/>')
lines2.append('  </emitter>')
lines2.append('')

# ── Camera ───────────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Camera (top-down overview) ─────────────────────── -->')
lines2.append('  <sensor type="perspective" id="Camera">')
lines2.append('    <string name="fov_axis" value="x"/>')
lines2.append('    <float  name="fov"      value="42.855"/>')
lines2.append('    <float  name="near_clip" value="0.1"/>')
lines2.append('    <float  name="far_clip"  value="10000.0"/>')
lines2.append('    <transform name="to_world">')
lines2.append('      <rotate z="1" angle="-90"/>')
lines2.append('      <translate value="0 0 500"/>')
lines2.append('    </transform>')
lines2.append('    <sampler type="independent">')
lines2.append('      <integer name="sample_count" value="4096"/>')
lines2.append('    </sampler>')
lines2.append('    <film type="hdrfilm">')
lines2.append('      <integer name="width"  value="1024"/>')
lines2.append('      <integer name="height" value="1024"/>')
lines2.append('    </film>')
lines2.append('  </sensor>')
lines2.append('')

# ── Terrain shape ─────────────────────────────────────────────────────────────
lines2.append('  <!-- ── Terrain ─────────────────────────────────────────── -->')
lines2.append('  <shape type="ply" id="mesh-ground">')
lines2.append(f'    <string name="filename" value="meshes/terrain.ply"/>')
lines2.append(f'    <ref id="mat-{TERRAIN_MATERIAL}" name="bsdf"/>')
lines2.append('    <boolean name="face_normals" value="true"/>')
lines2.append('  </shape>')
lines2.append('')

# ── Building shapes ───────────────────────────────────────────────────────────
lines2.append('  <!-- ── Buildings (one merged PLY per material) ──────────── -->')
for mat_name, ply_list in sorted(mat_plys.items()):
    for ply_path, role in ply_list:
        mesh_id = 'mesh-' + ply_path.split('/')[-1].replace('.ply', '')
        lines2.append(f'  <shape type="ply" id="{mesh_id}">')
        lines2.append(f'    <string name="filename" value="{ply_path}"/>')
        lines2.append(f'    <ref id="mat-{_s2_matid(mat_name)}" name="bsdf"/>')
        lines2.append('    <boolean name="face_normals" value="true"/>')
        lines2.append('  </shape>')

lines2.append('')
lines2.append('</scene>')

scene2_xml = os.path.join(SCENE_DIR, 'scene_with_full.xml')
# ── Permanent safety net: dedupe declared bsdf ids + auto-fallback for any
# referenced-but-undeclared material id. Protects against future material
# names (new OSM tags, new ITU types) ever again causing a "duplicate ID"
# or "unresolved reference" crash -- instead of failing, it prints a clear
# warning and patches the XML with a generic diffuse bsdf.
import re as _re_guard

def _guard_bsdf_refs(_lines, _label):
    _seen_ids = set()
    _out = []
    _i = 0
    while _i < len(_lines):
        _line = _lines[_i]
        _m = _re_guard.match(r'\\s*<bsdf[^>]+id="([^"]+)"', _line)
        if _m:
            _mid = _m.group(1)
            _block = [_line]
            _j = _i
            if not _line.rstrip().endswith('/>'):
                while _lines[_j].strip() != '</bsdf>':
                    _j += 1
                    _block.append(_lines[_j])
            if _mid in _seen_ids:
                print(f'[{_label}] WARNING: duplicate bsdf id "{_mid}" -- dropping repeat declaration')
                _i = _j + 1
                if _i < len(_lines) and _lines[_i].strip() == '':
                    _i += 1
                continue
            _seen_ids.add(_mid)
            _out.extend(_block)
            _i = _j + 1
            if _i < len(_lines) and _lines[_i].strip() == '':
                _out.append(_lines[_i])
                _i += 1
            continue
        _out.append(_line)
        _i += 1
    _lines = _out

    _joined = '\\n'.join(_lines)
    _declared = set(_re_guard.findall(r'<bsdf[^>]+id="([^"]+)"', _joined))
    _referenced = set(_re_guard.findall(r'<ref id="([^"]+)" name="bsdf"', _joined))
    _missing = _referenced - _declared
    if _missing:
        print(f'[{_label}] WARNING: {len(_missing)} referenced material id(s) had no declared '
              f'bsdf -- adding generic diffuse fallback so the scene still loads: {sorted(_missing)}')
        _insert_at = next(_k for _k, _l in enumerate(_lines) if '<!-- ── Terrain' in _l)
        _fallback = []
        for _mid in sorted(_missing):
            _fallback += [f'  <bsdf type="diffuse" id="{_mid}">',
                          f'    <rgb name="reflectance" value="0.5 0.5 0.5"/>',
                          f'  </bsdf>', '']
        _lines[_insert_at:_insert_at] = _fallback
    return _lines

lines2 = _guard_bsdf_refs(lines2, "CELL B3")

with open(scene2_xml, 'w') as f:
    f.write('\n'.join(lines2))

print(f'Wrote: {scene2_xml}')
print(f'  Materials : {len(used_mats2)}')
total2 = 1 + sum(len(v) for v in mat_plys.values())
print(f'  Shapes    : {total2}  (1 terrain + {total2-1} feature PLYs: buildings + roads)')
print(f'  Format    : Sionna 2.0 / Mitsuba 2.1.0  (twosided diffuse + face_normals)')



In [ ]:
# ============================================================
# CELL PREVIEW — RENDER THE BUILT SCENE  (same as 900 flat nb)
# ============================================================
# Loads the freshly written scene_sionna2.xml and shows the
# interactive preview so you can visually confirm terrain,
# buildings, roads, water and the new standing vegetation canopy.
# Run after CELL B3.
# ============================================================
%matplotlib inline
no_preview = False   # set True to skip the widget

from sionna.rt import load_scene

_preview_xml = SIONNA19_XML_OUT if "SIONNA19_XML_OUT" in dir() else os.path.join(SCENE_DIR, "scene_with_full_019.xml")
print(f"Loading scene: {_preview_xml}")
scene = load_scene(_preview_xml)
print(f"Objects: {len(scene.objects)}  |  Materials: {len(scene.radio_materials)}")

if not no_preview:
    scene.preview()
